# RQ3 — Frozen RoG vs AdaPruner LLM Reasoning

This notebook contains the final downstream LLM reasoning pipeline used
for the AdaPruner-KGQA RQ3 evaluation.

The pipeline preserves the frozen RoG downstream reasoner, tokenizer,
prompt construction, stochastic decoding contract, matched per-question
random seeds, frozen RoG retrieval evidence, and frozen AdaPruner
retrieval evidence.

No RQ2 retraining or hyperparameter tuning is performed in this notebook.

## Execution order

Run the notebook sequentially from a fresh Kaggle GPU session.

`Fresh Kernel → RQ3-BOOT → RQ3-0 → RQ3-1 → RQ3-2 → Equivalence Audit → RQ3-3 → RQ3-4`

The RQ3-0 manifest-recovery cell is included for reproducibility and
migration recovery and should only be used when required.


## 0. Fresh-Kernel Check

Ensures that the tokenizer libraries have not been imported before the
frozen compatibility environment is activated.


In [ ]:

# ==========================================================================================
# RQ3 FRESH-KERNEL CHECK
# ==========================================================================================

import sys

print("transformers imported:", "transformers" in sys.modules)
print("tokenizers imported:  ", "tokenizers" in sys.modules)
print("sentencepiece imported:", "sentencepiece" in sys.modules)

assert "transformers" not in sys.modules
assert "tokenizers" not in sys.modules
assert "sentencepiece" not in sys.modules

print("\nFRESH KERNEL FOR RQ3-BOOT: PASSED")


## 1. RQ3-BOOT — Exact RoG Tokenizer Environment

Restores the verified RoG tokenizer/software compatibility stack and
checks the exact slow SentencePiece tokenizer behavior.


In [ ]:

import sys
from pathlib import Path


print("=" * 120)
print("RQ3-BOOT — EXACT RoG SLOW-TOKENIZER ENVIRONMENT")
print("=" * 120)


# ==================================================================================================
# 1. HARD FRESH-PROCESS GATE
# ==================================================================================================

for forbidden in [
    "transformers",
    "tokenizers",
    "sentencepiece",
]:

    assert forbidden not in sys.modules, (
        f"STOP: {forbidden} is already imported.\n"
        "Restart the Kaggle session and run RQ3-BOOT FIRST."
    )


# ==================================================================================================
# 2. EXACT SAVED COMPATIBILITY ENVIRONMENT
# ==================================================================================================

RQ3_EXACT_ENV = Path(
    "/kaggle/input/datasets/"
    "sabitahnaf/"
    "adapruner-kgqa-migration/"
    "rog_exact_tokenizer_env_4442"
)


assert RQ3_EXACT_ENV.exists(), (
    f"Missing exact RoG environment:\n{RQ3_EXACT_ENV}"
)


assert (
    RQ3_EXACT_ENV
    /
    "transformers"
).exists()


assert (
    RQ3_EXACT_ENV
    /
    "tokenizers"
).exists()


assert (
    RQ3_EXACT_ENV
    /
    "sentencepiece"
).exists()


# Put the frozen environment BEFORE system site-packages.

sys.path.insert(
    0,
    str(
        RQ3_EXACT_ENV
    )
)


# ==================================================================================================
# 3. IMPORT EXACT STACK
# ==================================================================================================

import transformers
import tokenizers
import sentencepiece


print(
    "\ntransformers:",
    transformers.__version__
)

print(
    "tokenizers:",
    tokenizers.__version__
)

print(
    "sentencepiece:",
    sentencepiece.__version__
)


assert (
    transformers.__version__
    ==
    "4.44.2"
), (
    "Wrong Transformers version."
)


assert (
    tokenizers.__version__
    ==
    "0.19.1"
), (
    "Wrong tokenizers version."
)


assert (
    sentencepiece.__version__
    ==
    "0.2.0"
), (
    "Wrong sentencepiece version."
)


# ==================================================================================================
# 4. RoG REPOSITORY
# ==================================================================================================

RQ3_REPO_DIR = Path(
    "/kaggle/working/reasoning-on-graphs"
)


RQ3_MIGRATION_REPO = Path(
    "/kaggle/input/datasets/"
    "sabitahnaf/"
    "adapruner-kgqa-migration/"
    "reasoning-on-graphs"
)


# Prefer the already-recovered working copy.
# Otherwise use the frozen migration copy.

if not RQ3_REPO_DIR.exists():

    RQ3_REPO_DIR = (
        RQ3_MIGRATION_REPO
    )


assert (
    RQ3_REPO_DIR.exists()
)


RQ3_SRC_DIR = (
    RQ3_REPO_DIR
    /
    "src"
)


assert (
    RQ3_SRC_DIR.exists()
)


sys.path.insert(
    0,
    str(
        RQ3_SRC_DIR
    )
)


# ==================================================================================================
# 5. LOAD TOKENIZER — EXACT SLOW CONTRACT
# ==================================================================================================

from transformers import (
    AutoTokenizer,
)


RQ3_MODEL_ID = (
    "rmanluo/RoG"
)


RQ3_TOKENIZER = (
    AutoTokenizer.from_pretrained(

        RQ3_MODEL_ID,

        use_fast=False,

        clean_up_tokenization_spaces=True,
    )
)


print(
    "\nTokenizer class:",
    RQ3_TOKENIZER.__class__.__name__
)

print(
    "Tokenizer module:",
    RQ3_TOKENIZER.__class__.__module__
)

print(
    "is_fast:",
    RQ3_TOKENIZER.is_fast
)

print(
    "has sp_model:",
    hasattr(
        RQ3_TOKENIZER,
        "sp_model"
    )
)

print(
    "name_or_path:",
    RQ3_TOKENIZER.name_or_path
)


assert (
    RQ3_TOKENIZER.is_fast
    is False
), (
    "STOP: tokenizer is still fast."
)


assert hasattr(
    RQ3_TOKENIZER,
    "sp_model"
), (
    "STOP: expected SentencePiece-backed slow tokenizer."
)


assert (
    RQ3_TOKENIZER.bos_token_id
    ==
    1
)


assert (
    RQ3_TOKENIZER.eos_token_id
    ==
    2
)


assert (
    RQ3_TOKENIZER.unk_token_id
    ==
    0
)


# ==================================================================================================
# 6. TOKEN-ID FIDELITY PROBES
# ==================================================================================================

PROBES = {

    "people.person.place_of_birth":
        [
            2305,
            29889,
            10532,
            29889,
            6689,
            29918,
            974,
            29918,
            29890,
            7515,
        ],

    "location.location.containedby":
        [
            4423,
            29889,
            5479,
            29889,
            1285,
            7114,
            1609,
        ],

    "what does jamaican people speak":
        [
            825,
            947,
            432,
            3304,
            2185,
            2305,
            7726,
        ],
}


print(
    "\n"
    +
    "=" * 120
)

print(
    "SLOW-TOKENIZER FIDELITY PROBES"
)

print(
    "=" * 120
)


for text, expected in (
    PROBES.items()
):

    observed = (
        RQ3_TOKENIZER.encode(
            text,
            add_special_tokens=False
        )
    )


    print(
        "\nTEXT:"
    )

    print(
        repr(
            text
        )
    )


    print(
        "observed:",
        observed
    )


    print(
        "expected:",
        expected
    )


    print(
        "exact:",
        observed
        ==
        expected
    )


    assert (
        observed
        ==
        expected
    ), (
        f"STOP: slow-tokenizer fidelity failed for:\n{text}"
    )


# ==================================================================================================
# 7. FINAL BOOT FLAG
# ==================================================================================================

RQ3_EXACT_TOKENIZER_ENV_COMPLETE = (
    True
)


print(
    "\n"
    +
    "=" * 120
)

print(
    "RQ3 EXACT TOKENIZER ENVIRONMENT: PASSED"
)

print(
    "=" * 120
)


print(
    "\nRQ3_EXACT_TOKENIZER_ENV_COMPLETE:",
    RQ3_EXACT_TOKENIZER_ENV_COMPLETE
)


print(
    "\ntransformers:",
    transformers.__version__
)

print(
    "tokenizers:",
    tokenizers.__version__
)

print(
    "sentencepiece:",
    sentencepiece.__version__
)

print(
    "tokenizer class:",
    RQ3_TOKENIZER.__class__.__name__
)

print(
    "is_fast:",
    RQ3_TOKENIZER.is_fast
)

print(
    "SentencePiece-backed:",
    hasattr(
        RQ3_TOKENIZER,
        "sp_model"
    )
)


print(
    "\nNEXT: revised RQ3-0 reasoner contract audit"
)


## 2. RQ3-0 — Downstream Reasoner Contract Audit

Freezes and verifies the exact RoG downstream-reasoning contract before
full TEST answer generation.


In [ ]:

from pathlib import Path
from typing import Callable
import ast
import hashlib
import inspect
import json
import random
import re
import string
import subprocess
import sys
import types

import numpy as np
import pandas as pd
import torch
import transformers
import tokenizers
import sentencepiece

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)


print("=" * 140)
print("RQ3-0 — EXACT RoG DOWNSTREAM REASONER CONTRACT AUDIT — FINAL REVISED")
print("=" * 140)


# ==================================================================================================
# 1. HARD RQ3-BOOT GATE
# ==================================================================================================

assert globals().get(
    "RQ3_EXACT_TOKENIZER_ENV_COMPLETE",
    False
) is True, (
    "STOP: RQ3-BOOT has not completed successfully."
)


assert (
    transformers.__version__
    ==
    "4.44.2"
)


assert (
    tokenizers.__version__
    ==
    "0.19.1"
)


assert (
    sentencepiece.__version__
    ==
    "0.2.0"
)


print("\nRQ3-BOOT package gate: PASSED")

print(
    " transformers :",
    transformers.__version__
)

print(
    " tokenizers   :",
    tokenizers.__version__
)

print(
    " sentencepiece:",
    sentencepiece.__version__
)


# ==================================================================================================
# 2. RECOVER OR RECREATE THE EXACT SLOW RoG TOKENIZER
#    IMPORTANT: NEVER CLOBBER THE RQ3-BOOT TOKENIZER FIRST
# ==================================================================================================

def is_exact_rq3_tokenizer_0(
    obj
):

    if obj is None:

        return False


    try:

        return (

            type(
                obj
            ).__name__
            ==
            "LlamaTokenizer"

            and

            getattr(
                obj,
                "is_fast",
                None
            )
            is False

            and

            hasattr(
                obj,
                "sp_model"
            )

            and

            "rmanluo/rog"
            in
            str(
                getattr(
                    obj,
                    "name_or_path",
                    ""
                )
            ).lower()
        )

    except Exception:

        return False


_existing_tokenizer_0 = (
    globals().get(
        "RQ3_TOKENIZER",
        None
    )
)


RQ3_TOKENIZER_SOURCE_NAME = (
    None
)


if is_exact_rq3_tokenizer_0(
    _existing_tokenizer_0
):

    RQ3_TOKENIZER = (
        _existing_tokenizer_0
    )

    RQ3_TOKENIZER_SOURCE_NAME = (
        "RQ3_TOKENIZER from RQ3-BOOT/current kernel"
    )


else:

    _recovered_tokenizer_0 = (
        None
    )


    for _candidate_name_0 in [

        "rq3_tokenizer",
        "tokenizer",
        "RQ3_BOOT_TOKENIZER",

    ]:

        _candidate_0 = globals().get(
            _candidate_name_0,
            None
        )


        if is_exact_rq3_tokenizer_0(
            _candidate_0
        ):

            _recovered_tokenizer_0 = (
                _candidate_0
            )

            RQ3_TOKENIZER_SOURCE_NAME = (
                _candidate_name_0
            )

            break


    if (
        _recovered_tokenizer_0
        is None
    ):

        print(
            "\nExact tokenizer Python reference not available; "
            "recreating it under the verified RQ3-BOOT stack..."
        )


        _recovered_tokenizer_0 = (
            AutoTokenizer.from_pretrained(

                "rmanluo/RoG",

                use_fast=False,

                clean_up_tokenization_spaces=True,
            )
        )


        RQ3_TOKENIZER_SOURCE_NAME = (
            "recreated under verified RQ3-BOOT environment"
        )


    RQ3_TOKENIZER = (
        _recovered_tokenizer_0
    )


assert (
    type(
        RQ3_TOKENIZER
    ).__name__
    ==
    "LlamaTokenizer"
)


assert (
    RQ3_TOKENIZER.is_fast
    is False
)


assert hasattr(
    RQ3_TOKENIZER,
    "sp_model"
)


assert (
    str(
        RQ3_TOKENIZER.name_or_path
    ).lower()
    ==
    "rmanluo/rog"
)


print(
    "\nExact slow tokenizer recovery: PASSED"
)

print(
    " source   :",
    RQ3_TOKENIZER_SOURCE_NAME
)

print(
    " class    :",
    type(
        RQ3_TOKENIZER
    ).__name__
)

print(
    " module   :",
    type(
        RQ3_TOKENIZER
    ).__module__
)

print(
    " is_fast  :",
    RQ3_TOKENIZER.is_fast
)

print(
    " sp_model :",
    hasattr(
        RQ3_TOKENIZER,
        "sp_model"
    )
)

print(
    " model    :",
    RQ3_TOKENIZER.name_or_path
)


# ==================================================================================================
# 3. TOKENIZER FIDELITY PROBES
# ==================================================================================================

RQ3_TOKENIZER_PROBES_0 = {

    "people.person.place_of_birth":
        [
            2305,
            29889,
            10532,
            29889,
            6689,
            29918,
            974,
            29918,
            29890,
            7515,
        ],

    "location.location.containedby":
        [
            4423,
            29889,
            5479,
            29889,
            1285,
            7114,
            1609,
        ],

    "what does jamaican people speak":
        [
            825,
            947,
            432,
            3304,
            2185,
            2305,
            7726,
        ],
}


for _text_0, _expected_0 in (
    RQ3_TOKENIZER_PROBES_0.items()
):

    _observed_0 = (
        RQ3_TOKENIZER.encode(
            _text_0,
            add_special_tokens=False
        )
    )


    assert (
        _observed_0
        ==
        _expected_0
    ), (
        f"Tokenizer fidelity mismatch for {_text_0!r}:\n"
        f"observed={_observed_0}\n"
        f"expected={_expected_0}"
    )


print(
    "Tokenizer fidelity recheck:   PASSED"
)


# ==================================================================================================
# 4. HASH HELPERS
# ==================================================================================================

def sha256_file_rq3_0(
    path,
    chunk_size=1024 * 1024
):

    path = Path(
        path
    )


    h = hashlib.sha256()


    with path.open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )


            if not block:

                break


            h.update(
                block
            )


    return h.hexdigest()


def canonical_json_sha256_rq3_0(
    path
):

    with Path(
        path
    ).open(
        "r",
        encoding="utf-8"
    ) as f:

        obj = json.load(
            f
        )


    canonical = json.dumps(

        obj,

        sort_keys=True,

        separators=(
            ",",
            ":"
        ),

        ensure_ascii=False,
    )


    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


# ==================================================================================================
# 5. RESOLVE THE EXACT RoG SOURCE REPOSITORY
# ==================================================================================================

EXPECTED_ROG_COMMIT_0 = (
    "ccf8ec847bf61005a1b27cc9e5aff5c8ead7a24b"
)


EXPECTED_PROMPT_SHA_0 = (
    "1e7946ef161aed47730790ce559027ea01a6be047a2c8e84b1ea18afe7284109"
)


EXPECTED_PROMPT_BUILDER_SHA_0 = (
    "13e92196671a86a36f84063d0821a6fd7ba626b3d8081c90378c15137e4ab522"
)


EXPECTED_EVALUATOR_SHA_0 = (
    "a76961edd78951c9ccc7e0a675cd651b4eb234993b0d193ffaecc991308e8673"
)


def repo_contract_files_0(
    root
):

    root = Path(
        root
    )


    return {

        "prompt":
            root
            /
            "prompts"
            /
            "llama2_predict.txt",

        "prompt_builder":
            root
            /
            "src"
            /
            "qa_prediction"
            /
            "build_qa_input.py",

        "evaluator":
            root
            /
            "src"
            /
            "qa_prediction"
            /
            "evaluate_results.py",

        "predict_answer":
            root
            /
            "src"
            /
            "qa_prediction"
            /
            "predict_answer.py",

        "llama_wrapper":
            root
            /
            "src"
            /
            "llms"
            /
            "language_models"
            /
            "llama.py",
    }


_repo_candidates_0 = [

    Path(
        "/kaggle/working/"
        "reasoning-on-graphs"
    ),

    Path(
        "/kaggle/input/datasets/"
        "sabitahnaf/"
        "adapruner-kgqa-migration/"
        "reasoning-on-graphs"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/"
        "rog-ap/"
        "reasoning-on-graphs"
    ),
]


ROG_REPO_RQ3_0 = (
    None
)


ROG_FILES_RQ3_0 = (
    None
)


for _root_0 in (
    _repo_candidates_0
):

    _files_0 = repo_contract_files_0(
        _root_0
    )


    if not all(
        p.exists()
        for p in _files_0.values()
    ):

        continue


    if (
        sha256_file_rq3_0(
            _files_0[
                "prompt"
            ]
        )
        !=
        EXPECTED_PROMPT_SHA_0
    ):

        continue


    if (
        sha256_file_rq3_0(
            _files_0[
                "prompt_builder"
            ]
        )
        !=
        EXPECTED_PROMPT_BUILDER_SHA_0
    ):

        continue


    if (
        sha256_file_rq3_0(
            _files_0[
                "evaluator"
            ]
        )
        !=
        EXPECTED_EVALUATOR_SHA_0
    ):

        continue


    ROG_REPO_RQ3_0 = (
        _root_0
    )


    ROG_FILES_RQ3_0 = (
        _files_0
    )


    break


if (
    ROG_REPO_RQ3_0
    is None
):

    for _build_file_0 in (
        Path(
            "/kaggle/input"
        ).rglob(
            "build_qa_input.py"
        )
    ):

        try:

            _root_0 = (
                _build_file_0.parents[
                    2
                ]
            )


            _files_0 = (
                repo_contract_files_0(
                    _root_0
                )
            )


            if not all(
                p.exists()
                for p in _files_0.values()
            ):

                continue


            if (

                sha256_file_rq3_0(
                    _files_0[
                        "prompt"
                    ]
                )
                ==
                EXPECTED_PROMPT_SHA_0

                and

                sha256_file_rq3_0(
                    _files_0[
                        "prompt_builder"
                    ]
                )
                ==
                EXPECTED_PROMPT_BUILDER_SHA_0

                and

                sha256_file_rq3_0(
                    _files_0[
                        "evaluator"
                    ]
                )
                ==
                EXPECTED_EVALUATOR_SHA_0
            ):

                ROG_REPO_RQ3_0 = (
                    _root_0
                )


                ROG_FILES_RQ3_0 = (
                    _files_0
                )


                break


        except Exception:

            pass


assert (
    ROG_REPO_RQ3_0
    is not None
), (
    "STOP: could not locate the SHA-verified RoG source repository."
)


RQ3_PROMPT_SHA_0 = (
    sha256_file_rq3_0(
        ROG_FILES_RQ3_0[
            "prompt"
        ]
    )
)


RQ3_PROMPT_BUILDER_SHA_0 = (
    sha256_file_rq3_0(
        ROG_FILES_RQ3_0[
            "prompt_builder"
        ]
    )
)


RQ3_EVALUATOR_SHA_0 = (
    sha256_file_rq3_0(
        ROG_FILES_RQ3_0[
            "evaluator"
        ]
    )
)


RQ3_PREDICT_ANSWER_SHA_0 = (
    sha256_file_rq3_0(
        ROG_FILES_RQ3_0[
            "predict_answer"
        ]
    )
)


RQ3_LLAMA_WRAPPER_SHA_0 = (
    sha256_file_rq3_0(
        ROG_FILES_RQ3_0[
            "llama_wrapper"
        ]
    )
)


assert (
    RQ3_PROMPT_SHA_0
    ==
    EXPECTED_PROMPT_SHA_0
)


assert (
    RQ3_PROMPT_BUILDER_SHA_0
    ==
    EXPECTED_PROMPT_BUILDER_SHA_0
)


assert (
    RQ3_EVALUATOR_SHA_0
    ==
    EXPECTED_EVALUATOR_SHA_0
)


print(
    "\nExact RoG source repository: PASSED"
)

print(
    " repo          :",
    ROG_REPO_RQ3_0
)

print(
    " prompt SHA    :",
    RQ3_PROMPT_SHA_0
)

print(
    " builder SHA   :",
    RQ3_PROMPT_BUILDER_SHA_0
)

print(
    " evaluator SHA :",
    RQ3_EVALUATOR_SHA_0
)

print(
    " predict SHA   :",
    RQ3_PREDICT_ANSWER_SHA_0
)

print(
    " llama SHA     :",
    RQ3_LLAMA_WRAPPER_SHA_0
)


# ==================================================================================================
# 6. GIT COMMIT GATE WHEN AVAILABLE
# ==================================================================================================

RQ3_ROG_GIT_COMMIT_0 = (
    None
)


if (
    ROG_REPO_RQ3_0
    /
    ".git"
).exists():

    try:

        RQ3_ROG_GIT_COMMIT_0 = (

            subprocess.check_output(

                [
                    "git",
                    "-C",
                    str(
                        ROG_REPO_RQ3_0
                    ),
                    "rev-parse",
                    "HEAD",
                ],

                text=True,
            )

            .strip()
        )


    except Exception:

        RQ3_ROG_GIT_COMMIT_0 = (
            None
        )


if (
    RQ3_ROG_GIT_COMMIT_0
    is not None
):

    assert (
        RQ3_ROG_GIT_COMMIT_0
        ==
        EXPECTED_ROG_COMMIT_0
    )


    print(
        "\nRoG git commit gate: PASSED"
    )

    print(
        " commit:",
        RQ3_ROG_GIT_COMMIT_0
    )


else:

    print(
        "\nRoG .git metadata unavailable; "
        "exact source SHA gates remain active."
    )


# ==================================================================================================
# 7. AUDIT NATIVE RoG REASONER / PromptBuilder CONTRACT
# ==================================================================================================

PREDICT_SOURCE_0 = (
    ROG_FILES_RQ3_0[
        "predict_answer"
    ]
    .read_text(
        encoding="utf-8"
    )
)


LLAMA_SOURCE_0 = (
    ROG_FILES_RQ3_0[
        "llama_wrapper"
    ]
    .read_text(
        encoding="utf-8"
    )
)


BUILDER_SOURCE_0 = (
    ROG_FILES_RQ3_0[
        "prompt_builder"
    ]
    .read_text(
        encoding="utf-8"
    )
)


assert (
    "PromptBuilder("
    in
    PREDICT_SOURCE_0
)


assert (
    "self.maximun_token = 4096 - 100"
    in
    LLAMA_SOURCE_0
)


assert (
    "len(self.tokenizer.tokenize(text))"
    in
    LLAMA_SOURCE_0
)


assert (
    "max_new_tokens"
    in
    LLAMA_SOURCE_0
)


assert (
    "use_fast=False"
    in
    LLAMA_SOURCE_0
)


assert (
    "reasoning_paths = self.apply_rules"
    in
    BUILDER_SOURCE_0
)


assert (
    "utils.bfs_with_rule"
    in
    BUILDER_SOURCE_0
)


assert (
    "Reasoning Paths:"
    in
    BUILDER_SOURCE_0
)


assert (
    "Question:"
    in
    BUILDER_SOURCE_0
)


assert (
    "check_prompt_length"
    in
    BUILDER_SOURCE_0
)


assert (
    "random.shuffle"
    in
    BUILDER_SOURCE_0
)


print(
    "\nNative RoG reasoner-source audit: PASSED"
)

print(
    " prompt budget source : 4096 - 100"
)

print(
    " slow-token count     : len(tokenizer.tokenize(text))"
)

print(
    " internal BFS detected: YES"
)


print(
    "\nCRITICAL RQ3 RULE:"
)

print(
    "PromptBuilder.process_input() MUST NOT be used directly "
    "on AFP TEST graphs."
)

print(
    "RQ3-1 must inject the already-frozen retrieved paths "
    "into the exact prompt-formatting contract."
)


# ==================================================================================================
# 8. RECONSTRUCT EXACT PromptBuilder CLASS
# ==================================================================================================

_BUILDER_AST_0 = ast.parse(
    BUILDER_SOURCE_0
)


_PROMPT_BUILDER_CLASS_SOURCE_0 = (
    None
)


for _node_0 in (
    _BUILDER_AST_0.body
):

    if (
        isinstance(
            _node_0,
            ast.ClassDef
        )
        and
        _node_0.name
        ==
        "PromptBuilder"
    ):

        _PROMPT_BUILDER_CLASS_SOURCE_0 = (
            ast.get_source_segment(
                BUILDER_SOURCE_0,
                _node_0
            )
        )


        break


assert (
    _PROMPT_BUILDER_CLASS_SOURCE_0
)


_builder_ns_0 = {

    "__builtins__":
        __builtins__,

    "random":
        random,

    "Callable":
        Callable,

    "utils":
        types.SimpleNamespace(),
}


exec(
    _PROMPT_BUILDER_CLASS_SOURCE_0,
    _builder_ns_0
)


PromptBuilder_RQ3 = (
    _builder_ns_0[
        "PromptBuilder"
    ]
)


RQ3_REASONER_PROMPT_PATH = (
    ROG_FILES_RQ3_0[
        "prompt"
    ]
)


RQ3_PROMPT_MAXIMUM_TOKEN = (
    4096
    -
    100
)


assert (
    RQ3_PROMPT_MAXIMUM_TOKEN
    ==
    3996
)


def rq3_slow_token_count_0(
    text
):

    return len(
        RQ3_TOKENIZER.tokenize(
            text
        )
    )


RQ3_PROMPT_BUILDER = (
    PromptBuilder_RQ3(

        str(
            RQ3_REASONER_PROMPT_PATH
        ),

        add_rule=True,

        use_true=False,

        cot=False,

        explain=False,

        use_random=False,

        each_line=False,

        maximun_token=
            RQ3_PROMPT_MAXIMUM_TOKEN,

        tokenize=
            rq3_slow_token_count_0,
    )
)


assert (
    RQ3_PROMPT_BUILDER.add_rule
    is True
)


assert (
    RQ3_PROMPT_BUILDER.use_true
    is False
)


assert (
    RQ3_PROMPT_BUILDER.cot
    is False
)


assert (
    RQ3_PROMPT_BUILDER.explain
    is False
)


assert (
    RQ3_PROMPT_BUILDER.use_random
    is False
)


assert (
    RQ3_PROMPT_BUILDER.each_line
    is False
)


assert (
    int(
        RQ3_PROMPT_BUILDER.maximun_token
    )
    ==
    3996
)


print(
    "\nExact PromptBuilder contract: PASSED"
)

print(
    " signature      :",
    inspect.signature(
        PromptBuilder_RQ3
    )
)

print(
    " maximun_token :",
    RQ3_PROMPT_BUILDER.maximun_token
)


# ==================================================================================================
# 9. RECONSTRUCT EXACT RoG EVALUATOR HELPERS
# ==================================================================================================

EVAL_SOURCE_0 = (
    ROG_FILES_RQ3_0[
        "evaluator"
    ]
    .read_text(
        encoding="utf-8"
    )
)


EVAL_AST_0 = ast.parse(
    EVAL_SOURCE_0
)


_required_eval_functions_0 = [

    "normalize",
    "match",
    "eval_acc",
    "eval_hit",
    "eval_f1",
    "extract_topk_prediction",
]


_eval_segments_0 = {}


for _node_0 in (
    EVAL_AST_0.body
):

    if (
        isinstance(
            _node_0,
            ast.FunctionDef
        )
        and
        _node_0.name
        in
        _required_eval_functions_0
    ):

        _segment_0 = (
            ast.get_source_segment(
                EVAL_SOURCE_0,
                _node_0
            )
        )


        assert (
            _segment_0
        )


        _eval_segments_0[
            _node_0.name
        ] = (
            _segment_0
        )


assert (
    set(
        _eval_segments_0
    )
    ==
    set(
        _required_eval_functions_0
    )
)


_eval_ns_0 = {

    "__builtins__":
        __builtins__,

    "re":
        re,

    "string":
        string,
}


for _fn_name_0 in (
    _required_eval_functions_0
):

    exec(
        _eval_segments_0[
            _fn_name_0
        ],
        _eval_ns_0
    )


RQ3_EVAL_NORMALIZE = (
    _eval_ns_0[
        "normalize"
    ]
)


RQ3_EVAL_MATCH = (
    _eval_ns_0[
        "match"
    ]
)


RQ3_EVAL_ACC = (
    _eval_ns_0[
        "eval_acc"
    ]
)


RQ3_EVAL_HIT = (
    _eval_ns_0[
        "eval_hit"
    ]
)


RQ3_EVAL_F1 = (
    _eval_ns_0[
        "eval_f1"
    ]
)


RQ3_EXTRACT_TOPK = (
    _eval_ns_0[
        "extract_topk_prediction"
    ]
)


print(
    "\nExact RoG evaluator helpers: PASSED"
)


for _fn_name_0 in (
    _required_eval_functions_0
):

    print(
        f" {_fn_name_0:24s}",
        inspect.signature(
            _eval_ns_0[
                _fn_name_0
            ]
        )
    )


# ==================================================================================================
# 10. CUDA + EXACT RoG 7B REASONER
# ==================================================================================================

assert torch.cuda.is_available(), (
    "STOP: CUDA is unavailable."
)


RQ3_GPU_NAME = (
    torch.cuda.get_device_name(
        0
    )
)


print(
    "\nCUDA gate: PASSED"
)

print(
    " GPU:",
    RQ3_GPU_NAME
)


RQ3_REASONER_MODEL_ID = (
    "rmanluo/RoG"
)


def rq3_is_exact_reasoner_model_0(
    obj
):

    if (
        obj is None
        or
        not hasattr(
            obj,
            "generate"
        )
    ):

        return False


    try:

        ids = []


        for candidate_obj in [

            obj,

            getattr(
                obj,
                "config",
                None
            ),

        ]:

            if candidate_obj is None:

                continue


            for attr in [

                "name_or_path",
                "_name_or_path",

            ]:

                value = getattr(
                    candidate_obj,
                    attr,
                    None
                )


                if isinstance(
                    value,
                    str
                ):

                    ids.append(
                        value
                    )


        return any(

            RQ3_REASONER_MODEL_ID.lower()
            in
            x.lower()

            for x in ids
        )


    except Exception:

        return False


_existing_model_0 = (
    globals().get(
        "RQ3_MODEL",
        None
    )
)


if rq3_is_exact_reasoner_model_0(
    _existing_model_0
):

    RQ3_MODEL = (
        _existing_model_0
    )


    print(
        "\nReusing already-loaded exact RoG 7B reasoner."
    )


else:

    print(
        "\nLoading exact RoG 7B downstream reasoner..."
    )


    RQ3_MODEL = (

        AutoModelForCausalLM
        .from_pretrained(

            RQ3_REASONER_MODEL_ID,

            device_map="auto",

            torch_dtype=torch.float16,
        )
    )


RQ3_MODEL.eval()


assert rq3_is_exact_reasoner_model_0(
    RQ3_MODEL
)


RQ3_MODEL_PARAM_DTYPES = sorted(

    {
        str(
            p.dtype
        )

        for p in
        RQ3_MODEL.parameters()
    }
)


assert (
    RQ3_MODEL_PARAM_DTYPES
    ==
    [
        "torch.float16"
    ]
), (
    f"Unexpected model parameter dtypes: "
    f"{RQ3_MODEL_PARAM_DTYPES}"
)


RQ3_DEVICE_MAP = getattr(
    RQ3_MODEL,
    "hf_device_map",
    None
)


try:

    RQ3_PRIMARY_DEVICE = str(
        RQ3_MODEL.device
    )

except Exception:

    RQ3_PRIMARY_DEVICE = str(
        next(
            RQ3_MODEL.parameters()
        ).device
    )


print(
    "\nExact RoG downstream reasoner: PASSED"
)

print(
    " class            :",
    type(
        RQ3_MODEL
    ).__name__
)

print(
    " parameter dtypes :",
    RQ3_MODEL_PARAM_DTYPES
)

print(
    " primary device   :",
    RQ3_PRIMARY_DEVICE
)

print(
    " hf_device_map    :",
    RQ3_DEVICE_MAP
)


# ==================================================================================================
# 11. FREEZE EXACT GENERATION CONTRACT
# ==================================================================================================

RQ3_MAX_NEW_TOKENS = (
    512
)


RQ3_GENERATION_KWARGS = {

    "max_new_tokens":
        512,

    "do_sample":
        True,
}


assert (
    set(
        RQ3_GENERATION_KWARGS
    )
    ==
    {
        "max_new_tokens",
        "do_sample",
    }
)


_generation_default_fields_0 = [

    "temperature",
    "top_p",
    "top_k",
    "num_beams",
    "repetition_penalty",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",
]


RQ3_MODEL_GENERATION_DEFAULTS = {}


for _field_0 in (
    _generation_default_fields_0
):

    _value_0 = getattr(
        RQ3_MODEL.generation_config,
        _field_0,
        None
    )


    if (

        isinstance(
            _value_0,
            (
                str,
                int,
                float,
                bool,
            )
        )

        or

        _value_0 is None
    ):

        RQ3_MODEL_GENERATION_DEFAULTS[
            _field_0
        ] = (
            _value_0
        )


    else:

        try:

            RQ3_MODEL_GENERATION_DEFAULTS[
                _field_0
            ] = list(
                _value_0
            )

        except Exception:

            RQ3_MODEL_GENERATION_DEFAULTS[
                _field_0
            ] = str(
                _value_0
            )


print(
    "\nFrozen downstream generation contract: PASSED"
)

print(
    " max_new_tokens: 512"
)

print(
    " do_sample     : True"
)

print(
    " explicit temperature/top_p/top_k/num_beams overrides: NONE"
)


print(
    " inherited defaults:"
)


for _k_0, _v_0 in (
    RQ3_MODEL_GENERATION_DEFAULTS.items()
):

    print(
        f"  {_k_0:20s}: {_v_0}"
    )


# ==================================================================================================
# 12. CONTEXT-WINDOW AUDIT
# ==================================================================================================

RQ3_MODEL_MAX_POSITION = int(

    getattr(
        RQ3_MODEL.config,
        "max_position_embeddings",
        4096
    )
)


print(
    "\nContext contract:"
)

print(
    " model max_position_embeddings:",
    RQ3_MODEL_MAX_POSITION
)

print(
    " PromptBuilder maximum_token  :",
    RQ3_PROMPT_MAXIMUM_TOKEN
)

print(
    " max_new_tokens               :",
    RQ3_MAX_NEW_TOKENS
)


if (

    RQ3_PROMPT_MAXIMUM_TOKEN

    +

    RQ3_MAX_NEW_TOKENS

    >

    RQ3_MODEL_MAX_POSITION
):

    print(
        " NOTE: native prompt budget + max generation "
        "exceeds the nominal context window."
    )

    print(
        "       RQ3-1 must measure actual serialized prompt "
        "lengths before TEST generation."
    )


# ==================================================================================================
# 13. FREEZE MATCHED RoG-vs-AFP RNG POLICY
# ==================================================================================================

RQ3_FULL_TEST_BASE_SEED = (
    42
)


def rq3_question_seed(

    dataset,
    question_id,

    base_seed=
        RQ3_FULL_TEST_BASE_SEED,
):

    payload = (
        f"{int(base_seed)}|"
        f"{str(dataset).strip().lower()}|"
        f"{str(question_id)}"
    )


    digest = hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).digest()


    return (

        int.from_bytes(

            digest[
                :8
            ],

            byteorder="big",

            signed=False,
        )

        %

        (
            2**31
            -
            1
        )
    )


def rq3_set_question_seed(
    dataset,
    question_id
):

    seed = rq3_question_seed(
        dataset,
        question_id
    )


    random.seed(
        seed
    )


    np.random.seed(
        seed
    )


    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


    return seed


assert (

    rq3_question_seed(
        "webqsp",
        "__RQ3_SEED_PROBE__"
    )

    ==

    rq3_question_seed(
        "webqsp",
        "__RQ3_SEED_PROBE__"
    )
)


print(
    "\nMatched RNG policy: FROZEN"
)

print(
    " base seed: 42"
)

print(
    " mapping  : SHA256(base_seed|dataset|question_id) mod (2^31-1)"
)

print(
    " RoG/AFP  : same question -> same seed"
)

print(
    " TEST seed selection by performance: NO"
)

# ==================================================================================================
# 14. RESOLVE FINAL VERIFIED RQ2 / ABLATION ARTIFACT ROOT
# ==================================================================================================

FINAL_ABLATION_SUMMARY_NAME_0 = (
    "final_test_ablation_summary_FINAL_VERIFIED.csv"
)


FINAL_ABLATION_RQ3_NAME_0 = (
    "final_test_ablation_rq3_inputs_FINAL_VERIFIED.csv"
)


FINAL_ABLATION_AUDIT_NAME_0 = (
    "feature_dimension_routing_audit_FINAL_VERIFIED.json"
)


FINAL_ABLATION_MANIFEST_NAME_0 = (
    "final_test_ablation_manifest_FINAL_VERIFIED.json"
)


_abl_root_candidates_0 = [

    Path(
        "/kaggle/working/"
        "step3_rq2_dev_v1/"
        "15_final_frozen_test/"
        "rq2_ablation_test_FINAL_VERIFIED"
    ),

    Path(
        "/kaggle/input/datasets/"
        "sabitahnaf/"
        "adapruner-kgqa-migration/"
        "step3_rq2_dev_v1/"
        "15_final_frozen_test/"
        "rq2_ablation_test_FINAL_VERIFIED"
    ),

    Path(
        "/kaggle/input/notebooks/"
        "mdsadmansamikhan/"
        "rog-ap/"
        "step3_rq2_dev_v1/"
        "15_final_frozen_test/"
        "rq2_ablation_test_FINAL_VERIFIED"
    ),
]


RQ3_AFP_ARTIFACT_ROOT = (
    None
)


for _root_0 in (
    _abl_root_candidates_0
):

    if (
        _root_0
        /
        FINAL_ABLATION_SUMMARY_NAME_0
    ).exists():

        RQ3_AFP_ARTIFACT_ROOT = (
            _root_0
        )

        break


if (
    RQ3_AFP_ARTIFACT_ROOT
    is None
):

    _matches_0 = list(

        Path(
            "/kaggle/input"
        ).rglob(
            FINAL_ABLATION_SUMMARY_NAME_0
        )
    )


    if (
        _matches_0
    ):

        RQ3_AFP_ARTIFACT_ROOT = (
            _matches_0[
                0
            ].parent
        )


assert (
    RQ3_AFP_ARTIFACT_ROOT
    is not None
), (
    "STOP: FINAL VERIFIED RQ2 ablation artifacts are unavailable. "
    "Attach/persist them; do not rerun or retune RQ2."
)


print(
    "\nFinal verified RQ2 artifact root: FOUND"
)

print(
    " ",
    RQ3_AFP_ARTIFACT_ROOT
)


# ==================================================================================================
# 15. LOAD FINAL VERIFIED RQ2 ARTIFACTS
# ==================================================================================================

RQ3_AFP_SUMMARY_PATH = (
    RQ3_AFP_ARTIFACT_ROOT
    /
    FINAL_ABLATION_SUMMARY_NAME_0
)


RQ3_AFP_RQ3_INPUT_PATH = (
    RQ3_AFP_ARTIFACT_ROOT
    /
    FINAL_ABLATION_RQ3_NAME_0
)


RQ3_AFP_AUDIT_PATH = (
    RQ3_AFP_ARTIFACT_ROOT
    /
    FINAL_ABLATION_AUDIT_NAME_0
)


RQ3_AFP_MANIFEST_PATH = (
    RQ3_AFP_ARTIFACT_ROOT
    /
    FINAL_ABLATION_MANIFEST_NAME_0
)


for _required_path_0 in [

    RQ3_AFP_SUMMARY_PATH,
    RQ3_AFP_RQ3_INPUT_PATH,
    RQ3_AFP_AUDIT_PATH,
    RQ3_AFP_MANIFEST_PATH,

]:

    assert (
        _required_path_0.exists()
    ), (
        f"Missing FINAL VERIFIED artifact: "
        f"{_required_path_0}"
    )


RQ3_AFP_SUMMARY_DF = (
    pd.read_csv(
        RQ3_AFP_SUMMARY_PATH
    )
)


with RQ3_AFP_AUDIT_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    RQ3_AFP_AUDIT = json.load(
        f
    )


with RQ3_AFP_MANIFEST_PATH.open(
    "r",
    encoding="utf-8"
) as f:

    RQ3_AFP_MANIFEST = json.load(
        f
    )


RQ3_AFP_MANIFEST_SHA256 = (
    sha256_file_rq3_0(
        RQ3_AFP_MANIFEST_PATH
    )
)


print(
    "\nFINAL VERIFIED RQ2 artifacts loaded: PASSED"
)

print(
    " manifest SHA:",
    RQ3_AFP_MANIFEST_SHA256
)


# ==================================================================================================
# 16. AFP SCIENTIFIC FREEZE
#     AUTHORITATIVE MANIFEST GATE + OPTIONAL CANONICAL JSON CROSS-CHECK
# ==================================================================================================

RQ3_EXPECTED_AFP_FREEZE_SHA256 = (
    "bf7ef1783aa96fe483dfcd9819393a7ec1c2bb9936a842d99e2a4ad085113116"
)


assert (
    RQ3_AFP_MANIFEST.get(
        "scientific_freeze_sha256"
    )
    ==
    RQ3_EXPECTED_AFP_FREEZE_SHA256
), (
    "STOP: FINAL VERIFIED ablation manifest carries "
    "the wrong AFP scientific freeze SHA."
)


RQ3_DISCOVERED_FREEZE_PATH = (
    None
)


_step3_root_0 = (
    RQ3_AFP_ARTIFACT_ROOT
    .parent
    .parent
)


_freeze_candidates_0 = [

    (
        _step3_root_0
        /
        "14_selector_ablations_and_final_freeze"
        /
        "final_afp_development_freeze.json"
    ),

    Path(
        "/kaggle/input/datasets/"
        "sabitahnaf/"
        "adapruner-kgqa-migration/"
        "step3_rq2_dev_v1/"
        "14_selector_ablations_and_final_freeze/"
        "final_afp_development_freeze.json"
    ),
]


for _p_0 in (
    _freeze_candidates_0
):

    if not (
        _p_0.exists()
    ):

        continue


    try:

        if (
            canonical_json_sha256_rq3_0(
                _p_0
            )
            ==
            RQ3_EXPECTED_AFP_FREEZE_SHA256
        ):

            RQ3_DISCOVERED_FREEZE_PATH = (
                _p_0
            )

            break


    except Exception:

        pass


print(
    "\nAFP scientific freeze identifier: PASSED"
)

print(
    " expected:",
    RQ3_EXPECTED_AFP_FREEZE_SHA256
)

print(
    " manifest:",
    RQ3_AFP_MANIFEST[
        "scientific_freeze_sha256"
    ]
)


if (
    RQ3_DISCOVERED_FREEZE_PATH
    is not None
):

    print(
        " freeze JSON:",
        RQ3_DISCOVERED_FREEZE_PATH
    )

    print(
        " canonical JSON SHA:",
        canonical_json_sha256_rq3_0(
            RQ3_DISCOVERED_FREEZE_PATH
        )
    )

    print(
        " independent canonical freeze gate: PASSED"
    )


else:

    print(
        " separate Cell-15B freeze JSON not found in this filesystem; "
        "manifest freeze gate remains authoritative."
    )


# ==================================================================================================
# 17. FINAL VERIFIED ROUTING / DIMENSION / LEAKAGE AUDIT
#     CORRECT FINAL SCHEMA
# ==================================================================================================

assert (
    int(
        RQ3_AFP_AUDIT[
            "full_feature_dim"
        ]
    )
    ==
    27
)


assert (
    RQ3_AFP_AUDIT[
        "uppercase_runtime_binding"
    ]
    ==
    "AFP_RUNTIME_SCORE_GROUP"
)


assert (
    RQ3_AFP_MANIFEST[
        "routing"
    ][
        "get_group_logits_binding_patched"
    ]
    ==
    "AFP_RUNTIME_SCORE_GROUP"
)


assert (
    RQ3_AFP_MANIFEST[
        "routing"
    ][
        "feature_scorer_dict_contract_preserved"
    ]
    is True
)


assert (
    RQ3_AFP_MANIFEST[
        "routing"
    ][
        "score_cache_cross_variant_leakage_possible"
    ]
    is False
)


assert (
    RQ3_AFP_AUDIT[
        "test_training"
    ]
    is False
)


assert (
    RQ3_AFP_AUDIT[
        "test_tuning"
    ]
    is False
)


assert (
    RQ3_AFP_AUDIT[
        "test_model_selection"
    ]
    is False
)


assert (
    RQ3_AFP_MANIFEST[
        "leakage"
    ][
        "test_training"
    ]
    is False
)


assert (
    RQ3_AFP_MANIFEST[
        "leakage"
    ][
        "test_tuning"
    ]
    is False
)


assert (
    RQ3_AFP_MANIFEST[
        "leakage"
    ][
        "test_model_selection"
    ]
    is False
)


assert (
    RQ3_AFP_MANIFEST[
        "leakage"
    ][
        "gold_used_by_scorer"
    ]
    is False
)


assert (
    RQ3_AFP_MANIFEST[
        "leakage"
    ][
        "gold_used_by_selector"
    ]
    is False
)


_EXPECTED_RQ2_FEATURE_DIMS_0 = {

    "webqsp|w/o Semantic":
        [19],

    "webqsp|w/o Path Context":
        [20],

    "webqsp|w/o Structural":
        [19],

    "webqsp|w/o Progress":
        [23],

    "cwq|w/o Semantic":
        [19],

    "cwq|w/o Path Context":
        [20],

    "cwq|w/o Structural":
        [19],

    "cwq|w/o Progress":
        [23],
}


_observed_records_0 = (
    RQ3_AFP_AUDIT[
        "observed"
    ]
)


for _key_0, _expected_dims_0 in (
    _EXPECTED_RQ2_FEATURE_DIMS_0.items()
):

    assert (
        _key_0
        in
        _observed_records_0
    )


    _rec_0 = (
        _observed_records_0[
            _key_0
        ]
    )


    assert (
        int(
            _rec_0[
                "score_calls"
            ]
        )
        >
        0
    )


    assert (
        list(
            _rec_0[
                "observed_dims"
            ]
        )
        ==
        _expected_dims_0
    )


    assert (
        _rec_0[
            "logits_differ_from_full"
        ]
        is True
    )


_EXPECTED_VARIANT_DIMS_0 = {

    "w/o Semantic":
        19,

    "w/o Path Context":
        20,

    "w/o Structural":
        19,

    "w/o Progress":
        23,
}


for _variant_0, _expected_dim_0 in (
    _EXPECTED_VARIANT_DIMS_0.items()
):

    assert (
        int(
            RQ3_AFP_AUDIT[
                "feature_variants"
            ][
                _variant_0
            ][
                "expected_input_dim"
            ]
        )
        ==
        _expected_dim_0
    )


print(
    "\nFinal RQ2 routing/dimension audit: PASSED"
)

print(
    " uppercase runtime route : AFP_RUNTIME_SCORE_GROUP"
)

print(
    " full Feature-v2 dim     : 27"
)

print(
    " WebQSP reduced dims     : 19 / 20 / 19 / 23"
)

print(
    " CWQ reduced dims        : 19 / 20 / 19 / 23"
)

print(
    " reduced scorer calls    : > 0 for all feature ablations"
)

print(
    " logits differ from full : YES for all feature ablations"
)

print(
    " TEST training/tuning/model selection: NO"
)


# ==================================================================================================
# 18. HARD-GATE FINAL VERIFIED Full-AFP TEST CONTROLS
# ==================================================================================================

_summary_norm_0 = (
    RQ3_AFP_SUMMARY_DF.copy()
)


_summary_norm_0[
    "_dataset"
] = (
    _summary_norm_0[
        "dataset"
    ]
    .astype(
        str
    )
    .str
    .strip()
    .str
    .lower()
)


_summary_norm_0[
    "_variant"
] = (
    _summary_norm_0[
        "variant"
    ]
    .astype(
        str
    )
    .str
    .strip()
)


EXPECTED_FULL_AFP_TEST_0 = {

    "webqsp": {

        "edges_examined":
            2_268_901,

        "reachable_questions":
            1_358,

        "rog_reachable_questions":
            1_369,

        "retrieved_paths":
            31_487,

        "ssr":
            (
                1.0
                -
                (
                    2_268_901
                    /
                    2_320_856
                )
            ),
    },


    "cwq": {

        "edges_examined":
            5_525_531,

        "reachable_questions":
            2_391,

        "rog_reachable_questions":
            2_422,

        "retrieved_paths":
            194_115,

        "ssr":
            (
                1.0
                -
                (
                    5_525_531
                    /
                    5_819_125
                )
            ),
    },
}


RQ3_FULL_AFP_ROWS = {}


for _dataset_0, _expected_0 in (
    EXPECTED_FULL_AFP_TEST_0.items()
):

    _rows_0 = _summary_norm_0[

        (
            _summary_norm_0[
                "_dataset"
            ]
            ==
            _dataset_0
        )

        &

        (
            _summary_norm_0[
                "_variant"
            ]
            ==
            "Full AFP"
        )
    ]


    assert (
        len(
            _rows_0
        )
        ==
        1
    ), (
        f"Expected one Full AFP row for {_dataset_0}."
    )


    _row_0 = (
        _rows_0.iloc[
            0
        ]
    )


    for _integer_key_0 in [

        "edges_examined",
        "reachable_questions",
        "rog_reachable_questions",
        "retrieved_paths",

    ]:

        assert (
            int(
                _row_0[
                    _integer_key_0
                ]
            )
            ==
            int(
                _expected_0[
                    _integer_key_0
                ]
            )
        )


    assert (
        abs(
            float(
                _row_0[
                    "ssr"
                ]
            )
            -
            float(
                _expected_0[
                    "ssr"
                ]
            )
        )
        <
        1e-10
    )


    _calculated_ar_0 = (

        int(
            _row_0[
                "reachable_questions"
            ]
        )

        /

        int(
            _row_0[
                "rog_reachable_questions"
            ]
        )
    )


    assert (
        abs(
            float(
                _row_0[
                    "answer_retention"
                ]
            )
            -
            _calculated_ar_0
        )
        <
        1e-10
    )


    RQ3_FULL_AFP_ROWS[
        _dataset_0
    ] = (
        _row_0
    )


print(
    "\nFINAL VERIFIED Full-AFP control gates: PASSED"
)


for _dataset_0 in [

    "webqsp",
    "cwq",

]:

    _row_0 = (
        RQ3_FULL_AFP_ROWS[
            _dataset_0
        ]
    )


    print(

        f" {_dataset_0:7s} | "

        f"edges="
        f"{int(_row_0['edges_examined']):,} | "

        f"SSR="
        f"{float(_row_0['ssr']):.6f} | "

        f"reach="
        f"{int(_row_0['reachable_questions'])}/"
        f"{int(_row_0['rog_reachable_questions'])} | "

        f"AR="
        f"{float(_row_0['answer_retention']):.6f} | "

        f"paths="
        f"{int(_row_0['retrieved_paths']):,}"
    )


# ==================================================================================================
# 19. RESOLVE EXACT Full-AFP RETRIEVED-PATH FILES
# ==================================================================================================

def resolve_retrieved_path_file_0(
    row
):

    raw_value = row.get(
        "retrieved_paths_file",
        None
    )


    assert (
        raw_value is not None
        and
        not pd.isna(
            raw_value
        )
    ), (
        "Full AFP row is missing retrieved_paths_file."
    )


    original = Path(
        str(
            raw_value
        )
    )


    if original.exists():

        return original


    relocated = (

        RQ3_AFP_ARTIFACT_ROOT

        /

        "retrieved_paths"

        /

        original.name
    )


    if relocated.exists():

        return relocated


    matches = list(

        RQ3_AFP_ARTIFACT_ROOT.rglob(
            original.name
        )
    )


    assert (
        len(
            matches
        )
        ==
        1
    ), (
        f"Could not uniquely resolve {raw_value}; "
        f"matches={matches}"
    )


    return matches[
        0
    ]


RQ3_FULL_AFP_PATH_FILES = {}


for _dataset_0 in [

    "webqsp",
    "cwq",

]:

    _path_0 = (
        resolve_retrieved_path_file_0(
            RQ3_FULL_AFP_ROWS[
                _dataset_0
            ]
        )
    )


    assert (
        _path_0.exists()
        and
        _path_0.stat().st_size
        >
        0
    )


    RQ3_FULL_AFP_PATH_FILES[
        _dataset_0
    ] = (
        _path_0
    )


print(
    "\nFull-AFP retrieved-path artifacts: FOUND"
)


for _dataset_0, _path_0 in (
    RQ3_FULL_AFP_PATH_FILES.items()
):

    print(
        f" {_dataset_0:7s}: {_path_0}"
    )

    print(
        "          SHA256:",
        sha256_file_rq3_0(
            _path_0
        )
    )


# ==================================================================================================
# 20. NO-TEST-GENERATION / NO-LEAKAGE GATE
# ==================================================================================================

RQ3_TEST_ANSWERS_GENERATED_0 = (
    0
)


assert (
    RQ3_TEST_ANSWERS_GENERATED_0
    ==
    0
)


print(
    "\nRQ3-0 leakage gate: PASSED"
)

print(
    " TEST answers generated : 0"
)

print(
    " TEST QA metrics scored : 0"
)

print(
    " reasoner/seed tuning   : NO"
)

print(
    " AFP modification        : NO"
)

print(
    " retrieval rerun         : NO"
)


# ==================================================================================================
# 21. SAVE RQ3 REASONER CONTRACT MANIFEST
# ==================================================================================================

RQ3_WORK_ROOT = Path(
    "/kaggle/working/"
    "step4_rq3_llm_reasoning_v1"
)


RQ3_CONTRACT_DIR = (
    RQ3_WORK_ROOT
    /
    "00_reasoner_contract"
)


RQ3_CONTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


RQ3_CONTRACT_MANIFEST_PATH = (
    RQ3_CONTRACT_DIR
    /
    "rq3_reasoner_contract_manifest.json"
)


RQ3_REASONER_CONTRACT_MANIFEST = {

    "stage":
        "RQ3-0",

    "status":
        "contract_frozen_before_test_generation",

    "test_generation_performed":
        False,

    "test_tuning_performed":
        False,

    "test_model_selection_performed":
        False,

    "afp_retraining_performed_in_rq3":
        False,

    "retrieval_rerun_performed":
        False,


    "software_environment": {

        "transformers":
            transformers.__version__,

        "tokenizers":
            tokenizers.__version__,

        "sentencepiece":
            sentencepiece.__version__,

        "torch":
            torch.__version__,

        "gpu":
            RQ3_GPU_NAME,
    },


    "tokenizer": {

        "model":
            str(
                RQ3_TOKENIZER.name_or_path
            ),

        "class":
            type(
                RQ3_TOKENIZER
            ).__name__,

        "module":
            type(
                RQ3_TOKENIZER
            ).__module__,

        "is_fast":
            bool(
                RQ3_TOKENIZER.is_fast
            ),

        "sentencepiece_backed":
            bool(
                hasattr(
                    RQ3_TOKENIZER,
                    "sp_model"
                )
            ),

        "fidelity_probes_exact":
            True,
    },


    "rog_source": {

        "repository":
            str(
                ROG_REPO_RQ3_0
            ),

        "expected_git_commit":
            EXPECTED_ROG_COMMIT_0,

        "observed_git_commit":
            RQ3_ROG_GIT_COMMIT_0,

        "prompt_sha256":
            RQ3_PROMPT_SHA_0,

        "prompt_builder_sha256":
            RQ3_PROMPT_BUILDER_SHA_0,

        "evaluator_sha256":
            RQ3_EVALUATOR_SHA_0,

        "predict_answer_sha256":
            RQ3_PREDICT_ANSWER_SHA_0,

        "llama_wrapper_sha256":
            RQ3_LLAMA_WRAPPER_SHA_0,
    },


    "prompt_contract": {

        "prompt_path":
            str(
                RQ3_REASONER_PROMPT_PATH
            ),

        "add_rule":
            True,

        "use_true":
            False,

        "cot":
            False,

        "explain":
            False,

        "use_random":
            False,

        "each_line":
            False,

        "maximum_token":
            int(
                RQ3_PROMPT_MAXIMUM_TOKEN
            ),

        "token_counter":
            (
                "len("
                "slow_LlamaTokenizer."
                "tokenize(text)"
                ")"
            ),

        "oversize_behavior":
            (
                "PromptBuilder.check_prompt_length "
                "with path shuffling before "
                "incremental inclusion"
            ),

        "native_process_input_regenerates_graph_paths":
            True,

        "rq3_direct_retrieved_path_injection_required":
            True,
    },


    "reasoner": {

        "model_id":
            RQ3_REASONER_MODEL_ID,

        "model_class":
            type(
                RQ3_MODEL
            ).__name__,

        "parameter_dtypes":
            RQ3_MODEL_PARAM_DTYPES,

        "primary_device":
            RQ3_PRIMARY_DEVICE,

        "max_position_embeddings":
            int(
                RQ3_MODEL_MAX_POSITION
            ),
    },


    "generation_contract": {

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "explicit_overrides":
            sorted(
                RQ3_GENERATION_KWARGS.keys()
            ),

        "temperature_overridden":
            False,

        "top_p_overridden":
            False,

        "top_k_overridden":
            False,

        "num_beams_overridden":
            False,

        "model_generation_defaults":
            RQ3_MODEL_GENERATION_DEFAULTS,
    },


    "matched_rng_policy": {

        "base_seed":
            int(
                RQ3_FULL_TEST_BASE_SEED
            ),

        "mapping":
            (
                "SHA256("
                "base_seed|dataset|question_id"
                ") mod (2^31-1)"
            ),

        "same_question_same_seed_across_methods":
            True,

        "seed_selected_using_test_performance":
            False,

        "python_random_seeded":
            True,

        "numpy_seeded":
            True,

        "torch_seeded":
            True,

        "cuda_seeded":
            True,
    },


    "evaluator_contract": {

        "source_sha256":
            RQ3_EVALUATOR_SHA_0,

        "functions":
            _required_eval_functions_0,

        "test_predictions_scored_in_rq3_0":
            False,
    },


    "upstream_afp": {

        "scientific_freeze_sha256":
            RQ3_EXPECTED_AFP_FREEZE_SHA256,

        "discovered_freeze_artifact":
            (
                str(
                    RQ3_DISCOVERED_FREEZE_PATH
                )
                if
                RQ3_DISCOVERED_FREEZE_PATH
                else
                None
            ),

        "final_verified_root":
            str(
                RQ3_AFP_ARTIFACT_ROOT
            ),

        "final_verified_manifest":
            str(
                RQ3_AFP_MANIFEST_PATH
            ),

        "final_verified_manifest_sha256":
            RQ3_AFP_MANIFEST_SHA256,

        "runtime_binding":
            "AFP_RUNTIME_SCORE_GROUP",

        "full_feature_dim":
            27,

        "feature_dimensions_verified": {

            "minus_semantic":
                19,

            "minus_path_context":
                20,

            "minus_structural":
                19,

            "minus_progress":
                23,
        },

        "test_training":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,
    },


    "full_afp_test_controls": {

        _dataset_0: {

            "edges_examined":
                int(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "edges_examined"
                    ]
                ),

            "ssr":
                float(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "ssr"
                    ]
                ),

            "reachable_questions":
                int(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "reachable_questions"
                    ]
                ),

            "rog_reachable_questions":
                int(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "rog_reachable_questions"
                    ]
                ),

            "answer_retention":
                float(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "answer_retention"
                    ]
                ),

            "retrieved_paths":
                int(
                    RQ3_FULL_AFP_ROWS[
                        _dataset_0
                    ][
                        "retrieved_paths"
                    ]
                ),

            "retrieved_paths_file":
                str(
                    RQ3_FULL_AFP_PATH_FILES[
                        _dataset_0
                    ]
                ),

            "retrieved_paths_sha256":
                sha256_file_rq3_0(
                    RQ3_FULL_AFP_PATH_FILES[
                        _dataset_0
                    ]
                ),
        }

        for _dataset_0 in [

            "webqsp",
            "cwq",

        ]
    },
}


with RQ3_CONTRACT_MANIFEST_PATH.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(

        RQ3_REASONER_CONTRACT_MANIFEST,

        f,

        indent=2,

        ensure_ascii=False,
    )


RQ3_CONTRACT_MANIFEST_SHA256 = (
    sha256_file_rq3_0(
        RQ3_CONTRACT_MANIFEST_PATH
    )
)


assert (
    RQ3_CONTRACT_MANIFEST_PATH.exists()
    and
    RQ3_CONTRACT_MANIFEST_PATH.stat().st_size
    >
    0
)


assert (
    sha256_file_rq3_0(
        RQ3_CONTRACT_MANIFEST_PATH
    )
    ==
    RQ3_CONTRACT_MANIFEST_SHA256
)


# ==================================================================================================
# 22. EXPOSE LIVE OBJECTS FOR RQ3-1
# ==================================================================================================

RQ3_REASONER_CONTRACT_COMPLETE = (
    True
)


RQ3_EVALUATOR_FUNCTIONS = {

    "normalize":
        RQ3_EVAL_NORMALIZE,

    "match":
        RQ3_EVAL_MATCH,

    "eval_acc":
        RQ3_EVAL_ACC,

    "eval_hit":
        RQ3_EVAL_HIT,

    "eval_f1":
        RQ3_EVAL_F1,

    "extract_topk_prediction":
        RQ3_EXTRACT_TOPK,
}


# ==================================================================================================
# 23. FINAL REPORT
# ==================================================================================================

print(
    "\n"
    +
    "=" * 140
)


print(
    "RQ3-0 FINAL REASONER CONTRACT"
)


print(
    "=" * 140
)


print(
    "Exact slow RoG tokenizer:        PASSED"
)

print(
    "Tokenizer fidelity probes:       PASSED"
)

print(
    "RoG source SHA gates:            PASSED"
)

print(
    "PromptBuilder source/contract:   PASSED"
)

print(
    "Native 3996-token prompt budget: PASSED"
)

print(
    "Internal-BFS hazard identified:  YES"
)

print(
    "Exact RoG evaluator source:      PASSED"
)

print(
    "RoG 7B FP16 reasoner:            PASSED"
)

print(
    "Generation max_new_tokens=512:   FROZEN"
)

print(
    "Generation do_sample=True:       FROZEN"
)

print(
    "Extra generation overrides:      NONE"
)

print(
    "Matched per-question RNG:        FROZEN"
)

print(
    "AFP scientific freeze:           PASSED"
)

print(
    "Final verified AFP artifacts:    PASSED"
)

print(
    "Correct uppercase AFP route:     PASSED"
)

print(
    "Reduced feature dims:            19 / 20 / 19 / 23"
)

print(
    "TEST answers generated:          0"
)

print(
    "TEST tuning/model selection:     NO"
)

print(
    "AFP retraining in RQ3:           NO"
)


print(
    "\nReasoner contract manifest:"
)

print(
    RQ3_CONTRACT_MANIFEST_PATH
)

print(
    "Manifest SHA256:",
    RQ3_CONTRACT_MANIFEST_SHA256
)


print(
    "\nRQ3_REASONER_CONTRACT_COMPLETE:",
    RQ3_REASONER_CONTRACT_COMPLETE
)


print(
    "\nNEXT: RQ3-1 — frozen RoG/AFP retrieved-evidence schema "
    "+ direct-prompt fidelity audit"
)


print(
    "=" * 140
)


## 3. RQ3-0 Frozen-Manifest Recovery Audit

Recovery-only cell used to restore the historically frozen RQ3-0
manifest when required after migration or kernel recovery.


In [ ]:

from pathlib import Path
import hashlib
import zipfile
import json
import shutil
import io

EXPECTED = (
    "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8"
)

CURRENT = Path(
    "/kaggle/working/step4_rq3_llm_reasoning_v1/"
    "00_reasoner_contract/rq3_reasoner_contract_manifest.json"
)


def sha_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha_file(p):
    return sha_bytes(Path(p).read_bytes())


print("=" * 100)
print("RQ3-0 FROZEN MANIFEST RECOVERY AUDIT")
print("=" * 100)

print("\nExpected frozen SHA:")
print(EXPECTED)

if CURRENT.exists():
    print("\nCurrent regenerated manifest:")
    print(CURRENT)
    print("Current SHA:")
    print(sha_file(CURRENT))
else:
    print("\nCurrent manifest does not exist.")


# =====================================================================
# 1. Search ordinary files under Kaggle INPUT
# =====================================================================

found_bytes = None
found_source = None

normal_matches = list(
    Path("/kaggle/input").rglob(
        "rq3_reasoner_contract_manifest.json"
    )
)

print(
    "\nNormal candidate manifests:",
    len(normal_matches)
)

for p in normal_matches:

    try:

        h = sha_file(p)

        print(
            " ",
            h,
            "|",
            p
        )

        if h == EXPECTED:

            found_bytes = p.read_bytes()

            found_source = str(
                p
            )

            break

    except Exception as e:

        print(
            " unreadable:",
            p,
            e
        )


# =====================================================================
# 2. Search every ZIP under INPUT and WORKING
# =====================================================================

if found_bytes is None:

    zip_paths = (
        list(
            Path("/kaggle/input").rglob(
                "*.zip"
            )
        )
        +
        list(
            Path("/kaggle/working").rglob(
                "*.zip"
            )
        )
    )

    print(
        "\nZIP archives to inspect:",
        len(zip_paths)
    )

    for zp in zip_paths:

        try:

            with zipfile.ZipFile(
                zp,
                "r"
            ) as z:

                candidates = [
                    n
                    for n in z.namelist()
                    if Path(n).name
                    ==
                    "rq3_reasoner_contract_manifest.json"
                ]

                for member in candidates:

                    b = z.read(
                        member
                    )

                    h = sha_bytes(
                        b
                    )

                    print(
                        " ",
                        h,
                        "| ZIP:",
                        zp,
                        "| member:",
                        member
                    )

                    if h == EXPECTED:

                        found_bytes = b

                        found_source = (
                            f"{zp} :: {member}"
                        )

                        break

            if found_bytes is not None:
                break

        except zipfile.BadZipFile:
            pass


# =====================================================================
# 3. Result
# =====================================================================

print(
    "\n"
    +
    "=" * 100
)


if found_bytes is None:

    print(
        "ORIGINAL FROZEN RQ3-0 MANIFEST: NOT FOUND"
    )

    print(
        "Do NOT modify RQ3-1 expected SHA."
    )


else:

    print(
        "ORIGINAL FROZEN RQ3-0 MANIFEST: FOUND"
    )

    print(
        "Source:",
        found_source
    )

    frozen_manifest = json.loads(
        found_bytes.decode(
            "utf-8"
        )
    )

    current_manifest = (
        json.loads(
            CURRENT.read_text(
                encoding="utf-8"
            )
        )
        if CURRENT.exists()
        else None
    )

    # Preserve a diagnostic copy of the newly-generated manifest.
    if CURRENT.exists():

        backup = CURRENT.with_name(
            "rq3_reasoner_contract_manifest_CURRENT_REGENERATED.json"
        )

        shutil.copy2(
            CURRENT,
            backup
        )

        print(
            "\nCurrent regenerated manifest backed up:"
        )

        print(
            backup
        )

    # Restore ORIGINAL byte-exact frozen manifest.
    CURRENT.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    CURRENT.write_bytes(
        found_bytes
    )

    assert (
        sha_file(
            CURRENT
        )
        ==
        EXPECTED
    )

    # Refresh live variable expected by RQ3-1
    RQ3_CONTRACT_MANIFEST_PATH = CURRENT

    RQ3_CONTRACT_MANIFEST_SHA256 = (
        EXPECTED
    )

    print(
        "\nFROZEN MANIFEST RESTORED"
    )

    print(
        "Path:",
        CURRENT
    )

    print(
        "SHA :",
        sha_file(
            CURRENT
        )
    )

    print(
        "\nRQ3_CONTRACT_MANIFEST_SHA256:"
    )

    print(
        RQ3_CONTRACT_MANIFEST_SHA256
    )


print(
    "=" * 100
)


## 4. RQ3-1 — Frozen Evidence and Prompt Fidelity

Reconstructs the frozen RoG and AdaPruner retrieved evidence and verifies
direct-prompt fidelity without generating TEST answers.


In [ ]:

from pathlib import Path
from collections import Counter, deque
import ast, gzip, hashlib, io, json, random, types
import numpy as np
import pandas as pd
import networkx as nx

print("=" * 150)
print("RQ3-1 — FROZEN RoG/AFP RETRIEVED-EVIDENCE + DIRECT-PROMPT FIDELITY AUDIT — FINAL")
print("=" * 150)

# ==================================================================================================
# 0. HARD RQ3-0 GATE
# ==================================================================================================
EXPECTED_RQ3_0_MANIFEST_SHA256 = "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8"
assert globals().get("RQ3_REASONER_CONTRACT_COMPLETE", False) is True
for _name in [
    "RQ3_TOKENIZER", "RQ3_MODEL", "RQ3_PROMPT_BUILDER", "RQ3_PROMPT_MAXIMUM_TOKEN",
    "RQ3_MAX_NEW_TOKENS", "RQ3_GENERATION_KWARGS", "RQ3_FULL_AFP_PATH_FILES",
    "RQ3_FULL_TEST_BASE_SEED", "rq3_set_question_seed", "ROG_REPO_RQ3_0",
    "RQ3_CONTRACT_MANIFEST_PATH", "RQ3_CONTRACT_MANIFEST_SHA256",
]:
    assert _name in globals(), f"STOP: missing RQ3-0 object: {_name}"
assert str(RQ3_CONTRACT_MANIFEST_SHA256) == EXPECTED_RQ3_0_MANIFEST_SHA256
assert Path(RQ3_CONTRACT_MANIFEST_PATH).exists()
assert type(RQ3_TOKENIZER).__name__ == "LlamaTokenizer" and RQ3_TOKENIZER.is_fast is False
assert int(RQ3_PROMPT_MAXIMUM_TOKEN) == 3996
assert int(RQ3_MAX_NEW_TOKENS) == 512
assert RQ3_GENERATION_KWARGS == {"max_new_tokens": 512, "do_sample": True}
RQ3_1_LLM_GENERATION_CALLS = 0
print("\nRQ3-0 contract gate: PASSED")
print(" RQ3-0 manifest SHA:", EXPECTED_RQ3_0_MANIFEST_SHA256)
print(" TEST LLM generations:", RQ3_1_LLM_GENERATION_CALLS)

# ==================================================================================================
# 1. HELPERS
# ==================================================================================================
def sha256_file_rq3_1(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b: break
            h.update(b)
    return h.hexdigest()

def sha256_text_rq3_1(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def rows_sha256_rq3_1(rows):
    h = hashlib.sha256()
    for row in rows:
        h.update(json.dumps(row, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8"))
        h.update(b"\n")
    return h.hexdigest()

def read_jsonl_rq3_1(path):
    out = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip(): out.append(json.loads(line))
    return out

def read_gzip_jsonl_rq3_1(path):
    out = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip(): out.append(json.loads(line))
    return out

def write_deterministic_gzip_jsonl_rq3_1(rows, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as raw:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz:
            with io.TextIOWrapper(gz, encoding="utf-8", newline="\n") as txt:
                for row in rows:
                    txt.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
    return path

def first_present_rq3_1(rec, names, required=True):
    for name in names:
        if name in rec: return rec[name]
    if required:
        raise AssertionError(f"Missing one of {names}; keys={sorted(rec.keys())}")
    return None

# ==================================================================================================
# 2. RECOVER EXACT NATIVE RoG GRAPH/PATH FUNCTIONS
# ==================================================================================================
ROG_GRAPH_UTILS_PATH_1 = Path(ROG_REPO_RQ3_0) / "src" / "utils" / "graph_utils.py"
ROG_UTILS_PATH_1 = Path(ROG_REPO_RQ3_0) / "src" / "utils" / "utils.py"
assert ROG_GRAPH_UTILS_PATH_1.exists() and ROG_UTILS_PATH_1.exists()
_graph_source_1 = ROG_GRAPH_UTILS_PATH_1.read_text(encoding="utf-8")
_utils_source_1 = ROG_UTILS_PATH_1.read_text(encoding="utf-8")

def extract_function_source_rq3_1(source, name):
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            s = ast.get_source_segment(source, node); assert s; return s
    raise AssertionError(f"Could not recover function {name}")

_build_graph_source_1 = extract_function_source_rq3_1(_graph_source_1, "build_graph")
_bfs_source_1 = extract_function_source_rq3_1(_graph_source_1, "bfs_with_rule")
_path_to_string_source_1 = extract_function_source_rq3_1(_utils_source_1, "path_to_string")
_graph_ns_1 = {"__builtins__": __builtins__, "nx": nx, "deque": deque}
exec(_build_graph_source_1, _graph_ns_1); exec(_bfs_source_1, _graph_ns_1)
_utils_ns_1 = {"__builtins__": __builtins__}; exec(_path_to_string_source_1, _utils_ns_1)
RQ3_NATIVE_BUILD_GRAPH = _graph_ns_1["build_graph"]
RQ3_NATIVE_BFS_WITH_RULE = _graph_ns_1["bfs_with_rule"]
RQ3_NATIVE_PATH_TO_STRING = _utils_ns_1["path_to_string"]
print("\nExact native RoG utilities recovered: PASSED")
print(" build_graph function SHA :", sha256_text_rq3_1(_build_graph_source_1))
print(" bfs_with_rule function SHA:", sha256_text_rq3_1(_bfs_source_1))
print(" path_to_string SHA        :", sha256_text_rq3_1(_path_to_string_source_1))

# PromptBuilder reconstructed in RQ3-0 had an empty utils namespace; bridge only exact native funcs.
RQ3_PROMPT_UTILS_NAMESPACE_1 = types.SimpleNamespace(
    build_graph=RQ3_NATIVE_BUILD_GRAPH,
    bfs_with_rule=RQ3_NATIVE_BFS_WITH_RULE,
    path_to_string=RQ3_NATIVE_PATH_TO_STRING,
)
RQ3_PROMPT_BUILDER.process_input.__globals__["utils"] = RQ3_PROMPT_UTILS_NAMESPACE_1
RQ3_PROMPT_BUILDER.apply_rules.__globals__["utils"] = RQ3_PROMPT_UTILS_NAMESPACE_1
print("PromptBuilder audit-only native utils bridge: PASSED")

# ==================================================================================================
# 3. FROZEN TEST PLANNER ARTIFACT RESOLUTION BY FINAL VERIFIED AFP PROVENANCE
# ==================================================================================================
REQUIRED_PLAN_FIELDS_RQ3_1 = {"id", "question", "q_entity", "a_entity", "graph", "predicted_paths"}
EXPECTED_TEST_N_RQ3_1 = {"webqsp": 1628, "cwq": 3531}
RQ3_PLAN_SOURCE_AUDIT_ROWS = []

def planning_rows_valid_rq3_1(rows, expected_n):
    if not isinstance(rows, list) or len(rows) != expected_n: return False
    for rec in rows:
        if not isinstance(rec, dict): return False
        if not REQUIRED_PLAN_FIELDS_RQ3_1.issubset(rec): return False
        if not isinstance(rec["predicted_paths"], list): return False
    return True

def discover_frozen_test_plans_rq3_1(dataset_name):
    """Choose the canonical planner artifact ONLY by exact alignment with FINAL VERIFIED AFP capture."""
    dataset_name = str(dataset_name).strip().lower(); assert dataset_name in {"webqsp", "cwq"}
    expected_n = EXPECTED_TEST_N_RQ3_1[dataset_name]
    basename = f"planning_{dataset_name}_test.jsonl"

    # ---- frozen AFP provenance anchor ----
    capture_path = Path(RQ3_FULL_AFP_PATH_FILES[dataset_name]); assert capture_path.exists()
    capture_sha = sha256_file_rq3_1(capture_path)
    with Path(RQ3_CONTRACT_MANIFEST_PATH).open("r", encoding="utf-8") as f:
        m0 = json.load(f)
    expected_capture_sha = m0["full_afp_test_controls"][dataset_name]["retrieved_paths_sha256"]
    assert capture_sha == expected_capture_sha
    raw_capture = read_gzip_jsonl_rq3_1(capture_path); assert raw_capture
    capture_plan_map = {}
    for rec in raw_capture:
        qid = str(first_present_rq3_1(rec, ["question_id", "qid", "id"]))
        pi = int(first_present_rq3_1(rec, ["plan_index", "plan_idx"]))
        plan = first_present_rq3_1(rec, ["plan", "predicted_path", "relation_plan"])
        assert isinstance(plan, list)
        key = (qid, pi); assert key not in capture_plan_map
        capture_plan_map[key] = list(plan)
    capture_keys = set(capture_plan_map)
    print(f"\n{dataset_name.upper()} FINAL VERIFIED AFP provenance anchor:")
    print(" capture path:", capture_path)
    print(" capture SHA :", capture_sha)
    print(" plan records:", len(capture_plan_map))

    # ---- exact canonical basename only; permanently exclude quarantined v5 artifacts ----
    paths = [
        Path("/kaggle/working/step2_rq1_test") / basename,
        Path("/kaggle/input/datasets/sabitahnaf/adapruner-kgqa-migration/step2_rq1_test") / basename,
        Path("/kaggle/input/notebooks/mdsadmansamikhan/rog-ap/step2_rq1_test") / basename,
    ]
    for root in [
        Path("/kaggle/working"),
        Path("/kaggle/input/datasets/sabitahnaf/adapruner-kgqa-migration"),
        Path("/kaggle/input/notebooks/mdsadmansamikhan/rog-ap"),
    ]:
        if root.exists(): paths.extend(root.rglob(basename))
    unique, seen = [], set()
    for p in paths:
        p = Path(p)
        if not p.exists() or p.name != basename: continue
        low = str(p).lower()
        if any(x in low for x in ["invalid_transformers5", ".invalid_transformers5", "/invalid_", "\\invalid_"]):
            continue
        try: key = str(p.resolve())
        except Exception: key = str(p)
        if key in seen: continue
        seen.add(key); unique.append(p)
    assert unique, f"STOP: no eligible canonical {dataset_name.upper()} planner artifact found."

    # ---- audit every candidate against capture's (qid, plan_index, exact relation plan) ----
    candidates = []
    for p in unique:
        rows = read_jsonl_rq3_1(p)
        assert planning_rows_valid_rq3_1(rows, expected_n), f"Invalid planner artifact: {p}"
        ids = [str(r["id"]) for r in rows]
        assert len(ids) == len(set(ids)) == expected_n
        planner_map = {}
        for row in rows:
            qid = str(row["id"])
            for pi, plan in enumerate(row["predicted_paths"]):
                assert isinstance(plan, list)
                if plan: planner_map[(qid, int(pi))] = list(plan)
        planner_keys = set(planner_map)
        missing = planner_keys - capture_keys
        unexpected = capture_keys - planner_keys
        common = planner_keys & capture_keys
        relation_mismatch = [k for k in common if planner_map[k] != capture_plan_map[k]]
        exact = not missing and not unexpected and not relation_mismatch
        scientific_core = [{
            "id": r["id"], "question": r["question"], "q_entity": r["q_entity"],
            "a_entity": r["a_entity"], "graph": r["graph"], "predicted_paths": r["predicted_paths"],
        } for r in rows]
        rec = {
            "dataset": dataset_name, "path": p, "rows": rows,
            "file_sha": sha256_file_rq3_1(p), "content_sha": rows_sha256_rq3_1(rows),
            "scientific_core_sha": rows_sha256_rq3_1(scientific_core),
            "candidate_nonempty_plans": len(planner_keys), "capture_plan_records": len(capture_keys),
            "matched_plan_records": len(common) - len(relation_mismatch),
            "missing_capture_keys": len(missing), "unexpected_capture_keys": len(unexpected),
            "relation_mismatches": len(relation_mismatch), "exact_capture_alignment": bool(exact),
        }
        candidates.append(rec)
        RQ3_PLAN_SOURCE_AUDIT_ROWS.append({k: (str(v) if isinstance(v, Path) else v)
                                           for k, v in rec.items() if k != "rows"})

    print(f"\n{dataset_name.upper()} canonical planner candidate audit:")
    for rec in candidates:
        print("\n ", rec["path"])
        print("   file SHA                 :", rec["file_sha"])
        print("   content SHA              :", rec["content_sha"])
        print("   scientific-core SHA      :", rec["scientific_core_sha"])
        print("   candidate non-empty plans:", rec["candidate_nonempty_plans"])
        print("   AFP capture plan records :", rec["capture_plan_records"])
        print("   matched plan records     :", rec["matched_plan_records"])
        print("   missing capture keys     :", rec["missing_capture_keys"])
        print("   unexpected capture keys  :", rec["unexpected_capture_keys"])
        print("   relation mismatches      :", rec["relation_mismatches"])
        print("   EXACT AFP alignment      :", rec["exact_capture_alignment"])

    exact = [r for r in candidates if r["exact_capture_alignment"]]
    assert exact, (
        f"STOP: no {dataset_name.upper()} canonical planner artifact aligns with FINAL VERIFIED AFP capture. "
        "Do NOT choose manually or rerun planning."
    )
    # If >1 exact physical copy survives, scientific content must be identical.
    assert len({r["scientific_core_sha"] for r in exact}) == 1, (
        f"STOP: multiple {dataset_name.upper()} planner artifacts align with AFP but differ scientifically."
    )
    working = [r for r in exact if str(r["path"]).startswith("/kaggle/working/")]
    pool = working if working else exact
    chosen = sorted(pool, key=lambda r: str(r["path"]))[0]
    source = ("final_verified_afp_provenance:/kaggle/working" if working
              else "final_verified_afp_provenance:persisted-copy")
    print(f"\n{dataset_name.upper()} AUTHORITATIVE frozen TEST planning artifact: PASSED")
    print(" selected path  :", chosen["path"])
    print(" source         :", source)
    print(" content SHA    :", chosen["content_sha"])
    print(" file SHA       :", chosen["file_sha"])
    print(" scientific SHA :", chosen["scientific_core_sha"])
    print(" exact AFP provenance alignment: YES")
    print(" TEST QA performance used for selection: NO")
    print(" planner rerun: NO")
    print(" quarantined Transformers-5 artifact considered: NO")
    return chosen["rows"], chosen["path"], chosen["content_sha"], source

RQ3_FROZEN_PLAN_ROWS, RQ3_FROZEN_PLAN_PATHS = {}, {}
RQ3_FROZEN_PLAN_CONTENT_SHA, RQ3_FROZEN_PLAN_SOURCES = {}, {}
for _dataset_1 in ["webqsp", "cwq"]:
    rows, path, content_sha, source = discover_frozen_test_plans_rq3_1(_dataset_1)
    RQ3_FROZEN_PLAN_ROWS[_dataset_1] = rows
    RQ3_FROZEN_PLAN_PATHS[_dataset_1] = path
    RQ3_FROZEN_PLAN_CONTENT_SHA[_dataset_1] = content_sha
    RQ3_FROZEN_PLAN_SOURCES[_dataset_1] = source
    ids = [str(r["id"]) for r in rows]
    assert len(ids) == len(set(ids)) == EXPECTED_TEST_N_RQ3_1[_dataset_1]
    print(f"\n{_dataset_1.upper()} frozen TEST planning: PASSED")
    print(" source      :", source)
    print(" path        :", path)
    print(" questions   :", len(rows))
    print(" content SHA :", content_sha)
    print(" file SHA    :", sha256_file_rq3_1(path))

# ==================================================================================================
# 4. LOAD + NORMALIZE FINAL VERIFIED Full-AFP CAPTURES
# ==================================================================================================
EXPECTED_AFP_TOTAL_PATHS_RQ3_1 = {"webqsp": 31_487, "cwq": 194_115}
EXPECTED_AFP_FILE_SHA_RQ3_1 = {
    "webqsp": "3b6b61eec38ca0362ede19eba16a8b887ce1be9a29e768a233086dd99ddbde9a",
    "cwq": "627410074c3a469f8eaa4f23f8c1a9bbc6da97c7b801c71028969cbdee24a9a2",
}

def normalize_capture_record_rq3_1(row, expected_dataset):
    ds = first_present_rq3_1(row, ["dataset"], required=False)
    if ds is not None: assert str(ds).strip().lower() == expected_dataset
    variant = first_present_rq3_1(row, ["variant", "method"], required=False)
    if variant is not None: assert str(variant).strip().lower() in {"full afp", "afp"}
    qid = first_present_rq3_1(row, ["question_id", "qid", "id"])
    pi = first_present_rq3_1(row, ["plan_index", "plan_idx"])
    plan = first_present_rq3_1(row, ["plan", "predicted_path", "relation_plan"])
    paths = first_present_rq3_1(row, ["retrieved_paths", "final_prefixes", "paths", "final_paths"])
    assert isinstance(plan, list) and isinstance(paths, list)
    count = first_present_rq3_1(row, ["retrieved_path_count", "retrieved_paths_count", "path_count"], required=False)
    if count is not None: assert int(count) == len(paths)
    return {"question_id": str(qid), "plan_index": int(pi), "plan": list(plan), "retrieved_paths": paths}

RQ3_AFP_CAPTURE_ROWS, RQ3_AFP_CAPTURE_BY_QPLAN = {}, {}
for _dataset_1 in ["webqsp", "cwq"]:
    p = Path(RQ3_FULL_AFP_PATH_FILES[_dataset_1]); assert p.exists()
    actual_sha = sha256_file_rq3_1(p)
    assert actual_sha == EXPECTED_AFP_FILE_SHA_RQ3_1[_dataset_1]
    raw = read_gzip_jsonl_rq3_1(p); assert raw
    print(f"\n{_dataset_1.upper()} Full-AFP capture sample keys:", sorted(raw[0].keys()))
    norm = [normalize_capture_record_rq3_1(r, _dataset_1) for r in raw]
    mapping, total = {}, 0
    for rec in norm:
        key = (rec["question_id"], rec["plan_index"]); assert key not in mapping
        mapping[key] = rec; total += len(rec["retrieved_paths"])
    assert total == EXPECTED_AFP_TOTAL_PATHS_RQ3_1[_dataset_1]
    RQ3_AFP_CAPTURE_ROWS[_dataset_1] = norm
    RQ3_AFP_CAPTURE_BY_QPLAN[_dataset_1] = mapping
    print(f"{_dataset_1.upper()} Full-AFP capture schema/path total: PASSED")
    print(" records:", len(norm), " paths:", f"{total:,}", " SHA256:", actual_sha)

# ==================================================================================================
# 5. HARD PLAN <-> CAPTURE ALIGNMENT
# ==================================================================================================
RQ3_PLAN_ALIGNMENT_AUDIT = []
for _dataset_1 in ["webqsp", "cwq"]:
    cap = RQ3_AFP_CAPTURE_BY_QPLAN[_dataset_1]; expected_keys = set()
    for row in RQ3_FROZEN_PLAN_ROWS[_dataset_1]:
        qid = str(row["id"]); plans = row["predicted_paths"]
        assert isinstance(plans, list) and len(plans) <= 3
        for pi, plan in enumerate(plans):
            assert isinstance(plan, list)
            if not plan: continue
            key = (qid, pi); expected_keys.add(key)
            assert key in cap
            assert list(cap[key]["plan"]) == list(plan)
    unexpected = set(cap) - expected_keys
    assert not unexpected, f"Unexpected AFP capture keys: {list(sorted(unexpected))[:10]}"
    assert len(cap) == len(expected_keys)
    RQ3_PLAN_ALIGNMENT_AUDIT.append({"dataset": _dataset_1, "frozen_nonempty_plans": len(expected_keys),
                                     "capture_plan_records": len(cap), "exact": True})
    print(f"\n{_dataset_1.upper()} frozen-plan <-> AFP-capture alignment: PASSED")
    print(" non-empty frozen plans:", len(expected_keys), " capture records:", len(cap))

# ==================================================================================================
# 6. PATH HELPERS + EXACT EVIDENCE RECONSTRUCTION
# ==================================================================================================
def native_rog_paths_for_question_rq3_1(row):
    graph = RQ3_NATIVE_BUILD_GRAPH(row["graph"])
    assert isinstance(row["q_entity"], list) and isinstance(row["predicted_paths"], list)
    out = []
    # exact PromptBuilder.apply_rules order: topic entity outer, relation-plan inner
    for entity in row["q_entity"]:
        for plan in row["predicted_paths"]:
            out.extend(RQ3_NATIVE_BFS_WITH_RULE(graph, entity, plan))
    return out

def afp_captured_path_to_triples_rq3_1(captured_path, plan, qid, pi):
    assert isinstance(captured_path, (list, tuple)); p = list(captured_path)
    if p and all(isinstance(x, (list, tuple)) and len(x) == 3 for x in p):
        triples = [tuple(x) for x in p]
        assert [str(x[1]) for x in triples] == [str(r) for r in plan]
        return triples
    assert len(p) == len(plan) + 1, f"AFP prefix length mismatch: {qid} plan {pi}"
    return [(p[i], relation, p[i + 1]) for i, relation in enumerate(plan)]

def path_endpoint_rq3_1(path):
    return None if not path else str(path[-1][-1])

def has_newline_rq3_1(text):
    return "\n" in str(text) or "\r" in str(text)

RQ3_EVIDENCE_CACHE = {"webqsp": {}, "cwq": {}}
RQ3_SUBSET_AUDIT_ROWS = []
RQ3_EMPTY_PLAN_AUDIT_ROWS = []
EXPECTED_REACH_RQ3_1 = {"webqsp": {"RoG": 1369, "AFP": 1358}, "cwq": {"RoG": 2422, "AFP": 2391}}

# Native RoG legitimately emits blank serialization entries for empty relation plans:
# bfs_with_rule(graph, entity, []) -> [[]] and path_to_string([]) -> "".
# Frozen RQ2/AdaPruner, however, defines an empty plan as 0 retrieved paths.
# Preserve the native RoG blanks for byte fidelity, but never count them as retrieved paths
# and never invent synthetic AFP paths.
for _dataset_1 in ["webqsp", "cwq"]:
    cap_map = RQ3_AFP_CAPTURE_BY_QPLAN[_dataset_1]
    rog_reach = afp_reach = afp_total = 0
    total_empty_plans = questions_with_empty_plan = rog_empty_serialization_entries_total = 0

    for q_index, row in enumerate(RQ3_FROZEN_PLAN_ROWS[_dataset_1]):
        qid = str(row["id"]); plans = row["predicted_paths"]; topics = row["q_entity"]
        assert isinstance(plans, list) and isinstance(topics, list)
        empty_plan_count = sum(1 for p in plans if len(p) == 0)
        expected_empty_entries = len(topics) * empty_plan_count
        total_empty_plans += empty_plan_count
        questions_with_empty_plan += int(empty_plan_count > 0)

        rog_paths = native_rog_paths_for_question_rq3_1(row)
        rog_strings = [RQ3_NATIVE_PATH_TO_STRING(p) for p in rog_paths]
        rog_endpoints = [path_endpoint_rq3_1(p) for p in rog_paths]
        observed_empty_entries = sum(1 for s in rog_strings if s == "")
        assert observed_empty_entries == expected_empty_entries, (
            f"STOP: empty-plan serialization mismatch {_dataset_1} {qid}: "
            f"observed={observed_empty_entries}, expected={expected_empty_entries}"
        )
        assert not any(has_newline_rq3_1(s) for s in rog_strings)
        rog_empty_serialization_entries_total += observed_empty_entries
        rog_evidence_strings = [s for s in rog_strings if s != ""]

        afp_strings, afp_endpoints = [], []
        for pi, plan in enumerate(plans):
            if not plan: continue
            rec = cap_map[(qid, pi)]
            for captured in rec["retrieved_paths"]:
                triples = afp_captured_path_to_triples_rq3_1(captured, plan, qid, pi)
                s = RQ3_NATIVE_PATH_TO_STRING(triples)
                assert s != "" and not has_newline_rq3_1(s)
                afp_strings.append(s); afp_endpoints.append(path_endpoint_rq3_1(triples))
        afp_total += len(afp_strings)

        rog_counter, afp_counter = Counter(rog_evidence_strings), Counter(afp_strings)
        excess = afp_counter - rog_counter
        assert not excess, f"AFP invented path multiplicity for {_dataset_1} {qid}: {list(excess.items())[:5]}"
        need = afp_counter.copy(); native_filtered_nonempty = []
        for s in rog_strings:
            if s == "": continue
            if need[s] > 0:
                native_filtered_nonempty.append(s); need[s] -= 1
        assert sum(need.values()) == 0 and Counter(native_filtered_nonempty) == afp_counter
        raw_order_same = afp_strings == native_filtered_nonempty

        gold = {str(x) for x in row["a_entity"]}
        rr = any(e in gold for e in rog_endpoints if e is not None)
        ar = any(e in gold for e in afp_endpoints if e is not None)
        rog_reach += int(rr); afp_reach += int(ar)

        RQ3_EVIDENCE_CACHE[_dataset_1][qid] = {
            "question_index": q_index, "question": str(row["question"]),
            "rog_strings": rog_strings, "rog_evidence_strings": rog_evidence_strings,
            "rog_endpoints": rog_endpoints, "afp_strings": afp_strings, "afp_endpoints": afp_endpoints,
            "gold": sorted(gold), "empty_plan_count": int(empty_plan_count),
            "rog_empty_serialization_entries": int(observed_empty_entries),
            "rog_reachable_pre_budget": bool(rr), "afp_reachable_pre_budget": bool(ar),
        }
        RQ3_SUBSET_AUDIT_ROWS.append({
            "dataset": _dataset_1, "question_id": qid, "question_index": q_index,
            "rog_path_count": len(rog_evidence_strings), "afp_path_count": len(afp_strings),
            "rog_serialization_entry_count": len(rog_strings),
            "rog_empty_serialization_entries": int(observed_empty_entries),
            "empty_plan_count": int(empty_plan_count), "afp_multiset_subset_of_rog": True,
            "afp_raw_order_equals_native_filtered_order": bool(raw_order_same),
            "rog_reachable_pre_budget": int(rr), "afp_reachable_pre_budget": int(ar),
        })

    assert afp_total == EXPECTED_AFP_TOTAL_PATHS_RQ3_1[_dataset_1]
    assert rog_reach == EXPECTED_REACH_RQ3_1[_dataset_1]["RoG"]
    assert afp_reach == EXPECTED_REACH_RQ3_1[_dataset_1]["AFP"]
    RQ3_EMPTY_PLAN_AUDIT_ROWS.append({
        "dataset": _dataset_1, "questions": EXPECTED_TEST_N_RQ3_1[_dataset_1],
        "questions_with_empty_plan": int(questions_with_empty_plan), "empty_relation_plans": int(total_empty_plans),
        "native_rog_empty_serialization_entries": int(rog_empty_serialization_entries_total),
        "afp_synthetic_empty_paths_added": 0,
    })
    print(f"\n{_dataset_1.upper()} evidence reconstruction: PASSED")
    print(" RoG reachable:", rog_reach, " AFP reachable:", afp_reach, " AFP paths:", f"{afp_total:,}")
    print(" AFP multiset subset of RoG: EXACT (non-empty retrieved paths)")
    print(" AFP production order: FROZEN CAPTURE ORDER PRESERVED")
    print(" Empty relation plans:", total_empty_plans)
    print(" Questions with empty plan:", questions_with_empty_plan)
    print(" Native RoG blank serialization entries:", rog_empty_serialization_entries_total)
    print(" Synthetic AFP empty paths inserted: 0")

RQ3_SUBSET_AUDIT_DF = pd.DataFrame(RQ3_SUBSET_AUDIT_ROWS)
RQ3_EMPTY_PLAN_AUDIT_DF = pd.DataFrame(RQ3_EMPTY_PLAN_AUDIT_ROWS)
print("\nFrozen retrieval reachability reproduction: EXACT")

# ==================================================================================================
# 7. EXACT DIRECT-EVIDENCE PROMPT SERIALIZER
# ==================================================================================================
def native_check_prompt_length_with_trace_rq3_1(prompt, source_paths, maximum_token):
    """Exact PromptBuilder.check_prompt_length logic plus retained-list tracing."""
    paths = list(source_paths)
    all_paths = "\n".join(paths)
    if RQ3_PROMPT_BUILDER.tokenize(prompt + all_paths) < maximum_token:
        return all_paths, list(paths), False
    random.shuffle(paths)
    new_list_of_paths = []
    for p in paths:
        tmp_all_paths = "\n".join(new_list_of_paths + [p])
        if RQ3_PROMPT_BUILDER.tokenize(prompt + tmp_all_paths) > maximum_token:
            return "\n".join(new_list_of_paths), list(new_list_of_paths), True
        new_list_of_paths.append(p)
    return None, None, True


def direct_prompt_from_materialized_paths_rq3_1(dataset_name, question_id, question, path_strings, choices=None):
    choices = [] if choices is None else list(choices)
    question = str(question)
    if not question.endswith("?"): question += "?"
    input_text = RQ3_PROMPT_BUILDER.QUESTION.format(question=question)
    if choices:
        input_text += RQ3_PROMPT_BUILDER.CHOICES.format(choices="\n".join(choices))
        instruction = RQ3_PROMPT_BUILDER.MCQ_RULE_INSTRUCTION if RQ3_PROMPT_BUILDER.add_rule else RQ3_PROMPT_BUILDER.MCQ_INSTRUCTION
    else:
        instruction = RQ3_PROMPT_BUILDER.SAQ_RULE_INSTRUCTION if RQ3_PROMPT_BUILDER.add_rule else RQ3_PROMPT_BUILDER.SAQ_INSTRUCTION
    if RQ3_PROMPT_BUILDER.cot: instruction += RQ3_PROMPT_BUILDER.COT
    if RQ3_PROMPT_BUILDER.explain: instruction += RQ3_PROMPT_BUILDER.EXPLAIN
    if RQ3_PROMPT_BUILDER.each_line: instruction += RQ3_PROMPT_BUILDER.EACH_LINE

    other_prompt = RQ3_PROMPT_BUILDER.prompt_template.format(
        instruction=instruction,
        input=RQ3_PROMPT_BUILDER.GRAPH_CONTEXT.format(context="") + input_text,
    )
    source_paths = list(path_strings)
    precheck_tokens = int(RQ3_PROMPT_BUILDER.tokenize(other_prompt + "\n".join(source_paths)))
    seed = rq3_set_question_seed(dataset_name, question_id)
    context, retained, budget_triggered = native_check_prompt_length_with_trace_rq3_1(
        other_prompt, source_paths, int(RQ3_PROMPT_BUILDER.maximun_token)
    )
    assert context is not None and retained is not None, (
        f"STOP: native check_prompt_length would return None for {dataset_name} {question_id}; do not invent semantics."
    )
    assert "\n".join(retained) == context
    prompt = RQ3_PROMPT_BUILDER.prompt_template.format(
        instruction=instruction,
        input=RQ3_PROMPT_BUILDER.GRAPH_CONTEXT.format(context=context) + input_text,
    )
    slow_tokens = len(RQ3_TOKENIZER.tokenize(prompt))
    model_tokens = len(RQ3_TOKENIZER.encode(prompt, add_special_tokens=True))
    raw_evidence_paths = [s for s in source_paths if s != ""]
    retained_evidence_paths = [s for s in retained if s != ""]
    return {
        "seed": int(seed), "prompt": prompt,
        "raw_path_count": len(raw_evidence_paths), "retained_path_count": len(retained_evidence_paths),
        "dropped_path_count": len(raw_evidence_paths) - len(retained_evidence_paths),
        "raw_serialization_entry_count": len(source_paths),
        "retained_serialization_entry_count": len(retained),
        "raw_empty_serialization_entries": sum(1 for s in source_paths if s == ""),
        "retained_empty_serialization_entries": sum(1 for s in retained if s == ""),
        "retained_path_strings": retained, "budget_triggered": bool(budget_triggered),
        "precheck_slow_tokens": precheck_tokens, "slow_prompt_tokens": int(slow_tokens),
        "model_input_tokens": int(model_tokens), "prompt_chars": len(prompt),
        "prompt_bytes": len(prompt.encode("utf-8")),
        "input_plus_max_new_tokens": int(model_tokens + RQ3_MAX_NEW_TOKENS),
        "exceeds_nominal_4096_with_max_new_tokens": bool(model_tokens + RQ3_MAX_NEW_TOKENS > 4096),
    }

# ==================================================================================================
# 8. BYTE-FOR-BYTE DIRECT PROMPT FIDELITY VS NATIVE RoG
# ==================================================================================================
RQ3_PROMPT_FIDELITY_ROWS = []
for _dataset_1 in ["webqsp", "cwq"]:
    rows = RQ3_FROZEN_PLAN_ROWS[_dataset_1]
    counts, first_nonempty = [], []
    for i, row in enumerate(rows):
        qid = str(row["id"]); n = len(RQ3_EVIDENCE_CACHE[_dataset_1][qid]["rog_strings"])
        counts.append((i, n))
        if n > 0 and len(first_nonempty) < 5: first_nonempty.append(i)
    largest = [i for i, _ in sorted(counts, key=lambda x: (-x[1], x[0]))[:5]]
    audit_indices = sorted(set(first_nonempty + largest)); assert audit_indices
    for i in audit_indices:
        row = rows[i]; qid = str(row["id"]); ev = RQ3_EVIDENCE_CACHE[_dataset_1][qid]
        q_dict = {"question": row["question"], "graph": row["graph"], "q_entity": row["q_entity"],
                  "predicted_paths": row["predicted_paths"], "choices": []}
        rq3_set_question_seed(_dataset_1, qid)
        native_prompt = RQ3_PROMPT_BUILDER.process_input(q_dict)
        direct = direct_prompt_from_materialized_paths_rq3_1(_dataset_1, qid, row["question"], ev["rog_strings"], [])
        assert native_prompt == direct["prompt"], f"STOP: direct/native prompt mismatch: {_dataset_1} {qid}"
        RQ3_PROMPT_FIDELITY_ROWS.append({
            "dataset": _dataset_1, "question_index": i, "question_id": qid,
            "rog_raw_paths": len(ev["rog_strings"]), "budget_triggered": int(direct["budget_triggered"]),
            "native_prompt_sha256": sha256_text_rq3_1(native_prompt),
            "direct_prompt_sha256": sha256_text_rq3_1(direct["prompt"]), "byte_exact": True,
        })
    print(f"\n{_dataset_1.upper()} direct-prompt fidelity: PASSED")
    print(" audited questions:", len(audit_indices), " byte-for-byte exact:", len(audit_indices), "/", len(audit_indices))
RQ3_PROMPT_FIDELITY_DF = pd.DataFrame(RQ3_PROMPT_FIDELITY_ROWS)
assert bool(RQ3_PROMPT_FIDELITY_DF["byte_exact"].all())

# ==================================================================================================
# 9. BUILD/FREEZE ALL RoG + AFP REASONER INPUTS — STILL ZERO GENERATION
# ==================================================================================================
RQ3_1_ROOT = Path("/kaggle/working/step4_rq3_llm_reasoning_v1/01_evidence_prompt_audit")
RQ3_1_ROOT.mkdir(parents=True, exist_ok=True)
RQ3_REASONER_INPUT_ROWS = {"webqsp": {"RoG": [], "AFP": []}, "cwq": {"RoG": [], "AFP": []}}
RQ3_EVAL_REFERENCE_ROWS = {"webqsp": [], "cwq": []}
RQ3_PROMPT_AUDIT_ROWS = []

for _dataset_1 in ["webqsp", "cwq"]:
    for q_index, row in enumerate(RQ3_FROZEN_PLAN_ROWS[_dataset_1]):
        qid = str(row["id"]); ev = RQ3_EVIDENCE_CACHE[_dataset_1][qid]; gold = set(ev["gold"])
        method_inputs = {"RoG": (ev["rog_strings"], ev["rog_endpoints"]),
                         "AFP": (ev["afp_strings"], ev["afp_endpoints"])}
        eval_row = {
            "dataset": _dataset_1, "question_index": q_index, "question_id": qid,
            "gold_answers": list(ev["gold"]),
            "rog_reachable_pre_budget": bool(ev["rog_reachable_pre_budget"]),
            "afp_reachable_pre_budget": bool(ev["afp_reachable_pre_budget"]),
        }
        for method, (path_strings, endpoints) in method_inputs.items():
            built = direct_prompt_from_materialized_paths_rq3_1(_dataset_1, qid, row["question"], path_strings, [])
            endpoint_by_string = {}
            for s, endpoint in zip(path_strings, endpoints):
                if s in endpoint_by_string: assert endpoint_by_string[s] == endpoint
                else: endpoint_by_string[s] = endpoint
            retained_endpoints = [endpoint_by_string[s] for s in built["retained_path_strings"]]
            reachable_pre = any(e in gold for e in endpoints if e is not None)
            reachable_post = any(e in gold for e in retained_endpoints if e is not None)
            assert reachable_pre == (ev["rog_reachable_pre_budget"] if method == "RoG" else ev["afp_reachable_pre_budget"])
            input_row = {
                "dataset": _dataset_1, "method": method, "question_index": q_index, "question_id": qid,
                "reasoner_seed": int(built["seed"]), "question": str(row["question"]), "choices": [],
                "evidence_order_policy": ("native_promptbuilder_apply_rules_order" if method == "RoG"
                    else "frozen_rq2_capture_plan_order_then_in_record_final_prefix_order"),
                "raw_path_count": int(built["raw_path_count"]), "retained_path_count": int(built["retained_path_count"]),
                "dropped_path_count": int(built["dropped_path_count"]),
                "raw_serialization_entry_count": int(built["raw_serialization_entry_count"]),
                "retained_serialization_entry_count": int(built["retained_serialization_entry_count"]),
                "raw_empty_serialization_entries": int(built["raw_empty_serialization_entries"]),
                "retained_empty_serialization_entries": int(built["retained_empty_serialization_entries"]),
                "unique_raw_path_strings": len(set(s for s in path_strings if s != "")),
                "duplicate_raw_paths": len([s for s in path_strings if s != ""]) - len(set(s for s in path_strings if s != "")),
                "prompt_budget_triggered": bool(built["budget_triggered"]),
                "precheck_slow_tokens": int(built["precheck_slow_tokens"]),
                "slow_prompt_tokens": int(built["slow_prompt_tokens"]), "model_input_tokens": int(built["model_input_tokens"]),
                "prompt_chars": int(built["prompt_chars"]), "prompt_bytes": int(built["prompt_bytes"]),
                "input_plus_max_new_tokens": int(built["input_plus_max_new_tokens"]),
                "exceeds_nominal_4096_with_max_new_tokens": bool(built["exceeds_nominal_4096_with_max_new_tokens"]),
                "reachable_pre_prompt_budget": bool(reachable_pre), "reachable_post_prompt_budget": bool(reachable_post),
                "raw_evidence_sha256": sha256_text_rq3_1("\n".join(s for s in path_strings if s != "")),
                "retained_evidence_sha256": sha256_text_rq3_1("\n".join(s for s in built["retained_path_strings"] if s != "")),
                "raw_serialization_sha256": sha256_text_rq3_1("\n".join(path_strings)),
                "retained_serialization_sha256": sha256_text_rq3_1("\n".join(built["retained_path_strings"])),
                "prompt_sha256": sha256_text_rq3_1(built["prompt"]),
                "retained_path_strings": built["retained_path_strings"], "prompt": built["prompt"],
            }
            assert "gold_answers" not in input_row and "a_entity" not in input_row
            RQ3_REASONER_INPUT_ROWS[_dataset_1][method].append(input_row)
            RQ3_PROMPT_AUDIT_ROWS.append({k:v for k,v in input_row.items()
                                          if k not in {"prompt","retained_path_strings","question","choices"}})
            eval_row[f"{method.lower()}_reachable_post_budget"] = bool(reachable_post)
        RQ3_EVAL_REFERENCE_ROWS[_dataset_1].append(eval_row)

RQ3_PROMPT_AUDIT_DF = pd.DataFrame(RQ3_PROMPT_AUDIT_ROWS)
assert len(RQ3_PROMPT_AUDIT_DF) == 2 * (1628 + 3531)
for _dataset_1 in ["webqsp", "cwq"]:
    for method in ["RoG", "AFP"]:
        rows = RQ3_REASONER_INPUT_ROWS[_dataset_1][method]
        assert len(rows) == EXPECTED_TEST_N_RQ3_1[_dataset_1]
        ids = [r["question_id"] for r in rows]; assert len(ids) == len(set(ids))
    assert sum(int(r["rog_reachable_pre_budget"]) for r in RQ3_EVAL_REFERENCE_ROWS[_dataset_1]) == EXPECTED_REACH_RQ3_1[_dataset_1]["RoG"]
    assert sum(int(r["afp_reachable_pre_budget"]) for r in RQ3_EVAL_REFERENCE_ROWS[_dataset_1]) == EXPECTED_REACH_RQ3_1[_dataset_1]["AFP"]
print("\nAll-question frozen reasoner-input construction: PASSED")
print(" TEST answers generated: 0")

# ==================================================================================================
# 10. SAVE FROZEN INPUTS + GOLD-SEPARATE EVALUATION REFERENCES
# ==================================================================================================
RQ3_REASONER_INPUT_FILES, RQ3_EVAL_REFERENCE_FILES = {}, {}
for _dataset_1 in ["webqsp", "cwq"]:
    RQ3_REASONER_INPUT_FILES[_dataset_1] = {}
    for method, suffix in [("RoG", "rog"), ("AFP", "afp")]:
        p = RQ3_1_ROOT / f"{_dataset_1}_{suffix}_frozen_reasoner_inputs.jsonl.gz"
        write_deterministic_gzip_jsonl_rq3_1(RQ3_REASONER_INPUT_ROWS[_dataset_1][method], p)
        RQ3_REASONER_INPUT_FILES[_dataset_1][method] = p
    p = RQ3_1_ROOT / f"{_dataset_1}_evaluation_reference_GOLD_SEPARATE.jsonl.gz"
    write_deterministic_gzip_jsonl_rq3_1(RQ3_EVAL_REFERENCE_ROWS[_dataset_1], p)
    RQ3_EVAL_REFERENCE_FILES[_dataset_1] = p

# ==================================================================================================
# 11. SUMMARY / REDUCTION TABLES
# ==================================================================================================
summary_rows_1 = []
for (_dataset_1, method), g in RQ3_PROMPT_AUDIT_DF.groupby(["dataset", "method"], sort=False):
    summary_rows_1.append({
        "dataset": _dataset_1, "method": method, "questions": len(g),
        "raw_paths_total": int(g["raw_path_count"].sum()), "retained_paths_total": int(g["retained_path_count"].sum()),
        "dropped_paths_total": int(g["dropped_path_count"].sum()),
        "raw_serialization_entries_total": int(g["raw_serialization_entry_count"].sum()),
        "retained_serialization_entries_total": int(g["retained_serialization_entry_count"].sum()),
        "raw_empty_serialization_entries_total": int(g["raw_empty_serialization_entries"].sum()),
        "retained_empty_serialization_entries_total": int(g["retained_empty_serialization_entries"].sum()),
        "prompt_budget_triggered_questions": int(g["prompt_budget_triggered"].sum()),
        "context_risk_questions_input_plus_512_gt_4096": int(g["exceeds_nominal_4096_with_max_new_tokens"].sum()),
        "reachable_pre_budget": int(g["reachable_pre_prompt_budget"].sum()),
        "reachable_post_budget": int(g["reachable_post_prompt_budget"].sum()),
        "raw_paths_mean": float(g["raw_path_count"].mean()), "retained_paths_mean": float(g["retained_path_count"].mean()),
        "model_input_tokens_mean": float(g["model_input_tokens"].mean()),
        "model_input_tokens_median": float(g["model_input_tokens"].median()),
        "model_input_tokens_p95": float(g["model_input_tokens"].quantile(.95)),
        "model_input_tokens_max": int(g["model_input_tokens"].max()),
        "prompt_bytes_mean": float(g["prompt_bytes"].mean()), "prompt_bytes_total": int(g["prompt_bytes"].sum()),
    })
RQ3_PROMPT_INPUT_SUMMARY_DF = pd.DataFrame(summary_rows_1)

reduction_rows_1 = []
for _dataset_1 in ["webqsp", "cwq"]:
    g = RQ3_PROMPT_AUDIT_DF[RQ3_PROMPT_AUDIT_DF.dataset == _dataset_1]
    rog = g[g.method == "RoG"].sort_values("question_index").reset_index(drop=True)
    afp = g[g.method == "AFP"].sort_values("question_index").reset_index(drop=True)
    assert rog.question_id.tolist() == afp.question_id.tolist()
    row = {"dataset": _dataset_1, "questions": len(rog)}
    for metric in ["raw_path_count", "retained_path_count", "model_input_tokens", "prompt_bytes"]:
        rt, at = float(rog[metric].sum()), float(afp[metric].sum())
        row[f"rog_{metric}_total"] = rt; row[f"afp_{metric}_total"] = at
        row[f"delta_{metric}_afp_minus_rog"] = at - rt
        row[f"reduction_{metric}_fraction"] = 1.0 - at/rt if rt > 0 else 0.0
        row[f"paired_mean_delta_{metric}"] = float((afp[metric] - rog[metric]).mean())
    reduction_rows_1.append(row)
RQ3_AFP_VS_ROG_INPUT_REDUCTION_DF = pd.DataFrame(reduction_rows_1)

# ==================================================================================================
# 12. SAVE AUDITS + MANIFEST
# ==================================================================================================
RQ3_SUBSET_AUDIT_PATH = RQ3_1_ROOT / "rq3_1_afp_multiset_subset_audit.csv"
RQ3_PROMPT_FIDELITY_PATH = RQ3_1_ROOT / "rq3_1_direct_prompt_fidelity.csv"
RQ3_PROMPT_AUDIT_PATH = RQ3_1_ROOT / "rq3_1_per_question_prompt_input_audit.csv"
RQ3_PROMPT_SUMMARY_PATH = RQ3_1_ROOT / "rq3_1_prompt_input_summary.csv"
RQ3_REDUCTION_PATH = RQ3_1_ROOT / "rq3_1_afp_vs_rog_input_reduction.csv"
RQ3_PLAN_ALIGNMENT_PATH = RQ3_1_ROOT / "rq3_1_frozen_plan_capture_alignment.csv"
RQ3_PLAN_SOURCE_AUDIT_PATH = RQ3_1_ROOT / "rq3_1_planner_source_provenance_audit.csv"
RQ3_EMPTY_PLAN_AUDIT_PATH = RQ3_1_ROOT / "rq3_1_empty_plan_serialization_audit.csv"
RQ3_SUBSET_AUDIT_DF.to_csv(RQ3_SUBSET_AUDIT_PATH, index=False)
RQ3_PROMPT_FIDELITY_DF.to_csv(RQ3_PROMPT_FIDELITY_PATH, index=False)
RQ3_PROMPT_AUDIT_DF.to_csv(RQ3_PROMPT_AUDIT_PATH, index=False)
RQ3_PROMPT_INPUT_SUMMARY_DF.to_csv(RQ3_PROMPT_SUMMARY_PATH, index=False)
RQ3_AFP_VS_ROG_INPUT_REDUCTION_DF.to_csv(RQ3_REDUCTION_PATH, index=False)
pd.DataFrame(RQ3_PLAN_ALIGNMENT_AUDIT).to_csv(RQ3_PLAN_ALIGNMENT_PATH, index=False)
pd.DataFrame(RQ3_PLAN_SOURCE_AUDIT_ROWS).to_csv(RQ3_PLAN_SOURCE_AUDIT_PATH, index=False)
RQ3_EMPTY_PLAN_AUDIT_DF.to_csv(RQ3_EMPTY_PLAN_AUDIT_PATH, index=False)

RQ3_1_MANIFEST_PATH = RQ3_1_ROOT / "rq3_1_evidence_prompt_audit_manifest.json"
reasoner_input_manifest_1 = {
    d: {m: {"path": str(RQ3_REASONER_INPUT_FILES[d][m]),
             "sha256": sha256_file_rq3_1(RQ3_REASONER_INPUT_FILES[d][m]),
             "questions": EXPECTED_TEST_N_RQ3_1[d]}
        for m in ["RoG", "AFP"]}
    for d in ["webqsp", "cwq"]
}
manifest_1 = {
    "stage": "RQ3-1", "status": "frozen_evidence_and_prompt_audit_before_llm_generation",
    "source_rq3_0_manifest": str(RQ3_CONTRACT_MANIFEST_PATH),
    "source_rq3_0_manifest_sha256": EXPECTED_RQ3_0_MANIFEST_SHA256,
    "llm_generation_calls": 0, "test_tuning": False, "test_model_selection": False,
    "retrieval_rerun": False, "planner_rerun": False,
    "gold_used_to_construct_reasoner_input": False, "gold_used_posthoc_for_reachability_audit": True,
    "native_rog_sources": {
        "graph_utils_path": str(ROG_GRAPH_UTILS_PATH_1), "graph_utils_sha256": sha256_file_rq3_1(ROG_GRAPH_UTILS_PATH_1),
        "utils_path": str(ROG_UTILS_PATH_1), "utils_sha256": sha256_file_rq3_1(ROG_UTILS_PATH_1),
        "build_graph_function_sha256": sha256_text_rq3_1(_build_graph_source_1),
        "bfs_with_rule_function_sha256": sha256_text_rq3_1(_bfs_source_1),
        "path_to_string_function_sha256": sha256_text_rq3_1(_path_to_string_source_1),
    },
    "planner_source_resolution": {
        "policy": "exact qid+plan_index+relation_plan alignment with FINAL VERIFIED Full-AFP capture",
        "test_qa_performance_used": False, "gold_used_for_source_selection": False,
        "candidate_audit_csv": str(RQ3_PLAN_SOURCE_AUDIT_PATH),
    },
    "frozen_test_planning": {
        d: {"questions": EXPECTED_TEST_N_RQ3_1[d], "path": str(RQ3_FROZEN_PLAN_PATHS[d]),
            "file_sha256": sha256_file_rq3_1(RQ3_FROZEN_PLAN_PATHS[d]),
            "content_sha256": RQ3_FROZEN_PLAN_CONTENT_SHA[d], "source_resolution": RQ3_FROZEN_PLAN_SOURCES[d]}
        for d in ["webqsp", "cwq"]
    },
    "prompt_contract": {
        "native_prompt_builder_maximum_token": 3996, "slow_tokenizer": True,
        "direct_prompt_byte_exact_on_audit_sample": True, "audit_questions": len(RQ3_PROMPT_FIDELITY_DF),
        "max_new_tokens_frozen": 512, "do_sample_frozen": True,
    },
    "evidence_order_policy": {
        "rog": "exact native PromptBuilder.apply_rules order",
        "afp": "FINAL VERIFIED capture: relation-plan order then in-record retrieved_paths order",
        "sorting_or_deduplication": False, "afp_multiset_subset_gate": True,
    },
    "empty_plan_serialization_contract": {
        "native_rog_behavior": "bfs_with_rule(empty_rule) yields empty path; path_to_string yields empty string",
        "rog_blank_serialization_preserved_for_byte_fidelity": True,
        "blank_entries_counted_as_retrieved_paths": False,
        "afp_rq2_empty_plan_retrieved_paths": 0,
        "synthetic_afp_empty_paths_added": False,
        "audit_csv": str(RQ3_EMPTY_PLAN_AUDIT_PATH),
    },
    "full_afp_path_capture": {
        d: {"path": str(RQ3_FULL_AFP_PATH_FILES[d]), "sha256": sha256_file_rq3_1(RQ3_FULL_AFP_PATH_FILES[d]),
            "retrieved_paths": EXPECTED_AFP_TOTAL_PATHS_RQ3_1[d]}
        for d in ["webqsp", "cwq"]
    },
    "retrieval_reachability_exact": EXPECTED_REACH_RQ3_1,
    "reasoner_input_artifacts": reasoner_input_manifest_1,
    "evaluation_reference_artifacts": {
        d: {"path": str(RQ3_EVAL_REFERENCE_FILES[d]), "sha256": sha256_file_rq3_1(RQ3_EVAL_REFERENCE_FILES[d])}
        for d in ["webqsp", "cwq"]
    },
    "audit_tables": {
        "subset": str(RQ3_SUBSET_AUDIT_PATH), "prompt_fidelity": str(RQ3_PROMPT_FIDELITY_PATH),
        "per_question_prompt_input": str(RQ3_PROMPT_AUDIT_PATH), "summary": str(RQ3_PROMPT_SUMMARY_PATH),
        "input_reduction": str(RQ3_REDUCTION_PATH), "plan_capture_alignment": str(RQ3_PLAN_ALIGNMENT_PATH),
        "planner_source_provenance": str(RQ3_PLAN_SOURCE_AUDIT_PATH),
        "empty_plan_serialization": str(RQ3_EMPTY_PLAN_AUDIT_PATH),
    },
}
with RQ3_1_MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest_1, f, indent=2, ensure_ascii=False)
RQ3_1_MANIFEST_SHA256 = sha256_file_rq3_1(RQ3_1_MANIFEST_PATH)

# ==================================================================================================
# 13. FINAL FLAGS + REPORT
# ==================================================================================================
RQ3_1_CONTEXT_RISK_PRESENT = bool(RQ3_PROMPT_AUDIT_DF["exceeds_nominal_4096_with_max_new_tokens"].any())
RQ3_1_COMPLETE = True
print("\n" + "=" * 150)
print("RQ3-1 FINAL EVIDENCE / PROMPT AUDIT")
print("=" * 150)
print("\nPlanner source resolved by AFP provenance: PASSED")
print("Plan/capture alignment:             PASSED")
print("AFP evidence multiset subset:      PASSED")
print("AFP frozen capture order:          PRESERVED")
print("Frozen retrieval reachability:     EXACT")
print("Direct prompt vs native RoG:       BYTE-FOR-BYTE EXACT")
print("Slow tokenizer / 3996 budget:      PRESERVED")
print("Gold in reasoner-input artifacts:  NO")
print("LLM generation calls:              0")
print("TEST tuning/model selection:       NO")
print("Empty-plan native serialization:   AUDITED / PRESERVED FOR RoG")
print("Synthetic AFP empty paths:         NONE")

print("\nEMPTY-PLAN SERIALIZATION AUDIT")
print(RQ3_EMPTY_PLAN_AUDIT_DF.to_string(index=False))

print("\nPROMPT / INPUT SUMMARY")
print(RQ3_PROMPT_INPUT_SUMMARY_DF.to_string(index=False))
print("\nAFP vs RoG INPUT REDUCTION")
print(RQ3_AFP_VS_ROG_INPUT_REDUCTION_DF.to_string(index=False))
print("\nRaw AFP order already equals native-filtered order:")
for d in ["webqsp", "cwq"]:
    g = RQ3_SUBSET_AUDIT_DF[RQ3_SUBSET_AUDIT_DF.dataset == d]
    print(f" {d:7s}: {int(g.afp_raw_order_equals_native_filtered_order.sum())}/{len(g)} questions")
print("\nCONTEXT-WINDOW RISK AUDIT")
for d in ["webqsp", "cwq"]:
    for method in ["RoG", "AFP"]:
        g = RQ3_PROMPT_AUDIT_DF[(RQ3_PROMPT_AUDIT_DF.dataset == d) & (RQ3_PROMPT_AUDIT_DF.method == method)]
        print(f" {d:7s} {method:3s} | budget_triggered={int(g.prompt_budget_triggered.sum()):4d} | "
              f"input+512>4096={int(g.exceeds_nominal_4096_with_max_new_tokens.sum()):4d} | "
              f"post-budget reachable={int(g.reachable_post_prompt_budget.sum())}")
print("\nReasoner-input artifact hashes:")
for d in ["webqsp", "cwq"]:
    for method in ["RoG", "AFP"]:
        p = RQ3_REASONER_INPUT_FILES[d][method]
        print(f" {d:7s} {method:3s}: {sha256_file_rq3_1(p)}")
print("\nRQ3-1 manifest:")
print(RQ3_1_MANIFEST_PATH)
print("Manifest SHA256:", RQ3_1_MANIFEST_SHA256)
print("\nRQ3_1_CONTEXT_RISK_PRESENT:", RQ3_1_CONTEXT_RISK_PRESENT)
print("RQ3_1_COMPLETE:", RQ3_1_COMPLETE)
print("\nNEXT: RQ3-2 — small matched-seed LLM fidelity gate ONLY after reviewing the context-risk counts above.")
print("=" * 150)


## 5. RQ3-2 — Small Matched-Seed Generation Fidelity Gate

Runs the predeclared small generation audit using the frozen reasoner and
matched RoG/AdaPruner random seeds before the full TEST generation.


In [ ]:

from pathlib import Path
import gzip
import hashlib
import io
import json
import time
import warnings

import numpy as np
import pandas as pd
import torch


print("=" * 150)
print("RQ3-2 — SMALL MATCHED-SEED LLM GENERATION FIDELITY GATE")
print("=" * 150)


# ==================================================================================================
# 0. HARD RQ3-0 / RQ3-1 GATES
# ==================================================================================================

EXPECTED_RQ3_0_MANIFEST_SHA256_RQ3_2 = (
    "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8"
)


EXPECTED_RQ3_1_MANIFEST_SHA256_RQ3_2 = (
    "76d6a927654449fe84e12ef175abe1131da44780e887bdea0890ea5c8a4c3620"
)


assert globals().get(
    "RQ3_REASONER_CONTRACT_COMPLETE",
    False
) is True, (
    "STOP: RQ3-0 is not complete."
)


assert globals().get(
    "RQ3_1_COMPLETE",
    False
) is True, (
    "STOP: RQ3-1 is not complete."
)


assert globals().get(
    "RQ3_1_CONTEXT_RISK_PRESENT",
    False
) is True, (
    "Unexpected state: RQ3-1 reported no context risk."
)


assert (
    str(
        RQ3_CONTRACT_MANIFEST_SHA256
    )
    ==
    EXPECTED_RQ3_0_MANIFEST_SHA256_RQ3_2
), (
    "STOP: RQ3-0 manifest SHA changed."
)


assert (
    str(
        RQ3_1_MANIFEST_SHA256
    )
    ==
    EXPECTED_RQ3_1_MANIFEST_SHA256_RQ3_2
), (
    "STOP: RQ3-1 manifest SHA changed."
)


assert Path(
    RQ3_CONTRACT_MANIFEST_PATH
).exists()


assert Path(
    RQ3_1_MANIFEST_PATH
).exists()


for required_name in [

    "RQ3_MODEL",
    "RQ3_TOKENIZER",
    "RQ3_GENERATION_KWARGS",
    "RQ3_MAX_NEW_TOKENS",
    "RQ3_PROMPT_AUDIT_DF",
    "RQ3_REASONER_INPUT_ROWS",
    "rq3_question_seed",
    "rq3_set_question_seed",

]:

    assert required_name in globals(), (
        f"STOP: missing required live object: "
        f"{required_name}"
    )


assert (
    int(
        RQ3_MAX_NEW_TOKENS
    )
    ==
    512
)


assert (
    RQ3_GENERATION_KWARGS
    ==
    {
        "max_new_tokens":
            512,

        "do_sample":
            True,
    }
)


assert (
    type(
        RQ3_TOKENIZER
    ).__name__
    ==
    "LlamaTokenizer"
)


assert (
    RQ3_TOKENIZER.is_fast
    is False
)


assert (
    RQ3_MODEL.training
    is False
)


print(
    "\nUpstream contract gates: PASSED"
)

print(
    " RQ3-0 manifest:",
    EXPECTED_RQ3_0_MANIFEST_SHA256_RQ3_2
)

print(
    " RQ3-1 manifest:",
    EXPECTED_RQ3_1_MANIFEST_SHA256_RQ3_2
)

print(
    " model:",
    type(
        RQ3_MODEL
    ).__name__
)

print(
    " tokenizer:",
    type(
        RQ3_TOKENIZER
    ).__name__,
    "| slow =",
    not RQ3_TOKENIZER.is_fast
)

print(
    " generation:",
    RQ3_GENERATION_KWARGS
)


# ==================================================================================================
# 1. GENERAL HELPERS
# ==================================================================================================

def sha256_text_rq3_2(
    text
):

    return hashlib.sha256(
        str(
            text
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def sha256_int_list_rq3_2(
    values
):

    payload = json.dumps(
        [
            int(
                x
            )
            for x in values
        ],
        separators=(
            ",",
            ":"
        )
    )

    return sha256_text_rq3_2(
        payload
    )


def sha256_file_rq3_2(
    path,
    chunk_size=1024 * 1024
):

    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:

                break

            h.update(
                block
            )

    return h.hexdigest()


def write_deterministic_gzip_jsonl_rq3_2(
    rows,
    path
):

    path = Path(
        path
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )


    with path.open(
        "wb"
    ) as raw:

        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw,
            mtime=0
        ) as gz:

            with io.TextIOWrapper(
                gz,
                encoding="utf-8",
                newline="\n"
            ) as txt:

                for row in rows:

                    txt.write(
                        json.dumps(
                            row,
                            ensure_ascii=False,
                            separators=(
                                ",",
                                ":"
                            )
                        )
                        +
                        "\n"
                    )

    return path


def synchronize_all_cuda_rq3_2():

    if not torch.cuda.is_available():

        return


    for device_index in range(
        torch.cuda.device_count()
    ):

        torch.cuda.synchronize(
            device_index
        )


# ==================================================================================================
# 2. RE-VERIFY FROZEN MODEL GENERATION DEFAULTS
# ==================================================================================================

CURRENT_GENERATION_DEFAULTS_RQ3_2 = {}


for field in [

    "temperature",
    "top_p",
    "top_k",
    "num_beams",
    "repetition_penalty",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",

]:

    value = getattr(
        RQ3_MODEL.generation_config,
        field,
        None
    )


    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool
        )
    ) or value is None:

        CURRENT_GENERATION_DEFAULTS_RQ3_2[
            field
        ] = value


    else:

        try:

            CURRENT_GENERATION_DEFAULTS_RQ3_2[
                field
            ] = list(
                value
            )

        except Exception:

            CURRENT_GENERATION_DEFAULTS_RQ3_2[
                field
            ] = str(
                value
            )


assert (
    CURRENT_GENERATION_DEFAULTS_RQ3_2
    ==
    RQ3_MODEL_GENERATION_DEFAULTS
), (
    "STOP: model generation defaults changed "
    "after RQ3-0."
)


print(
    "\nGeneration defaults recheck: PASSED"
)


for key, value in (
    CURRENT_GENERATION_DEFAULTS_RQ3_2.items()
):

    print(
        f" {key:20s}: {value}"
    )


# ==================================================================================================
# 3. BUILD EXACT FROZEN INPUT INDEX
# ==================================================================================================

RQ3_2_INPUT_BY_QID = {

    "webqsp": {
        "RoG": {},
        "AFP": {},
    },

    "cwq": {
        "RoG": {},
        "AFP": {},
    },
}


for dataset_name in [

    "webqsp",
    "cwq",

]:

    for method in [

        "RoG",
        "AFP",

    ]:

        rows = (
            RQ3_REASONER_INPUT_ROWS[
                dataset_name
            ][
                method
            ]
        )


        for row in rows:

            qid = str(
                row[
                    "question_id"
                ]
            )


            assert (
                qid
                not in
                RQ3_2_INPUT_BY_QID[
                    dataset_name
                ][
                    method
                ]
            )


            RQ3_2_INPUT_BY_QID[
                dataset_name
            ][
                method
            ][
                qid
            ] = row


    assert (
        set(
            RQ3_2_INPUT_BY_QID[
                dataset_name
            ][
                "RoG"
            ]
        )
        ==
        set(
            RQ3_2_INPUT_BY_QID[
                dataset_name
            ][
                "AFP"
            ]
        )
    )


print(
    "\nFrozen reasoner-input index: PASSED"
)


# ==================================================================================================
# 4. PREDECLARE SMALL TEST SAMPLE USING ONLY INPUT STATISTICS
# ==================================================================================================
#
# No gold or QA result is used here.
#
# The selection policy itself is recorded in the RQ3-2 manifest.
# ==================================================================================================

RQ3_2_SELECTION_ROWS = []

RQ3_2_SELECTED_QIDS = {
    "webqsp": [],
    "cwq": [],
}

RQ3_2_SELECTION_REASONS = {
    "webqsp": {},
    "cwq": {},
}


def add_selected_qid_rq3_2(
    dataset_name,
    qid,
    reason
):

    qid = str(
        qid
    )


    if (
        qid
        not in
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    ):

        RQ3_2_SELECTED_QIDS[
            dataset_name
        ].append(
            qid
        )


    RQ3_2_SELECTION_REASONS[
        dataset_name
    ].setdefault(
        qid,
        []
    )


    if (
        reason
        not in
        RQ3_2_SELECTION_REASONS[
            dataset_name
        ][
            qid
        ]
    ):

        RQ3_2_SELECTION_REASONS[
            dataset_name
        ][
            qid
        ].append(
            reason
        )


for dataset_name in [

    "webqsp",
    "cwq",

]:

    rog = (
        RQ3_PROMPT_AUDIT_DF[
            (
                RQ3_PROMPT_AUDIT_DF[
                    "dataset"
                ]
                ==
                dataset_name
            )
            &
            (
                RQ3_PROMPT_AUDIT_DF[
                    "method"
                ]
                ==
                "RoG"
            )
        ]
        .sort_values(
            [
                "question_index",
                "question_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    afp = (
        RQ3_PROMPT_AUDIT_DF[
            (
                RQ3_PROMPT_AUDIT_DF[
                    "dataset"
                ]
                ==
                dataset_name
            )
            &
            (
                RQ3_PROMPT_AUDIT_DF[
                    "method"
                ]
                ==
                "AFP"
            )
        ]
        .sort_values(
            [
                "question_index",
                "question_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    assert (
        rog[
            "question_id"
        ].tolist()
        ==
        afp[
            "question_id"
        ].tolist()
    )


    pair = pd.DataFrame(
        {
            "question_id":
                rog[
                    "question_id"
                ].astype(
                    str
                ),

            "question_index":
                rog[
                    "question_index"
                ].astype(
                    int
                ),

            "rog_input_tokens":
                rog[
                    "model_input_tokens"
                ].astype(
                    int
                ),

            "afp_input_tokens":
                afp[
                    "model_input_tokens"
                ].astype(
                    int
                ),

            "rog_budget":
                rog[
                    "prompt_budget_triggered"
                ].astype(
                    bool
                ),

            "afp_budget":
                afp[
                    "prompt_budget_triggered"
                ].astype(
                    bool
                ),

            "rog_risk":
                rog[
                    "exceeds_nominal_4096_with_max_new_tokens"
                ].astype(
                    bool
                ),

            "afp_risk":
                afp[
                    "exceeds_nominal_4096_with_max_new_tokens"
                ].astype(
                    bool
                ),
        }
    )


    safe = pair[
        (
            ~pair[
                "rog_budget"
            ]
        )
        &
        (
            ~pair[
                "afp_budget"
            ]
        )
        &
        (
            ~pair[
                "rog_risk"
            ]
        )
        &
        (
            ~pair[
                "afp_risk"
            ]
        )
    ]


    assert (
        len(
            safe
        )
        >
        0
    )


    safe_row = (
        safe.iloc[
            0
        ]
    )


    add_selected_qid_rq3_2(
        dataset_name,
        safe_row[
            "question_id"
        ],
        "first_safe_no_budget_no_context_risk"
    )


    rog_max = (
        pair.sort_values(
            [
                "rog_input_tokens",
                "question_index"
            ],
            ascending=[
                False,
                True
            ]
        )
        .iloc[
            0
        ]
    )


    add_selected_qid_rq3_2(
        dataset_name,
        rog_max[
            "question_id"
        ],
        "maximum_rog_model_input_tokens"
    )


    afp_max = (
        pair.sort_values(
            [
                "afp_input_tokens",
                "question_index"
            ],
            ascending=[
                False,
                True
            ]
        )
        .iloc[
            0
        ]
    )


    add_selected_qid_rq3_2(
        dataset_name,
        afp_max[
            "question_id"
        ],
        "maximum_afp_model_input_tokens"
    )


    risk_relief = pair[
        (
            pair[
                "rog_risk"
            ]
        )
        &
        (
            ~pair[
                "afp_risk"
            ]
        )
    ]


    if (
        len(
            risk_relief
        )
        >
        0
    ):

        relief_row = (
            risk_relief.iloc[
                0
            ]
        )


        add_selected_qid_rq3_2(
            dataset_name,
            relief_row[
                "question_id"
            ],
            "first_rog_context_risk_removed_by_afp"
        )


    else:

        triggered = pair[
            (
                pair[
                    "rog_budget"
                ]
            )
            |
            (
                pair[
                    "afp_budget"
                ]
            )
        ]


        assert (
            len(
                triggered
            )
            >
            0
        )


        trigger_row = (
            triggered.iloc[
                0
            ]
        )


        add_selected_qid_rq3_2(
            dataset_name,
            trigger_row[
                "question_id"
            ],
            "first_prompt_budget_triggered_question"
        )


    if (
        len(
            RQ3_2_SELECTED_QIDS[
                dataset_name
            ]
        )
        <
        4
    ):

        for _, row in pair.iterrows():

            qid = str(
                row[
                    "question_id"
                ]
            )


            if (
                qid
                in
                RQ3_2_SELECTED_QIDS[
                    dataset_name
                ]
            ):

                continue


            add_selected_qid_rq3_2(
                dataset_name,
                qid,
                "deterministic_fill_by_question_index"
            )


            if (
                len(
                    RQ3_2_SELECTED_QIDS[
                        dataset_name
                    ]
                )
                ==
                4
            ):

                break


    assert (
        len(
            RQ3_2_SELECTED_QIDS[
                dataset_name
            ]
        )
        ==
        4
    )


    selected_pair = pair[
        pair[
            "question_id"
        ]
        .astype(
            str
        )
        .isin(
            RQ3_2_SELECTED_QIDS[
                dataset_name
            ]
        )
    ]


    assert bool(
        (
            selected_pair[
                "rog_risk"
            ]
            |
            selected_pair[
                "afp_risk"
            ]
        ).any()
    ), (
        f"STOP: {dataset_name} audit sample contains "
        "no nominal context-risk case."
    )


    for qid in (
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    ):

        row = pair[
            pair[
                "question_id"
            ]
            .astype(
                str
            )
            ==
            qid
        ].iloc[
            0
        ]


        RQ3_2_SELECTION_ROWS.append(
            {
                "dataset":
                    dataset_name,

                "question_id":
                    qid,

                "question_index":
                    int(
                        row[
                            "question_index"
                        ]
                    ),

                "selection_reasons":
                    "|".join(
                        RQ3_2_SELECTION_REASONS[
                            dataset_name
                        ][
                            qid
                        ]
                    ),

                "rog_input_tokens":
                    int(
                        row[
                            "rog_input_tokens"
                        ]
                    ),

                "afp_input_tokens":
                    int(
                        row[
                            "afp_input_tokens"
                        ]
                    ),

                "rog_budget_triggered":
                    bool(
                        row[
                            "rog_budget"
                        ]
                    ),

                "afp_budget_triggered":
                    bool(
                        row[
                            "afp_budget"
                        ]
                    ),

                "rog_nominal_context_risk":
                    bool(
                        row[
                            "rog_risk"
                        ]
                    ),

                "afp_nominal_context_risk":
                    bool(
                        row[
                            "afp_risk"
                        ]
                    ),
            }
        )


RQ3_2_SELECTION_DF = pd.DataFrame(
    RQ3_2_SELECTION_ROWS
)


print(
    "\nPredeclared small-generation sample:"
)

print(
    RQ3_2_SELECTION_DF.to_string(
        index=False
    )
)


print(
    "\nSample selection uses QA performance/gold: NO"
)

print(
    "Selected question IDs:"
)


for dataset_name in [

    "webqsp",
    "cwq",

]:

    print(
        f" {dataset_name:7s}:",
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    )


# ==================================================================================================
# 5. HARD CHECK THAT SEEDS ARE MATCHED RoG <-> AFP
# ==================================================================================================

for dataset_name in [

    "webqsp",
    "cwq",

]:

    for qid in (
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    ):

        rog_row = (
            RQ3_2_INPUT_BY_QID[
                dataset_name
            ][
                "RoG"
            ][
                qid
            ]
        )


        afp_row = (
            RQ3_2_INPUT_BY_QID[
                dataset_name
            ][
                "AFP"
            ][
                qid
            ]
        )


        expected_seed = (
            rq3_question_seed(
                dataset_name,
                qid
            )
        )


        assert (
            int(
                rog_row[
                    "reasoner_seed"
                ]
            )
            ==
            int(
                afp_row[
                    "reasoner_seed"
                ]
            )
            ==
            int(
                expected_seed
            )
        )


print(
    "\nMatched RoG/AFP seed gate: PASSED"
)


# ==================================================================================================
# 6. EXACT ONE-CASE GENERATION FUNCTION
# ==================================================================================================

def generate_one_frozen_case_rq3_2(
    dataset_name,
    method,
    qid,
    replay_tag="primary"
):

    row = (
        RQ3_2_INPUT_BY_QID[
            dataset_name
        ][
            method
        ][
            str(
                qid
            )
        ]
    )


    prompt = row[
        "prompt"
    ]


    assert isinstance(
        prompt,
        str
    )


    prompt_sha = (
        sha256_text_rq3_2(
            prompt
        )
    )


    assert (
        prompt_sha
        ==
        row[
            "prompt_sha256"
        ]
    ), (
        f"STOP: prompt SHA mismatch for "
        f"{dataset_name} {method} {qid}"
    )


    input_ids = (
        RQ3_TOKENIZER.encode(
            prompt,
            return_tensors="pt"
        )
    )


    observed_input_tokens = int(
        input_ids.shape[
            1
        ]
    )


    assert (
        observed_input_tokens
        ==
        int(
            row[
                "model_input_tokens"
            ]
        )
    ), (
        f"Input-token mismatch for "
        f"{dataset_name} {method} {qid}: "
        f"{observed_input_tokens} != "
        f"{row['model_input_tokens']}"
    )


    input_ids = input_ids.to(
        RQ3_MODEL.device
    )


    observed_seed = (
        rq3_set_question_seed(
            dataset_name,
            qid
        )
    )


    expected_seed = int(
        row[
            "reasoner_seed"
        ]
    )


    assert (
        int(
            observed_seed
        )
        ==
        expected_seed
    )


    synchronize_all_cuda_rq3_2()


    start = time.perf_counter()


    generated = None

    error_type = None

    error_message = None

    python_warning_messages = []


    try:

        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )


            with torch.inference_mode():

                generated = (
                    RQ3_MODEL.generate(

                        input_ids=
                            input_ids,

                        **RQ3_GENERATION_KWARGS,
                    )
                )


            python_warning_messages = [

                str(
                    w.message
                )

                for w in caught
            ]


    except Exception as exc:

        error_type = (
            type(
                exc
            ).__name__
        )

        error_message = str(
            exc
        )


    synchronize_all_cuda_rq3_2()


    latency_sec = (
        time.perf_counter()
        -
        start
    )


    if generated is None:

        return {

            "dataset":
                dataset_name,

            "method":
                method,

            "question_id":
                str(
                    qid
                ),

            "question_index":
                int(
                    row[
                        "question_index"
                    ]
                ),

            "replay_tag":
                replay_tag,

            "reasoner_seed":
                int(
                    observed_seed
                ),

            "rng_reset_immediately_before_generate":
                True,

            "prompt_sha256":
                prompt_sha,

            "input_tokens":
                observed_input_tokens,

            "nominal_input_plus_512":
                int(
                    observed_input_tokens
                    +
                    RQ3_MAX_NEW_TOKENS
                ),

            "nominal_context_risk":
                bool(
                    observed_input_tokens
                    +
                    RQ3_MAX_NEW_TOKENS
                    >
                    4096
                ),

            "generation_status":
                "ERROR",

            "error_type":
                error_type,

            "error_message":
                error_message,

            "latency_sec":
                float(
                    latency_sec
                ),

            "python_warnings":
                python_warning_messages,

            "generated_tokens":
                None,

            "actual_total_tokens":
                None,

            "actual_total_gt_4096":
                None,

            "reached_max_new_tokens":
                None,

            "eos_seen":
                None,

            "generated_token_ids_sha256":
                None,

            "output_text_sha256":
                None,

            "generated_token_ids":
                None,

            "output_text":
                None,
        }


    new_ids_tensor = (
        generated[
            0
        ][
            observed_input_tokens:
        ]
    )


    new_ids = [

        int(
            x
        )

        for x in (
            new_ids_tensor
            .detach()
            .cpu()
            .tolist()
        )
    ]


    generated_tokens = len(
        new_ids
    )


    assert (
        generated_tokens
        <=
        int(
            RQ3_MAX_NEW_TOKENS
        )
    )


    output_text = (
        RQ3_TOKENIZER.decode(

            new_ids_tensor,

            skip_special_tokens=True

        )
        .strip()
    )


    actual_total_tokens = (
        observed_input_tokens
        +
        generated_tokens
    )


    eos_token_id = (
        RQ3_MODEL.generation_config.eos_token_id
    )


    if isinstance(
        eos_token_id,
        (
            list,
            tuple
        )
    ):

        eos_set = {
            int(
                x
            )
            for x in eos_token_id
        }

    elif eos_token_id is None:

        eos_set = set()

    else:

        eos_set = {
            int(
                eos_token_id
            )
        }


    eos_seen = any(
        token_id in eos_set
        for token_id in new_ids
    )


    return {

        "dataset":
            dataset_name,

        "method":
            method,

        "question_id":
            str(
                qid
            ),

        "question_index":
            int(
                row[
                    "question_index"
                ]
            ),

        "replay_tag":
            replay_tag,

        "reasoner_seed":
            int(
                observed_seed
            ),

        "rng_reset_immediately_before_generate":
            True,

        "prompt_sha256":
            prompt_sha,

        "input_tokens":
            observed_input_tokens,

        "nominal_input_plus_512":
            int(
                observed_input_tokens
                +
                RQ3_MAX_NEW_TOKENS
            ),

        "nominal_context_risk":
            bool(
                observed_input_tokens
                +
                RQ3_MAX_NEW_TOKENS
                >
                4096
            ),

        "generation_status":
            "OK",

        "error_type":
            None,

        "error_message":
            None,

        "latency_sec":
            float(
                latency_sec
            ),

        "python_warnings":
            python_warning_messages,

        "generated_tokens":
            int(
                generated_tokens
            ),

        "actual_total_tokens":
            int(
                actual_total_tokens
            ),

        "actual_total_gt_4096":
            bool(
                actual_total_tokens
                >
                4096
            ),

        "reached_max_new_tokens":
            bool(
                generated_tokens
                ==
                int(
                    RQ3_MAX_NEW_TOKENS
                )
            ),

        "eos_seen":
            bool(
                eos_seen
            ),

        "generated_token_ids_sha256":
            sha256_int_list_rq3_2(
                new_ids
            ),

        "output_text_sha256":
            sha256_text_rq3_2(
                output_text
            ),

        "generated_token_ids":
            new_ids,

        "output_text":
            output_text,
    }


# ==================================================================================================
# 7. RUN THE PREDECLARED PAIRED GENERATION SAMPLE
# ==================================================================================================

RQ3_2_GENERATION_ROWS = []


print(
    "\n"
    +
    "=" * 150
)

print(
    "RUNNING PREDECLARED SMALL GENERATION GATE"
)

print(
    "=" * 150
)


for dataset_name in [

    "webqsp",
    "cwq",

]:

    for qid in (
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    ):

        for method in [

            "RoG",
            "AFP",

        ]:

            frozen_row = (
                RQ3_2_INPUT_BY_QID[
                    dataset_name
                ][
                    method
                ][
                    qid
                ]
            )


            print(
                f"\n{dataset_name.upper()} | "
                f"qid={qid} | "
                f"{method} | "
                f"input={int(frozen_row['model_input_tokens'])} | "
                f"input+512="
                f"{int(frozen_row['model_input_tokens']) + 512}"
            )


            result = (
                generate_one_frozen_case_rq3_2(

                    dataset_name,
                    method,
                    qid,
                    replay_tag=
                        "primary",
                )
            )


            RQ3_2_GENERATION_ROWS.append(
                result
            )


            if (
                result[
                    "generation_status"
                ]
                ==
                "OK"
            ):

                print(
                    " status=OK"
                    f" | generated={result['generated_tokens']}"
                    f" | actual_total={result['actual_total_tokens']}"
                    f" | actual>4096={result['actual_total_gt_4096']}"
                    f" | cap512={result['reached_max_new_tokens']}"
                    f" | eos={result['eos_seen']}"
                    f" | latency={result['latency_sec']:.3f}s"
                )


            else:

                print(
                    " status=ERROR"
                    f" | {result['error_type']}: "
                    f"{result['error_message']}"
                )


# ==================================================================================================
# 8. SAVE PRIMARY DIAGNOSTICS BEFORE HARD ASSERTIONS
# ==================================================================================================

RQ3_2_ROOT = Path(
    "/kaggle/working/"
    "step4_rq3_llm_reasoning_v1/"
    "02_small_generation_fidelity"
)


RQ3_2_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


RQ3_2_SELECTION_PATH = (
    RQ3_2_ROOT
    /
    "rq3_2_predeclared_sample.csv"
)


RQ3_2_PRIMARY_OUTPUT_PATH = (
    RQ3_2_ROOT
    /
    "rq3_2_small_generation_primary.jsonl.gz"
)


RQ3_2_DIAGNOSTIC_CSV_PATH = (
    RQ3_2_ROOT
    /
    "rq3_2_small_generation_diagnostics.csv"
)


RQ3_2_SELECTION_DF.to_csv(
    RQ3_2_SELECTION_PATH,
    index=False
)


write_deterministic_gzip_jsonl_rq3_2(
    RQ3_2_GENERATION_ROWS,
    RQ3_2_PRIMARY_OUTPUT_PATH
)


RQ3_2_GENERATION_DF = pd.DataFrame(
    [
        {
            k:
                v

            for k, v in row.items()

            if k
            not in
            {
                "generated_token_ids",
                "output_text",
                "python_warnings",
            }
        }

        for row in RQ3_2_GENERATION_ROWS
    ]
)


RQ3_2_GENERATION_DF.to_csv(
    RQ3_2_DIAGNOSTIC_CSV_PATH,
    index=False
)


# ==================================================================================================
# 9. HARD MECHANICAL GENERATION GATE
# ==================================================================================================

generation_errors = [

    row

    for row in RQ3_2_GENERATION_ROWS

    if row[
        "generation_status"
    ]
    !=
    "OK"
]


if generation_errors:

    print(
        "\n"
        +
        "=" * 150
    )

    print(
        "RQ3-2 GENERATION ERRORS DETECTED"
    )

    print(
        "=" * 150
    )


    for row in generation_errors:

        print(
            row[
                "dataset"
            ],
            row[
                "method"
            ],
            row[
                "question_id"
            ],
            row[
                "error_type"
            ],
            row[
                "error_message"
            ]
        )


    print(
        "\nDiagnostics were saved before stopping:"
    )

    print(
        RQ3_2_PRIMARY_OUTPUT_PATH
    )


    raise AssertionError(
        "STOP: at least one predeclared fidelity generation failed. "
        "Do NOT modify the generation contract or continue to full TEST reasoning."
    )


assert (
    len(
        RQ3_2_GENERATION_ROWS
    )
    ==
    16
), (
    "Expected 8 matched questions x 2 methods = "
    "16 primary generations."
)


assert all(
    row[
        "generated_tokens"
    ]
    <=
    512

    for row in (
        RQ3_2_GENERATION_ROWS
    )
)


print(
    "\nMechanical generation gate: PASSED"
)

print(
    " Primary generation calls:",
    len(
        RQ3_2_GENERATION_ROWS
    )
)

print(
    " Generation exceptions:   0"
)


# ==================================================================================================
# 10. MATCHED-SEED / IDENTICAL-PROMPT CONSISTENCY GATE
# ==================================================================================================

identical_prompt_pair_checks = 0


for dataset_name in [

    "webqsp",
    "cwq",

]:

    for qid in (
        RQ3_2_SELECTED_QIDS[
            dataset_name
        ]
    ):

        rog = next(
            r

            for r in RQ3_2_GENERATION_ROWS

            if (
                r[
                    "dataset"
                ]
                ==
                dataset_name

                and

                r[
                    "method"
                ]
                ==
                "RoG"

                and

                r[
                    "question_id"
                ]
                ==
                qid
            )
        )


        afp = next(
            r

            for r in RQ3_2_GENERATION_ROWS

            if (
                r[
                    "dataset"
                ]
                ==
                dataset_name

                and

                r[
                    "method"
                ]
                ==
                "AFP"

                and

                r[
                    "question_id"
                ]
                ==
                qid
            )
        )


        assert (
            rog[
                "reasoner_seed"
            ]
            ==
            afp[
                "reasoner_seed"
            ]
        )


        if (
            rog[
                "prompt_sha256"
            ]
            ==
            afp[
                "prompt_sha256"
            ]
        ):

            identical_prompt_pair_checks += (
                1
            )


            assert (
                rog[
                    "generated_token_ids_sha256"
                ]
                ==
                afp[
                    "generated_token_ids_sha256"
                ]
            ), (
                f"STOP: identical matched prompts with the same "
                f"seed produced different sampled token sequences: "
                f"{dataset_name} {qid}"
            )


print(
    "\nMatched-seed consistency: PASSED"
)

print(
    " Identical RoG/AFP prompt pairs checked:",
    identical_prompt_pair_checks
)


# ==================================================================================================
# 11. EXPLICIT DETERMINISTIC REPLAY PROBE
# ==================================================================================================

safe_webqsp_candidates = (

    RQ3_2_SELECTION_DF[
        (
            RQ3_2_SELECTION_DF[
                "dataset"
            ]
            ==
            "webqsp"
        )
        &
        (
            RQ3_2_SELECTION_DF[
                "selection_reasons"
            ]
            .str
            .contains(
                "first_safe_no_budget_no_context_risk",
                regex=False
            )
        )
    ]
)


assert (
    len(
        safe_webqsp_candidates
    )
    >=
    1
)


REPLAY_QID_RQ3_2 = str(
    safe_webqsp_candidates.iloc[
        0
    ][
        "question_id"
    ]
)


primary_replay_reference = next(

    row

    for row in (
        RQ3_2_GENERATION_ROWS
    )

    if (
        row[
            "dataset"
        ]
        ==
        "webqsp"

        and

        row[
            "method"
        ]
        ==
        "RoG"

        and

        row[
            "question_id"
        ]
        ==
        REPLAY_QID_RQ3_2
    )
)


print(
    "\nDeterministic replay probe:"
)

print(
    " dataset : webqsp"
)

print(
    " method  : RoG"
)

print(
    " qid     :",
    REPLAY_QID_RQ3_2
)

print(
    " seed    :",
    primary_replay_reference[
        "reasoner_seed"
    ]
)


RQ3_2_REPLAY_RESULT = (
    generate_one_frozen_case_rq3_2(

        "webqsp",

        "RoG",

        REPLAY_QID_RQ3_2,

        replay_tag=
            "deterministic_replay",
    )
)


assert (
    RQ3_2_REPLAY_RESULT[
        "generation_status"
    ]
    ==
    "OK"
)


assert (
    RQ3_2_REPLAY_RESULT[
        "reasoner_seed"
    ]
    ==
    primary_replay_reference[
        "reasoner_seed"
    ]
)


assert (
    RQ3_2_REPLAY_RESULT[
        "prompt_sha256"
    ]
    ==
    primary_replay_reference[
        "prompt_sha256"
    ]
)


assert (
    RQ3_2_REPLAY_RESULT[
        "generated_token_ids_sha256"
    ]
    ==
    primary_replay_reference[
        "generated_token_ids_sha256"
    ]
), (
    "STOP: same frozen prompt + same frozen seed did not "
    "reproduce the same sampled token sequence."
)


assert (
    RQ3_2_REPLAY_RESULT[
        "output_text_sha256"
    ]
    ==
    primary_replay_reference[
        "output_text_sha256"
    ]
)


RQ3_2_REPLAY_PATH = (
    RQ3_2_ROOT
    /
    "rq3_2_deterministic_replay.jsonl.gz"
)


write_deterministic_gzip_jsonl_rq3_2(
    [
        primary_replay_reference,
        RQ3_2_REPLAY_RESULT,
    ],
    RQ3_2_REPLAY_PATH
)


print(
    " deterministic token replay: BYTE-EXACT PASSED"
)


# ==================================================================================================
# 12. CONTEXT-WINDOW OUTCOME AUDIT
# ==================================================================================================

primary_df = pd.DataFrame(
    [
        {
            "dataset":
                r[
                    "dataset"
                ],

            "method":
                r[
                    "method"
                ],

            "question_id":
                r[
                    "question_id"
                ],

            "input_tokens":
                int(
                    r[
                        "input_tokens"
                    ]
                ),

            "nominal_input_plus_512":
                int(
                    r[
                        "nominal_input_plus_512"
                    ]
                ),

            "nominal_context_risk":
                bool(
                    r[
                        "nominal_context_risk"
                    ]
                ),

            "generated_tokens":
                int(
                    r[
                        "generated_tokens"
                    ]
                ),

            "actual_total_tokens":
                int(
                    r[
                        "actual_total_tokens"
                    ]
                ),

            "actual_total_gt_4096":
                bool(
                    r[
                        "actual_total_gt_4096"
                    ]
                ),

            "reached_max_new_tokens":
                bool(
                    r[
                        "reached_max_new_tokens"
                    ]
                ),

            "eos_seen":
                bool(
                    r[
                        "eos_seen"
                    ]
                ),

            "latency_sec":
                float(
                    r[
                        "latency_sec"
                    ]
                ),
        }

        for r in (
            RQ3_2_GENERATION_ROWS
        )
    ]
)


RQ3_2_NOMINAL_CONTEXT_RISK_CASES = int(
    primary_df[
        "nominal_context_risk"
    ].sum()
)


RQ3_2_ACTUAL_CONTEXT_EXTENSION_CASES = int(
    primary_df[
        "actual_total_gt_4096"
    ].sum()
)


RQ3_2_MAX_ACTUAL_TOTAL_TOKENS = int(
    primary_df[
        "actual_total_tokens"
    ].max()
)


RQ3_2_MAX_NEW_TOKEN_CAP_HITS = int(
    primary_df[
        "reached_max_new_tokens"
    ].sum()
)


RQ3_2_ACTUAL_CONTEXT_EXTENSION_SEEN = bool(
    RQ3_2_ACTUAL_CONTEXT_EXTENSION_CASES
    >
    0
)


print(
    "\n"
    +
    "=" * 150
)

print(
    "CONTEXT-WINDOW GENERATION OUTCOME"
)

print(
    "=" * 150
)


print(
    primary_df.to_string(
        index=False
    )
)


print(
    "\nNominal input+512 > 4096 cases:",
    RQ3_2_NOMINAL_CONTEXT_RISK_CASES
)

print(
    "Actual input+generated > 4096 cases:",
    RQ3_2_ACTUAL_CONTEXT_EXTENSION_CASES
)

print(
    "Maximum actual total sequence length:",
    RQ3_2_MAX_ACTUAL_TOTAL_TOKENS
)

print(
    "Cases reaching full 512-new-token cap:",
    RQ3_2_MAX_NEW_TOKEN_CAP_HITS
)


# ==================================================================================================
# 13. IMPORTANT METHODOLOGICAL GATES
# ==================================================================================================

assert (
    len(
        generation_errors
    )
    ==
    0
)


assert (
    RQ3_2_REPLAY_RESULT[
        "generated_token_ids_sha256"
    ]
    ==
    primary_replay_reference[
        "generated_token_ids_sha256"
    ]
)


print(
    "\nSoftware fidelity gates: PASSED"
)

print(
    " Generation contract modified: NO"
)

print(
    " Prompt budget modified:       NO"
)

print(
    " Seed modified/selected:        NO"
)

print(
    " Gold consulted:                NO"
)

print(
    " QA metrics calculated:         NO"
)

print(
    " QA semantic outputs reviewed:  NO"
)


# ==================================================================================================
# 14. SAVE MANIFEST
# ==================================================================================================

RQ3_2_MANIFEST_PATH = (
    RQ3_2_ROOT
    /
    "rq3_2_small_generation_fidelity_manifest.json"
)


RQ3_2_MANIFEST = {

    "stage":
        "RQ3-2",

    "status":
        "small_matched_seed_generation_fidelity_gate",

    "source_rq3_0_manifest_sha256":
        EXPECTED_RQ3_0_MANIFEST_SHA256_RQ3_2,

    "source_rq3_1_manifest_sha256":
        EXPECTED_RQ3_1_MANIFEST_SHA256_RQ3_2,


    "scientific_controls": {

        "test_qa_metrics_calculated":
            False,

        "gold_used":
            False,

        "test_performance_used_for_selection":
            False,

        "seed_tuning":
            False,

        "reasoner_tuning":
            False,

        "model_selection":
            False,

        "prompt_budget_modified":
            False,

        "generation_contract_modified":
            False,

        "retrieval_rerun":
            False,

        "planner_rerun":
            False,
    },


    "sample_selection": {

        "datasets":
            [
                "webqsp",
                "cwq",
            ],

        "question_ids_per_dataset":
            4,

        "methods_per_question":
            [
                "RoG",
                "AFP",
            ],

        "primary_generation_calls":
            int(
                len(
                    RQ3_2_GENERATION_ROWS
                )
            ),

        "deterministic_replay_calls":
            1,

        "policy":
            [
                "first safe no-budget/no-context-risk paired question",
                "maximum RoG model-input-token question",
                "maximum AFP model-input-token question",
                (
                    "first question where AFP removes RoG nominal context risk; "
                    "otherwise first prompt-budget-triggered question"
                ),
                (
                    "deterministic fill by question index if categories overlap"
                ),
            ],

        "selection_csv":
            str(
                RQ3_2_SELECTION_PATH
            ),

        "selection_csv_sha256":
            sha256_file_rq3_2(
                RQ3_2_SELECTION_PATH
            ),
    },


    "generation_contract": {

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "explicit_generation_kwargs":
            sorted(
                RQ3_GENERATION_KWARGS.keys()
            ),

        "inherited_generation_defaults":
            CURRENT_GENERATION_DEFAULTS_RQ3_2,

        "rng_reset_immediately_before_generate":
            True,

        "seed_mapping":
            (
                "SHA256(base_seed|dataset|question_id) "
                "mod (2^31-1)"
            ),

        "same_seed_for_matched_rog_afp":
            True,
    },


    "context_window_audit": {

        "nominal_model_max_position_embeddings":
            int(
                getattr(
                    RQ3_MODEL.config,
                    "max_position_embeddings",
                    4096
                )
            ),

        "nominal_input_plus_512_risk_cases_in_sample":
            int(
                RQ3_2_NOMINAL_CONTEXT_RISK_CASES
            ),

        "actual_input_plus_generated_gt_4096_cases":
            int(
                RQ3_2_ACTUAL_CONTEXT_EXTENSION_CASES
            ),

        "actual_context_extension_seen":
            bool(
                RQ3_2_ACTUAL_CONTEXT_EXTENSION_SEEN
            ),

        "maximum_actual_total_tokens":
            int(
                RQ3_2_MAX_ACTUAL_TOTAL_TOKENS
            ),

        "cases_reaching_512_new_token_cap":
            int(
                RQ3_2_MAX_NEW_TOKEN_CAP_HITS
            ),
    },


    "reproducibility": {

        "explicit_replay_dataset":
            "webqsp",

        "explicit_replay_method":
            "RoG",

        "explicit_replay_question_id":
            REPLAY_QID_RQ3_2,

        "same_prompt":
            True,

        "same_seed":
            True,

        "generated_token_sequence_byte_exact":
            True,

        "primary_generated_token_sha256":
            primary_replay_reference[
                "generated_token_ids_sha256"
            ],

        "replay_generated_token_sha256":
            RQ3_2_REPLAY_RESULT[
                "generated_token_ids_sha256"
            ],
    },


    "outputs": {

        "primary_generation_jsonl_gz":
            str(
                RQ3_2_PRIMARY_OUTPUT_PATH
            ),

        "primary_generation_sha256":
            sha256_file_rq3_2(
                RQ3_2_PRIMARY_OUTPUT_PATH
            ),

        "diagnostic_csv":
            str(
                RQ3_2_DIAGNOSTIC_CSV_PATH
            ),

        "diagnostic_csv_sha256":
            sha256_file_rq3_2(
                RQ3_2_DIAGNOSTIC_CSV_PATH
            ),

        "deterministic_replay_jsonl_gz":
            str(
                RQ3_2_REPLAY_PATH
            ),

        "deterministic_replay_sha256":
            sha256_file_rq3_2(
                RQ3_2_REPLAY_PATH
            ),
    },
}


with RQ3_2_MANIFEST_PATH.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RQ3_2_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


RQ3_2_MANIFEST_SHA256 = (
    sha256_file_rq3_2(
        RQ3_2_MANIFEST_PATH
    )
)


# ==================================================================================================
# 15. FINAL FLAGS
# ==================================================================================================

RQ3_2_GENERATION_ERRORS = (
    len(
        generation_errors
    )
)


RQ3_2_DETERMINISTIC_REPLAY_PASSED = (
    True
)


RQ3_2_COMPLETE = (
    True
)


print(
    "\n"
    +
    "=" * 150
)

print(
    "RQ3-2 FINAL FIDELITY REPORT"
)

print(
    "=" * 150
)


print(
    "Frozen RoG 7B reasoner:              PRESERVED"
)

print(
    "Frozen slow tokenizer:               PRESERVED"
)

print(
    "Frozen prompts from RQ3-1:           PRESERVED"
)

print(
    "max_new_tokens=512:                  PRESERVED"
)

print(
    "do_sample=True:                      PRESERVED"
)

print(
    "Matched per-question RNG:            PASSED"
)

print(
    "RNG reset immediately before generate: YES"
)

print(
    "Primary generation calls:            ",
    len(
        RQ3_2_GENERATION_ROWS
    )
)

print(
    "Generation exceptions:               ",
    RQ3_2_GENERATION_ERRORS
)

print(
    "Deterministic replay:                BYTE-EXACT PASSED"
)

print(
    "Nominal context-risk cases sampled:   ",
    RQ3_2_NOMINAL_CONTEXT_RISK_CASES
)

print(
    "Actual >4096 sequence cases:          ",
    RQ3_2_ACTUAL_CONTEXT_EXTENSION_CASES
)

print(
    "Maximum actual sequence length:       ",
    RQ3_2_MAX_ACTUAL_TOTAL_TOKENS
)

print(
    "512-token cap reached:                ",
    RQ3_2_MAX_NEW_TOKEN_CAP_HITS
)

print(
    "QA metrics calculated:                NO"
)

print(
    "Gold consulted:                       NO"
)

print(
    "TEST tuning/model selection:          NO"
)


print(
    "\nRQ3-2 manifest:"
)

print(
    RQ3_2_MANIFEST_PATH
)

print(
    "Manifest SHA256:",
    RQ3_2_MANIFEST_SHA256
)


print(
    "\nRQ3_2_ACTUAL_CONTEXT_EXTENSION_SEEN:",
    RQ3_2_ACTUAL_CONTEXT_EXTENSION_SEEN
)

print(
    "RQ3_2_DETERMINISTIC_REPLAY_PASSED:",
    RQ3_2_DETERMINISTIC_REPLAY_PASSED
)

print(
    "RQ3_2_COMPLETE:",
    RQ3_2_COMPLETE
)


print(
    "\nIMPORTANT:"
)

print(
    "Do NOT begin full TEST generation until the "
    "actual context-extension result above is reviewed."
)

print(
    "=" * 150
)


## 6. RQ3-2 Rerun Equivalence Audit

Verifies that the recovered/rerun RQ3-2 artifacts remain scientifically
equivalent to the frozen generation protocol.


In [ ]:

from pathlib import Path
import hashlib
import json
import gzip


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for b in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b""
        ):
            h.update(b)

    return h.hexdigest()


EXPECTED_RQ3_0 = (
    "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8"
)

EXPECTED_RQ3_1 = (
    "76d6a927654449fe84e12ef175abe1131da44780e887bdea0890ea5c8a4c3620"
)


print("=" * 120)
print("RQ3-2 RERUN EQUIVALENCE AUDIT")
print("=" * 120)


# ------------------------------------------------------------------------------------------
# 1. Current manifest
# ------------------------------------------------------------------------------------------

p = Path(
    "/kaggle/working/"
    "step4_rq3_llm_reasoning_v1/"
    "02_small_generation_fidelity/"
    "rq3_2_small_generation_fidelity_manifest.json"
)

assert p.exists()

actual_sha = sha256_file(
    p
)


with p.open(
    "r",
    encoding="utf-8"
) as f:

    m = json.load(
        f
    )


print(
    "\nCurrent RQ3-2 manifest SHA:"
)

print(
    actual_sha
)


assert (
    str(
        RQ3_2_MANIFEST_SHA256
    )
    ==
    actual_sha
)


# ------------------------------------------------------------------------------------------
# 2. Upstream scientific provenance must still be identical
# ------------------------------------------------------------------------------------------

assert (
    m[
        "source_rq3_0_manifest_sha256"
    ]
    ==
    EXPECTED_RQ3_0
)


assert (
    m[
        "source_rq3_1_manifest_sha256"
    ]
    ==
    EXPECTED_RQ3_1
)


# ------------------------------------------------------------------------------------------
# 3. Scientific controls must remain frozen
# ------------------------------------------------------------------------------------------

sc = m[
    "scientific_controls"
]


assert (
    sc[
        "test_qa_metrics_calculated"
    ]
    is False
)

assert (
    sc[
        "gold_used"
    ]
    is False
)

assert (
    sc[
        "test_performance_used_for_selection"
    ]
    is False
)

assert (
    sc[
        "seed_tuning"
    ]
    is False
)

assert (
    sc[
        "reasoner_tuning"
    ]
    is False
)

assert (
    sc[
        "model_selection"
    ]
    is False
)

assert (
    sc[
        "prompt_budget_modified"
    ]
    is False
)

assert (
    sc[
        "generation_contract_modified"
    ]
    is False
)

assert (
    sc[
        "retrieval_rerun"
    ]
    is False
)

assert (
    sc[
        "planner_rerun"
    ]
    is False
)


# ------------------------------------------------------------------------------------------
# 4. Exact frozen generation contract
# ------------------------------------------------------------------------------------------

gc = m[
    "generation_contract"
]


assert (
    gc[
        "max_new_tokens"
    ]
    ==
    512
)

assert (
    gc[
        "do_sample"
    ]
    is True
)

assert (
    gc[
        "rng_reset_immediately_before_generate"
    ]
    is True
)

assert (
    gc[
        "same_seed_for_matched_rog_afp"
    ]
    is True
)


expected_defaults = {

    "temperature":
        0.6,

    "top_p":
        0.9,

    "top_k":
        50,

    "num_beams":
        1,

    "repetition_penalty":
        1.0,

    "bos_token_id":
        1,

    "eos_token_id":
        2,

    "pad_token_id":
        0,
}


assert (
    gc[
        "inherited_generation_defaults"
    ]
    ==
    expected_defaults
)


# ------------------------------------------------------------------------------------------
# 5. Sample protocol must still be identical
# ------------------------------------------------------------------------------------------

ss = m[
    "sample_selection"
]


assert (
    ss[
        "question_ids_per_dataset"
    ]
    ==
    4
)

assert (
    ss[
        "methods_per_question"
    ]
    ==
    [
        "RoG",
        "AFP",
    ]
)

assert (
    ss[
        "primary_generation_calls"
    ]
    ==
    16
)

assert (
    ss[
        "deterministic_replay_calls"
    ]
    ==
    1
)


# ------------------------------------------------------------------------------------------
# 6. Mechanical generation outcome must remain valid
# ------------------------------------------------------------------------------------------

ca = m[
    "context_window_audit"
]


assert (
    ca[
        "actual_context_extension_seen"
    ]
    is False
)

assert (
    ca[
        "actual_input_plus_generated_gt_4096_cases"
    ]
    ==
    0
)

assert (
    ca[
        "cases_reaching_512_new_token_cap"
    ]
    ==
    0
)


# ------------------------------------------------------------------------------------------
# 7. Deterministic replay must still be exact
# ------------------------------------------------------------------------------------------

rp = m[
    "reproducibility"
]


assert (
    rp[
        "same_prompt"
    ]
    is True
)

assert (
    rp[
        "same_seed"
    ]
    is True
)

assert (
    rp[
        "generated_token_sequence_byte_exact"
    ]
    is True
)


assert (
    rp[
        "primary_generated_token_sha256"
    ]
    ==
    rp[
        "replay_generated_token_sha256"
    ]
)


# ------------------------------------------------------------------------------------------
# 8. Primary checkpoint artifact must exist and contain 16 successful rows
# ------------------------------------------------------------------------------------------

cp = Path(
    m[
        "outputs"
    ][
        "primary_generation_jsonl_gz"
    ]
)


assert (
    cp.exists()
)


assert (
    sha256_file(
        cp
    )
    ==
    m[
        "outputs"
    ][
        "primary_generation_sha256"
    ]
)


with gzip.open(
    cp,
    "rt",
    encoding="utf-8"
) as f:

    rows = [
        json.loads(
            line
        )

        for line in f

        if line.strip()
    ]


assert (
    len(
        rows
    )
    ==
    16
)


assert all(
    r[
        "generation_status"
    ]
    ==
    "OK"

    for r in rows
)


web = [
    r

    for r in rows

    if r[
        "dataset"
    ]
    ==
    "webqsp"
]


cwq = [
    r

    for r in rows

    if r[
        "dataset"
    ]
    ==
    "cwq"
]


assert (
    len(
        web
    )
    ==
    8
)


assert (
    len(
        cwq
    )
    ==
    8
)


assert all(
    r[
        "generated_tokens"
    ]
    <=
    512

    for r in rows
)


assert all(
    r[
        "actual_total_gt_4096"
    ]
    is False

    for r in rows
)


# ------------------------------------------------------------------------------------------
# 9. Live state
# ------------------------------------------------------------------------------------------

assert (
    RQ3_2_COMPLETE
    is True
)


assert (
    RQ3_2_DETERMINISTIC_REPLAY_PASSED
    is True
)


RQ3_2_RERUN_EQUIVALENT = (
    True
)


RQ3_2_CURRENT_FROZEN_MANIFEST_SHA256 = (
    actual_sha
)


print(
    "\nScientific upstream provenance:     PASSED"
)

print(
    "Generation contract:               EXACT"
)

print(
    "No TEST tuning / gold / QA use:    PASSED"
)

print(
    "16 primary generations:            PASSED"
)

print(
    "8 WebQSP checkpoints:              PASSED"
)

print(
    "8 CWQ checkpoints:                 PASSED"
)

print(
    "Deterministic replay:              BYTE-EXACT"
)

print(
    "Actual >4096 cases:                0"
)

print(
    "512-token-cap hits:                0"
)


print(
    "\nCURRENT VALID RQ3-2 MANIFEST SHA:"
)

print(
    RQ3_2_CURRENT_FROZEN_MANIFEST_SHA256
)


print(
    "\nRQ3_2_RERUN_EQUIVALENT:",
    RQ3_2_RERUN_EQUIVALENT
)

print(
    "=" * 120
)


## 7. RQ3-3 — Full WebQSP Generation

Performs the resumable frozen RoG-vs-AdaPruner downstream generation on
the full WebQSP TEST set. No QA scoring or TEST tuning occurs in this
stage.


In [ ]:

from pathlib import Path
import gzip, io, json, hashlib, os, time, warnings
import pandas as pd
import torch

print("=" * 150)
print("RQ3-3 — FULL WEBQSP FROZEN RoG-vs-AFP GENERATION")
print("=" * 150)

# --------------------------------------------------------------------------------------------------
# 0. FROZEN CONTRACTS
# --------------------------------------------------------------------------------------------------
RQ3_3_EXPECTED = {
    "rq3_0": "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8",
    "rq3_1": "76d6a927654449fe84e12ef175abe1131da44780e887bdea0890ea5c8a4c3620",
    "rq3_2": "bcf4d0a84bf983f2c0c32a6904c69242d03ead3f3446e36262dafb8869095e84",
    "rq3_2_historical_equivalent": "e9c4f558d95887a75fb8f8c381d728374dc5238bf47405e676b27347e70f200f",
    "rog_input": "db8167702384332bc0aa18815688bd4000569320329f61b65b3f3f3952c45638",
    "afp_input": "6e89afddaa32b621870bd72128476c35e705084c9001dc895a0a9d58f750c03f",
}
NQ = 1628
NCALLS = 3256

assert globals().get("RQ3_REASONER_CONTRACT_COMPLETE", False) is True
assert globals().get("RQ3_1_COMPLETE", False) is True
assert globals().get("RQ3_2_COMPLETE", False) is True
assert globals().get("RQ3_2_DETERMINISTIC_REPLAY_PASSED", False) is True
assert globals().get("RQ3_2_RERUN_EQUIVALENT", False) is True
assert str(globals().get("RQ3_2_CURRENT_FROZEN_MANIFEST_SHA256", "")) == RQ3_3_EXPECTED["rq3_2"]
assert str(RQ3_CONTRACT_MANIFEST_SHA256) == RQ3_3_EXPECTED["rq3_0"]
assert str(RQ3_1_MANIFEST_SHA256) == RQ3_3_EXPECTED["rq3_1"]
assert str(RQ3_2_MANIFEST_SHA256) == RQ3_3_EXPECTED["rq3_2"]
assert RQ3_GENERATION_KWARGS == {"max_new_tokens": 512, "do_sample": True}
assert int(RQ3_MAX_NEW_TOKENS) == 512
assert type(RQ3_TOKENIZER).__name__ == "LlamaTokenizer"
assert RQ3_TOKENIZER.is_fast is False
assert RQ3_MODEL.training is False

# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------
def fsha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(1024 * 1024), b""):
            h.update(b)
    return h.hexdigest()


def tsha(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def isha(vals):
    return tsha(json.dumps([int(x) for x in vals], separators=(",", ":")))


def read_gz(path):
    out = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                out.append(json.loads(line))
    return out


def write_gz(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as raw:
        with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz:
            with io.TextIOWrapper(gz, encoding="utf-8", newline="\n") as txt:
                for r in rows:
                    txt.write(json.dumps(r, ensure_ascii=False, separators=(",", ":")) + "\n")


def append_fsync(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")) + "\n")
        f.flush()
        os.fsync(f.fileno())


def load_working(path):
    """Allow only a truncated final line after a hard interruption; reject internal corruption."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return []
    raw = path.read_bytes()
    lines = raw.splitlines(keepends=True)
    rows, valid_end = [], 0
    for i, bline in enumerate(lines):
        if not bline.strip():
            valid_end += len(bline)
            continue
        try:
            rows.append(json.loads(bline.decode("utf-8")))
            valid_end += len(bline)
        except Exception:
            assert i == len(lines) - 1, f"STOP: internal JSONL corruption: {path}"
            print("Recovered one truncated final JSONL line after interruption:", path)
            with path.open("r+b") as f:
                f.truncate(valid_end)
                f.flush()
                os.fsync(f.fileno())
            break
    return rows


def sync_cuda():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

# --------------------------------------------------------------------------------------------------
# 2. GENERATION DEFAULTS MUST STILL MATCH RQ3-0
# --------------------------------------------------------------------------------------------------
fields = [
    "temperature", "top_p", "top_k", "num_beams", "repetition_penalty",
    "bos_token_id", "eos_token_id", "pad_token_id"
]

CURRENT_DEFAULTS = {}

for k in fields:
    v = getattr(RQ3_MODEL.generation_config, k, None)

    if isinstance(v, (str, int, float, bool)) or v is None:
        CURRENT_DEFAULTS[k] = v
    else:
        try:
            CURRENT_DEFAULTS[k] = list(v)
        except Exception:
            CURRENT_DEFAULTS[k] = str(v)

assert CURRENT_DEFAULTS == RQ3_MODEL_GENERATION_DEFAULTS

print("\nUpstream frozen-contract gates: PASSED")
print(" generation:", RQ3_GENERATION_KWARGS)
print(" inherited defaults:", CURRENT_DEFAULTS)

# --------------------------------------------------------------------------------------------------
# 3. LOAD FROZEN WEBQSP INPUTS FROM RQ3-1 ARTIFACTS
# --------------------------------------------------------------------------------------------------
assert Path(RQ3_1_MANIFEST_PATH).exists()
assert fsha(RQ3_1_MANIFEST_PATH) == RQ3_3_EXPECTED["rq3_1"]

with Path(RQ3_1_MANIFEST_PATH).open("r", encoding="utf-8") as f:
    M1 = json.load(f)

INPUT_ROWS, INPUT_PATHS = {}, {}

for method, expected_sha in [
    ("RoG", RQ3_3_EXPECTED["rog_input"]),
    ("AFP", RQ3_3_EXPECTED["afp_input"])
]:
    rec = M1["reasoner_input_artifacts"]["webqsp"][method]
    p = Path(rec["path"])

    assert p.exists()
    assert rec["sha256"] == expected_sha
    assert fsha(p) == expected_sha

    rows = read_gz(p)

    assert len(rows) == NQ == int(rec["questions"])

    for r in rows:
        assert r["dataset"] == "webqsp"
        assert r["method"] == method
        assert "gold_answers" not in r
        assert "a_entity" not in r
        assert tsha(r["prompt"]) == r["prompt_sha256"]
        assert int(r["reasoner_seed"]) == int(
            rq3_question_seed(
                "webqsp",
                r["question_id"]
            )
        )

    INPUT_ROWS[method] = rows
    INPUT_PATHS[method] = p


ROG = sorted(
    INPUT_ROWS["RoG"],
    key=lambda r: int(r["question_index"])
)

AFP = sorted(
    INPUT_ROWS["AFP"],
    key=lambda r: int(r["question_index"])
)

assert [int(r["question_index"]) for r in ROG] == list(range(NQ))
assert [int(r["question_index"]) for r in AFP] == list(range(NQ))
assert [str(r["question_id"]) for r in ROG] == [str(r["question_id"]) for r in AFP]


INPUT = {}

for method, rows in [
    ("RoG", ROG),
    ("AFP", AFP)
]:
    for r in rows:
        key = (
            method,
            str(r["question_id"])
        )

        assert key not in INPUT

        INPUT[key] = r


print("\nFrozen WebQSP reasoner-input artifacts: PASSED")
print(" RoG SHA:", RQ3_3_EXPECTED["rog_input"])
print(" AFP SHA:", RQ3_3_EXPECTED["afp_input"])
print(" questions per method:", NQ)
print(" gold in generation inputs: NO")

# --------------------------------------------------------------------------------------------------
# 4. LOAD THE 8 WEBQSP RQ3-2 CHECKPOINT GENERATIONS
# --------------------------------------------------------------------------------------------------
assert Path(RQ3_2_MANIFEST_PATH).exists()
assert fsha(RQ3_2_MANIFEST_PATH) == RQ3_3_EXPECTED["rq3_2"]

with Path(RQ3_2_MANIFEST_PATH).open("r", encoding="utf-8") as f:
    M2 = json.load(f)

cp_path = Path(
    M2["outputs"]["primary_generation_jsonl_gz"]
)

assert cp_path.exists()
assert fsha(cp_path) == M2["outputs"]["primary_generation_sha256"]

CHECKPOINTS = {}

for r in read_gz(cp_path):

    if r["dataset"] == "webqsp":

        assert r["generation_status"] == "OK"

        key = (
            r["method"],
            str(r["question_id"])
        )

        assert key not in CHECKPOINTS

        CHECKPOINTS[key] = r


assert len(CHECKPOINTS) == 8

print(
    "\nRQ3-2 WebQSP checkpoint gate: PASSED | cases=8"
)

# --------------------------------------------------------------------------------------------------
# 5. PREDECLARE FULL ORDER AND OUTPUT PATHS
# --------------------------------------------------------------------------------------------------
CASES = []

for i in range(NQ):

    qid = str(
        ROG[i]["question_id"]
    )

    assert qid == str(
        AFP[i]["question_id"]
    )

    for method in [
        "RoG",
        "AFP"
    ]:

        CASES.append(
            {
                "question_index": i,
                "question_id": qid,
                "method": method,
            }
        )


assert len(CASES) == NCALLS


ROOT = Path(
    "/kaggle/working/"
    "step4_rq3_llm_reasoning_v1/"
    "03_full_webqsp_generation"
)

ROOT.mkdir(
    parents=True,
    exist_ok=True
)

WORK = ROOT / "webqsp_full_generation_WORKING.jsonl"
ERRORS = ROOT / "webqsp_full_generation_ERRORS.jsonl"
FINAL = ROOT / "webqsp_full_generation_FINAL.jsonl.gz"
DIAG = ROOT / "webqsp_full_generation_diagnostics.csv"
SUMMARY = ROOT / "webqsp_full_generation_nonsemantic_summary.csv"
PRE = ROOT / "rq3_3_webqsp_generation_protocol_PRE.json"
MANIFEST = ROOT / "rq3_3_full_webqsp_generation_manifest.json"


PRE_PROTOCOL = {

    "stage":
        "RQ3-3",

    "dataset":
        "webqsp",

    "questions":
        NQ,

    "methods":
        [
            "RoG",
            "AFP",
        ],

    "expected_generation_calls":
        NCALLS,

    "upstream_manifest_sha256": {

        "RQ3-0":
            RQ3_3_EXPECTED["rq3_0"],

        "RQ3-1":
            RQ3_3_EXPECTED["rq3_1"],

        "RQ3-2":
            RQ3_3_EXPECTED["rq3_2"],
    },

    "rq3_2_rerun_provenance": {

        "current_authoritative_manifest_sha256":
            RQ3_3_EXPECTED["rq3_2"],

        "historical_equivalent_manifest_sha256":
            RQ3_3_EXPECTED["rq3_2_historical_equivalent"],

        "equivalence_audit_passed":
            True,

        "equivalence_basis":
            (
                "same frozen upstream provenance, decoding contract, "
                "sample protocol, deterministic token replay; runtime timing may differ"
            ),
    },

    "frozen_input_sha256": {

        "RoG":
            RQ3_3_EXPECTED["rog_input"],

        "AFP":
            RQ3_3_EXPECTED["afp_input"],
    },

    "execution_order":
        "question_index ascending; RoG then AFP",

    "generation_contract": {

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "inherited_generation_defaults":
            CURRENT_DEFAULTS,

        "rng_reset_immediately_before_every_generate":
            True,

        "seed_mapping":
            "SHA256(base_seed|dataset|question_id) mod (2^31-1)",
    },

    "scientific_controls": {

        "gold_used":
            False,

        "qa_metrics_calculated":
            False,

        "qa_outputs_used_for_selection":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,

        "seed_tuning":
            False,

        "reasoner_tuning":
            False,

        "planner_rerun":
            False,

        "retrieval_rerun":
            False,
    },

    "latency_policy":
        "operational diagnostic only; not final controlled comparative timing",
}


if PRE.exists():

    with PRE.open(
        "r",
        encoding="utf-8"
    ) as f:

        assert (
            json.load(f)
            ==
            PRE_PROTOCOL
        ), (
            "STOP: existing RQ3-3 PRE protocol differs."
        )

else:

    with PRE.open(
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            PRE_PROTOCOL,
            f,
            indent=2,
            ensure_ascii=False
        )


PRE_SHA = fsha(
    PRE
)


print(
    "\nRQ3-3 PRE protocol frozen before full generation:"
)

print(
    " ",
    PRE
)

print(
    " SHA256:",
    PRE_SHA
)

# --------------------------------------------------------------------------------------------------
# 6. VALIDATE / RESUME APPEND-ONLY OUTPUT
# --------------------------------------------------------------------------------------------------
existing = load_working(
    WORK
)

assert len(existing) <= NCALLS

seen = set()

for pos, r in enumerate(existing):

    e = CASES[pos]

    assert r["generation_status"] == "OK"
    assert r["dataset"] == "webqsp"

    assert r["method"] == e["method"]

    assert (
        int(r["question_index"])
        ==
        int(e["question_index"])
    )

    assert (
        str(r["question_id"])
        ==
        str(e["question_id"])
    )

    key = (
        r["method"],
        str(r["question_id"])
    )

    assert key not in seen
    seen.add(key)

    frozen = INPUT[key]

    assert (
        r["prompt_sha256"]
        ==
        frozen["prompt_sha256"]
    )

    assert (
        int(r["reasoner_seed"])
        ==
        int(frozen["reasoner_seed"])
    )

    assert (
        int(r["input_tokens"])
        ==
        int(frozen["model_input_tokens"])
    )

    if key in CHECKPOINTS:

        assert (
            r["generated_token_ids_sha256"]
            ==
            CHECKPOINTS[key]["generated_token_ids_sha256"]
        )

        assert (
            r["output_text_sha256"]
            ==
            CHECKPOINTS[key]["output_text_sha256"]
        )


print(
    "\nResume audit: PASSED"
)

print(
    " already completed:",
    len(existing),
    "/",
    NCALLS
)

print(
    " working file:",
    WORK
)

# --------------------------------------------------------------------------------------------------
# 7. EXACT GENERATION FUNCTION
# --------------------------------------------------------------------------------------------------
def generate_one(
    method,
    qid
):

    row = INPUT[
        (
            method,
            str(qid)
        )
    ]

    prompt = row[
        "prompt"
    ]

    assert (
        tsha(prompt)
        ==
        row["prompt_sha256"]
    )

    input_ids = RQ3_TOKENIZER.encode(
        prompt,
        return_tensors="pt"
    )

    input_tokens = int(
        input_ids.shape[1]
    )

    assert (
        input_tokens
        ==
        int(
            row["model_input_tokens"]
        )
    )

    input_ids = input_ids.to(
        RQ3_MODEL.device
    )

    seed = int(
        rq3_set_question_seed(
            "webqsp",
            qid
        )
    )

    assert (
        seed
        ==
        int(
            row["reasoner_seed"]
        )
    )

    sync_cuda()

    t0 = time.perf_counter()

    try:

        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            with torch.inference_mode():

                out = RQ3_MODEL.generate(
                    input_ids=input_ids,
                    **RQ3_GENERATION_KWARGS,
                )

            warn = [
                str(w.message)
                for w in caught
            ]

    except Exception as exc:

        sync_cuda()

        err = {

            "dataset":
                "webqsp",

            "method":
                method,

            "question_index":
                int(
                    row["question_index"]
                ),

            "question_id":
                str(qid),

            "reasoner_seed":
                seed,

            "prompt_sha256":
                row["prompt_sha256"],

            "input_tokens":
                input_tokens,

            "error_type":
                type(exc).__name__,

            "error_message":
                str(exc),

            "latency_sec_before_error":
                time.perf_counter()
                -
                t0,

            "timestamp_unix":
                time.time(),
        }

        append_fsync(
            ERRORS,
            err
        )

        raise

    sync_cuda()

    latency = (
        time.perf_counter()
        -
        t0
    )

    new_t = out[
        0
    ][
        input_tokens:
    ]

    new_ids = [
        int(x)
        for x in
        new_t.detach().cpu().tolist()
    ]

    assert len(new_ids) <= 512

    text = (
        RQ3_TOKENIZER.decode(
            new_t,
            skip_special_tokens=True
        )
        .strip()
    )

    eos = (
        RQ3_MODEL
        .generation_config
        .eos_token_id
    )

    eos_set = (
        set()
        if eos is None

        else
        (
            {
                int(x)
                for x in eos
            }
            if isinstance(
                eos,
                (list, tuple)
            )

            else
            {
                int(eos)
            }
        )
    )

    actual_total = (
        input_tokens
        +
        len(new_ids)
    )

    return {

        "dataset":
            "webqsp",

        "method":
            method,

        "question_index":
            int(
                row["question_index"]
            ),

        "question_id":
            str(qid),

        "generation_status":
            "OK",

        "reasoner_seed":
            seed,

        "rng_reset_immediately_before_generate":
            True,

        "prompt_sha256":
            row["prompt_sha256"],

        "input_tokens":
            input_tokens,

        "prompt_budget_triggered":
            bool(
                row["prompt_budget_triggered"]
            ),

        "nominal_input_plus_512":
            input_tokens
            +
            512,

        "nominal_context_risk":
            bool(
                input_tokens
                +
                512
                >
                4096
            ),

        "generated_tokens":
            len(new_ids),

        "actual_total_tokens":
            actual_total,

        "actual_total_gt_4096":
            bool(
                actual_total
                >
                4096
            ),

        "reached_max_new_tokens":
            bool(
                len(new_ids)
                ==
                512
            ),

        "eos_seen":
            bool(
                any(
                    x in eos_set
                    for x in new_ids
                )
            ),

        "latency_sec_operational":
            latency,

        "python_warnings":
            warn,

        "generated_token_ids_sha256":
            isha(
                new_ids
            ),

        "output_text_sha256":
            tsha(
                text
            ),

        "generated_token_ids":
            new_ids,

        "output_text":
            text,
    }

# --------------------------------------------------------------------------------------------------
# 8. FULL RESUMABLE GENERATION
# --------------------------------------------------------------------------------------------------
start_pos = len(
    existing
)

session_t0 = (
    time.perf_counter()
)

session_n = (
    0
)


if start_pos < NCALLS:

    print(
        "\n"
        +
        "=" * 150
    )

    print(
        "RUNNING / RESUMING FULL WEBQSP GENERATION"
    )

    print(
        "=" * 150
    )


for pos in range(
    start_pos,
    NCALLS
):

    case = CASES[
        pos
    ]

    key = (
        case["method"],
        case["question_id"]
    )

    try:

        r = generate_one(
            *key
        )

    except Exception as exc:

        print(
            "\nRQ3-3 STOPPED SAFELY ON GENERATION ERROR"
        )

        print(
            " position:",
            pos,
            "/",
            NCALLS
        )

        print(
            " q_index :",
            case["question_index"]
        )

        print(
            " qid     :",
            case["question_id"]
        )

        print(
            " method  :",
            case["method"]
        )

        print(
            " error   :",
            type(exc).__name__,
            str(exc)
        )

        print(
            "Rerun THIS SAME CELL after resolving only the runtime problem; "
            "successful rows are already fsynced."
        )

        raise


    if key in CHECKPOINTS:

        cp = CHECKPOINTS[
            key
        ]

        assert (
            r["prompt_sha256"]
            ==
            cp["prompt_sha256"]
        )

        assert (
            int(
                r["reasoner_seed"]
            )
            ==
            int(
                cp["reasoner_seed"]
            )
        )

        assert (
            r["generated_token_ids_sha256"]
            ==
            cp["generated_token_ids_sha256"]
        ), (
            f"Checkpoint replay failed: {key}"
        )

        assert (
            r["output_text_sha256"]
            ==
            cp["output_text_sha256"]
        )


    append_fsync(
        WORK,
        r
    )

    session_n += (
        1
    )

    done = (
        pos
        +
        1
    )


    if (
        done <= 10
        or
        done % 25 == 0
        or
        done == NCALLS
    ):

        elapsed = (
            time.perf_counter()
            -
            session_t0
        )

        avg = (
            elapsed
            /
            max(
                session_n,
                1
            )
        )

        eta_min = (
            (
                NCALLS
                -
                done
            )
            *
            avg
            /
            60.0
        )

        print(
            f" completed={done:4d}/{NCALLS} | "
            f"q_index={case['question_index']:4d} | "
            f"method={case['method']:3s} | "
            f"input={r['input_tokens']:4d} | "
            f"output={r['generated_tokens']:3d} | "
            f"actual>4096={r['actual_total_gt_4096']} | "
            f"avg={avg:.2f}s/call | "
            f"ETA~{eta_min:.1f} min"
        )

# --------------------------------------------------------------------------------------------------
# 9. FINAL VALIDATION + FREEZE
# --------------------------------------------------------------------------------------------------
rows = load_working(
    WORK
)

assert len(rows) == NCALLS

checkpoint_hits = (
    0
)

seen = set()


for pos, r in enumerate(
    rows
):

    e = CASES[
        pos
    ]

    assert (
        r["generation_status"]
        ==
        "OK"
    )

    assert (
        r["dataset"]
        ==
        "webqsp"
    )

    assert (
        r["method"]
        ==
        e["method"]
    )

    assert (
        int(
            r["question_index"]
        )
        ==
        e["question_index"]
    )

    assert (
        str(
            r["question_id"]
        )
        ==
        e["question_id"]
    )

    key = (
        r["method"],
        str(
            r["question_id"]
        )
    )

    assert (
        key
        not in
        seen
    )

    seen.add(
        key
    )

    frozen = INPUT[
        key
    ]

    assert (
        r["prompt_sha256"]
        ==
        frozen["prompt_sha256"]
    )

    assert (
        int(
            r["reasoner_seed"]
        )
        ==
        int(
            frozen["reasoner_seed"]
        )
    )

    assert (
        int(
            r["input_tokens"]
        )
        ==
        int(
            frozen["model_input_tokens"]
        )
    )

    assert (
        int(
            r["generated_tokens"]
        )
        <=
        512
    )

    if key in CHECKPOINTS:

        cp = CHECKPOINTS[
            key
        ]

        assert (
            r["generated_token_ids_sha256"]
            ==
            cp["generated_token_ids_sha256"]
        )

        assert (
            r["output_text_sha256"]
            ==
            cp["output_text_sha256"]
        )

        checkpoint_hits += (
            1
        )


assert (
    len(
        seen
    )
    ==
    NCALLS
)

assert (
    checkpoint_hits
    ==
    8
)

assert (
    sum(
        r["method"]
        ==
        "RoG"

        for r in rows
    )
    ==
    NQ
)

assert (
    sum(
        r["method"]
        ==
        "AFP"

        for r in rows
    )
    ==
    NQ
)


write_gz(
    rows,
    FINAL
)


FINAL_SHA = fsha(
    FINAL
)

WORK_SHA = fsha(
    WORK
)


diag_rows = [
    {
        k:
            v

        for k, v in r.items()

        if k not in {
            "output_text",
            "generated_token_ids",
            "python_warnings",
        }
    }

    for r in rows
]


RQ3_3_DIAGNOSTIC_DF = pd.DataFrame(
    diag_rows
)


RQ3_3_DIAGNOSTIC_DF.to_csv(
    DIAG,
    index=False
)


DIAG_SHA = fsha(
    DIAG
)


sum_rows = []


for method in [
    "RoG",
    "AFP"
]:

    g = RQ3_3_DIAGNOSTIC_DF[
        RQ3_3_DIAGNOSTIC_DF[
            "method"
        ]
        ==
        method
    ]

    sum_rows.append(
        {
            "method":
                method,

            "questions":
                len(g),

            "input_tokens_mean":
                float(
                    g[
                        "input_tokens"
                    ].mean()
                ),

            "input_tokens_median":
                float(
                    g[
                        "input_tokens"
                    ].median()
                ),

            "generated_tokens_mean":
                float(
                    g[
                        "generated_tokens"
                    ].mean()
                ),

            "generated_tokens_median":
                float(
                    g[
                        "generated_tokens"
                    ].median()
                ),

            "nominal_context_risk_cases":
                int(
                    g[
                        "nominal_context_risk"
                    ].sum()
                ),

            "actual_total_gt_4096_cases":
                int(
                    g[
                        "actual_total_gt_4096"
                    ].sum()
                ),

            "max_actual_total_tokens":
                int(
                    g[
                        "actual_total_tokens"
                    ].max()
                ),

            "reached_512_token_cap":
                int(
                    g[
                        "reached_max_new_tokens"
                    ].sum()
                ),

            "eos_seen_cases":
                int(
                    g[
                        "eos_seen"
                    ].sum()
                ),

            "operational_latency_mean_sec":
                float(
                    g[
                        "latency_sec_operational"
                    ].mean()
                ),

            "operational_latency_median_sec":
                float(
                    g[
                        "latency_sec_operational"
                    ].median()
                ),
        }
    )


RQ3_3_SUMMARY_DF = pd.DataFrame(
    sum_rows
)


RQ3_3_SUMMARY_DF.to_csv(
    SUMMARY,
    index=False
)


SUMMARY_SHA = fsha(
    SUMMARY
)


actual_over = int(
    RQ3_3_DIAGNOSTIC_DF[
        "actual_total_gt_4096"
    ].sum()
)


cap_hits = int(
    RQ3_3_DIAGNOSTIC_DF[
        "reached_max_new_tokens"
    ].sum()
)


historical_errors = (
    len(
        load_working(
            ERRORS
        )
    )
    if ERRORS.exists()
    else
    0
)

# --------------------------------------------------------------------------------------------------
# 10. FINAL MANIFEST
# --------------------------------------------------------------------------------------------------
RQ3_3_MANIFEST = {

    "stage":
        "RQ3-3",

    "status":
        "full_webqsp_frozen_generation_complete",

    "dataset":
        "webqsp",

    "questions":
        NQ,

    "methods":
        [
            "RoG",
            "AFP",
        ],

    "successful_generation_calls":
        NCALLS,

    "generation_errors_in_final_complete_output":
        0,

    "historical_runtime_error_records":
        historical_errors,

    "pre_generation_protocol": {

        "path":
            str(
                PRE
            ),

        "sha256":
            PRE_SHA,
    },

    "upstream_contracts": {

        "rq3_0_manifest_sha256":
            RQ3_3_EXPECTED[
                "rq3_0"
            ],

        "rq3_1_manifest_sha256":
            RQ3_3_EXPECTED[
                "rq3_1"
            ],

        "rq3_2_manifest_sha256":
            RQ3_3_EXPECTED[
                "rq3_2"
            ],

        "rq3_2_historical_equivalent_manifest_sha256":
            RQ3_3_EXPECTED[
                "rq3_2_historical_equivalent"
            ],

        "rq3_2_rerun_equivalence_audit_passed":
            True,
    },

    "frozen_input_artifacts": {

        "RoG": {

            "path":
                str(
                    INPUT_PATHS[
                        "RoG"
                    ]
                ),

            "sha256":
                RQ3_3_EXPECTED[
                    "rog_input"
                ],

            "questions":
                NQ,
        },

        "AFP": {

            "path":
                str(
                    INPUT_PATHS[
                        "AFP"
                    ]
                ),

            "sha256":
                RQ3_3_EXPECTED[
                    "afp_input"
                ],

            "questions":
                NQ,
        },
    },

    "generation_contract": {

        "model_id":
            "rmanluo/RoG",

        "tokenizer_class":
            type(
                RQ3_TOKENIZER
            ).__name__,

        "tokenizer_is_fast":
            bool(
                RQ3_TOKENIZER.is_fast
            ),

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "explicit_generation_kwargs":
            sorted(
                RQ3_GENERATION_KWARGS.keys()
            ),

        "inherited_generation_defaults":
            CURRENT_DEFAULTS,

        "rng_reset_immediately_before_every_generate":
            True,

        "seed_mapping":
            "SHA256(base_seed|dataset|question_id) mod (2^31-1)",

        "same_seed_for_matched_rog_afp":
            True,

        "prompt_truncation_added_in_rq3_3":
            False,
    },

    "execution_order": {

        "question_order":
            "question_index ascending 0..1627",

        "method_order_per_question":
            [
                "RoG",
                "AFP",
            ],

        "latency_interpretation":
            (
                "operational diagnostic only; "
                "final comparison requires controlled timing"
            ),
    },

    "reproducibility": {

        "rq3_2_webqsp_checkpoint_cases":
            8,

        "checkpoint_generated_token_hashes_exact":
            True,
    },

    "scientific_controls": {

        "gold_loaded_or_used_in_generation":
            False,

        "qa_metrics_calculated":
            False,

        "qa_semantic_outputs_used_for_selection":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,

        "seed_tuning":
            False,

        "reasoner_tuning":
            False,

        "retrieval_rerun":
            False,

        "planner_rerun":
            False,
    },

    "context_audit": {

        "nominal_model_max_position_embeddings":
            int(
                getattr(
                    RQ3_MODEL.config,
                    "max_position_embeddings",
                    4096
                )
            ),

        "actual_input_plus_generated_gt_4096_cases":
            actual_over,

        "cases_reaching_512_new_token_cap":
            cap_hits,
    },

    "outputs": {

        "append_only_working_jsonl":
            str(
                WORK
            ),

        "append_only_working_jsonl_sha256":
            WORK_SHA,

        "final_deterministic_jsonl_gz":
            str(
                FINAL
            ),

        "final_deterministic_jsonl_gz_sha256":
            FINAL_SHA,

        "diagnostic_csv":
            str(
                DIAG
            ),

        "diagnostic_csv_sha256":
            DIAG_SHA,

        "nonsemantic_summary_csv":
            str(
                SUMMARY
            ),

        "nonsemantic_summary_csv_sha256":
            SUMMARY_SHA,

        "error_jsonl":
            str(
                ERRORS
            ),
    },
}


with MANIFEST.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RQ3_3_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


RQ3_3_MANIFEST_PATH = (
    MANIFEST
)

RQ3_3_MANIFEST_SHA256 = (
    fsha(
        MANIFEST
    )
)

RQ3_3_FINAL_GZIP_PATH = (
    FINAL
)

RQ3_3_FINAL_GZIP_SHA256 = (
    FINAL_SHA
)

RQ3_3_ACTUAL_CONTEXT_EXTENSION_CASES = (
    actual_over
)

RQ3_3_512_CAP_HITS = (
    cap_hits
)

RQ3_3_COMPLETE = (
    True
)


print(
    "\n"
    +
    "=" * 150
)

print(
    "RQ3-3 FINAL WEBQSP GENERATION REPORT"
)

print(
    "=" * 150
)

print(
    "Full frozen WebQSP generation:         PASSED"
)

print(
    "RoG generations:                        1628"
)

print(
    "AFP generations:                        1628"
)

print(
    "Generation errors in final output:      0"
)

print(
    "Historical runtime-error records:       ",
    historical_errors
)

print(
    "RQ3-2 checkpoint token replays:         8/8 EXACT"
)

print(
    "Gold used in generation:                NO"
)

print(
    "Hits@1/F1 calculated:                   NO"
)

print(
    "TEST tuning/model selection:            NO"
)

print(
    "Latency interpretation:                 OPERATIONAL ONLY"
)

print(
    "\nNON-SEMANTIC GENERATION SUMMARY"
)

print(
    RQ3_3_SUMMARY_DF.to_string(
        index=False
    )
)

print(
    "\nActual input+generated >4096 cases:",
    actual_over
)

print(
    "Cases reaching 512-token cap:       ",
    cap_hits
)

print(
    "\nFrozen final output:",
    FINAL
)

print(
    "SHA256:",
    FINAL_SHA
)

print(
    "\nRQ3-3 manifest:",
    MANIFEST
)

print(
    "Manifest SHA256:",
    RQ3_3_MANIFEST_SHA256
)

print(
    "\nRQ3_3_COMPLETE:",
    RQ3_3_COMPLETE
)

print(
    "\nNEXT: RQ3-4 — FULL CWQ frozen RoG-vs-AFP generation "
    "using the same resumable protocol."
)

print(
    "=" * 150
)


## 8. RQ3-4 — Full CWQ Generation

Performs the resumable frozen RoG-vs-AdaPruner downstream generation on
the full CWQ TEST set using the same frozen reasoning contract.


In [ ]:

from pathlib import Path
import gzip, io, json, hashlib, os, time, warnings
import pandas as pd
import torch

print("=" * 150)
print("RQ3-4 — FULL CWQ FROZEN RoG-vs-AFP GENERATION")
print("=" * 150)

# --------------------------------------------------------------------------------------------------
# 0. FROZEN CONTRACTS
# --------------------------------------------------------------------------------------------------
RQ3_4_EXPECTED = {
    "rq3_0": "3d96aa6b834a059f1e6b9cafc25289536861ba1f91b777275becd0dcf4d360e8",
    "rq3_1": "76d6a927654449fe84e12ef175abe1131da44780e887bdea0890ea5c8a4c3620",
    "rq3_2": "bcf4d0a84bf983f2c0c32a6904c69242d03ead3f3446e36262dafb8869095e84",
    "rq3_2_historical_equivalent": "e9c4f558d95887a75fb8f8c381d728374dc5238bf47405e676b27347e70f200f",
    "rog_input": "cf09610b926b2994e211c122f3ec34c5da1c2b6ce72ea89c94c42d6e841b4e8e",
    "afp_input": "8d257d5b9c1c2d503cabebe1f03d6c74f6fbf910917a49f33fd17f5499ab07c6",
}

NQ = 3531
NCALLS = 7062

assert globals().get("RQ3_REASONER_CONTRACT_COMPLETE", False) is True
assert globals().get("RQ3_1_COMPLETE", False) is True
assert globals().get("RQ3_2_COMPLETE", False) is True
assert globals().get("RQ3_2_DETERMINISTIC_REPLAY_PASSED", False) is True
assert globals().get("RQ3_2_RERUN_EQUIVALENT", False) is True
assert str(globals().get("RQ3_2_CURRENT_FROZEN_MANIFEST_SHA256", "")) == RQ3_4_EXPECTED["rq3_2"]
assert str(RQ3_CONTRACT_MANIFEST_SHA256) == RQ3_4_EXPECTED["rq3_0"]
assert str(RQ3_1_MANIFEST_SHA256) == RQ3_4_EXPECTED["rq3_1"]
assert str(RQ3_2_MANIFEST_SHA256) == RQ3_4_EXPECTED["rq3_2"]
assert RQ3_GENERATION_KWARGS == {"max_new_tokens": 512, "do_sample": True}
assert int(RQ3_MAX_NEW_TOKENS) == 512
assert type(RQ3_TOKENIZER).__name__ == "LlamaTokenizer"
assert RQ3_TOKENIZER.is_fast is False
assert RQ3_MODEL.training is False


# --------------------------------------------------------------------------------------------------
# 1. HELPERS
# --------------------------------------------------------------------------------------------------
def fsha(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for b in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(b)

    return h.hexdigest()


def tsha(text):
    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()


def isha(vals):
    return tsha(
        json.dumps(
            [int(x) for x in vals],
            separators=(",", ":")
        )
    )


def read_gz(path):
    out = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8"
    ) as f:

        for line in f:
            if line.strip():
                out.append(
                    json.loads(line)
                )

    return out


def write_gz(rows, path):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with path.open("wb") as raw:

        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw,
            mtime=0
        ) as gz:

            with io.TextIOWrapper(
                gz,
                encoding="utf-8",
                newline="\n"
            ) as txt:

                for r in rows:
                    txt.write(
                        json.dumps(
                            r,
                            ensure_ascii=False,
                            separators=(",", ":")
                        )
                        +
                        "\n"
                    )


def append_fsync(path, row):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with path.open(
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                row,
                ensure_ascii=False,
                separators=(",", ":")
            )
            +
            "\n"
        )

        f.flush()
        os.fsync(
            f.fileno()
        )


def load_working(path):
    """
    Allow only a truncated final line after a hard interruption;
    reject internal corruption.
    """

    path = Path(path)

    if (
        not path.exists()
        or
        path.stat().st_size == 0
    ):
        return []

    raw = path.read_bytes()

    lines = raw.splitlines(
        keepends=True
    )

    rows = []
    valid_end = 0

    for i, bline in enumerate(lines):

        if not bline.strip():
            valid_end += len(bline)
            continue

        try:
            rows.append(
                json.loads(
                    bline.decode("utf-8")
                )
            )

            valid_end += len(bline)

        except Exception:

            assert (
                i
                ==
                len(lines) - 1
            ), (
                f"STOP: internal JSONL corruption: {path}"
            )

            print(
                "Recovered one truncated final JSONL line after interruption:",
                path
            )

            with path.open(
                "r+b"
            ) as f:

                f.truncate(
                    valid_end
                )

                f.flush()
                os.fsync(
                    f.fileno()
                )

            break

    return rows


def sync_cuda():

    for i in range(
        torch.cuda.device_count()
    ):
        torch.cuda.synchronize(i)


# --------------------------------------------------------------------------------------------------
# 2. GENERATION DEFAULTS MUST STILL MATCH RQ3-0
# --------------------------------------------------------------------------------------------------
fields = [
    "temperature",
    "top_p",
    "top_k",
    "num_beams",
    "repetition_penalty",
    "bos_token_id",
    "eos_token_id",
    "pad_token_id",
]

CURRENT_DEFAULTS = {}

for k in fields:

    v = getattr(
        RQ3_MODEL.generation_config,
        k,
        None
    )

    if isinstance(
        v,
        (
            str,
            int,
            float,
            bool
        )
    ) or v is None:

        CURRENT_DEFAULTS[k] = v

    else:

        try:
            CURRENT_DEFAULTS[k] = list(v)

        except Exception:
            CURRENT_DEFAULTS[k] = str(v)


assert (
    CURRENT_DEFAULTS
    ==
    RQ3_MODEL_GENERATION_DEFAULTS
)


print(
    "\nUpstream frozen-contract gates: PASSED"
)

print(
    " generation:",
    RQ3_GENERATION_KWARGS
)

print(
    " inherited defaults:",
    CURRENT_DEFAULTS
)


# --------------------------------------------------------------------------------------------------
# 3. LOAD FROZEN CWQ INPUTS FROM RQ3-1 ARTIFACTS
# --------------------------------------------------------------------------------------------------
assert Path(
    RQ3_1_MANIFEST_PATH
).exists()


assert (
    fsha(
        RQ3_1_MANIFEST_PATH
    )
    ==
    RQ3_4_EXPECTED[
        "rq3_1"
    ]
)


with Path(
    RQ3_1_MANIFEST_PATH
).open(
    "r",
    encoding="utf-8"
) as f:

    M1 = json.load(f)


INPUT_ROWS = {}
INPUT_PATHS = {}


for method, expected_sha in [

    (
        "RoG",
        RQ3_4_EXPECTED[
            "rog_input"
        ]
    ),

    (
        "AFP",
        RQ3_4_EXPECTED[
            "afp_input"
        ]
    ),

]:

    rec = (
        M1[
            "reasoner_input_artifacts"
        ][
            "cwq"
        ][
            method
        ]
    )


    p = Path(
        rec[
            "path"
        ]
    )


    assert (
        p.exists()
    )

    assert (
        rec[
            "sha256"
        ]
        ==
        expected_sha
    )

    assert (
        fsha(
            p
        )
        ==
        expected_sha
    )


    rows = read_gz(
        p
    )


    assert (
        len(
            rows
        )
        ==
        NQ
        ==
        int(
            rec[
                "questions"
            ]
        )
    )


    for r in rows:

        assert (
            r[
                "dataset"
            ]
            ==
            "cwq"
        )

        assert (
            r[
                "method"
            ]
            ==
            method
        )

        assert (
            "gold_answers"
            not in
            r
        )

        assert (
            "a_entity"
            not in
            r
        )

        assert (
            tsha(
                r[
                    "prompt"
                ]
            )
            ==
            r[
                "prompt_sha256"
            ]
        )

        assert (
            int(
                r[
                    "reasoner_seed"
                ]
            )
            ==
            int(
                rq3_question_seed(
                    "cwq",
                    r[
                        "question_id"
                    ]
                )
            )
        )


    INPUT_ROWS[
        method
    ] = rows

    INPUT_PATHS[
        method
    ] = p


ROG = sorted(
    INPUT_ROWS[
        "RoG"
    ],
    key=lambda r: int(
        r[
            "question_index"
        ]
    )
)


AFP = sorted(
    INPUT_ROWS[
        "AFP"
    ],
    key=lambda r: int(
        r[
            "question_index"
        ]
    )
)


assert (
    [
        int(
            r[
                "question_index"
            ]
        )
        for r in ROG
    ]
    ==
    list(
        range(
            NQ
        )
    )
)


assert (
    [
        int(
            r[
                "question_index"
            ]
        )
        for r in AFP
    ]
    ==
    list(
        range(
            NQ
        )
    )
)


assert (
    [
        str(
            r[
                "question_id"
            ]
        )
        for r in ROG
    ]
    ==
    [
        str(
            r[
                "question_id"
            ]
        )
        for r in AFP
    ]
)


INPUT = {}


for method, rows in [

    (
        "RoG",
        ROG
    ),

    (
        "AFP",
        AFP
    ),

]:

    for r in rows:

        key = (
            method,
            str(
                r[
                    "question_id"
                ]
            )
        )


        assert (
            key
            not in
            INPUT
        )


        INPUT[
            key
        ] = r


print(
    "\nFrozen CWQ reasoner-input artifacts: PASSED"
)

print(
    " RoG SHA:",
    RQ3_4_EXPECTED[
        "rog_input"
    ]
)

print(
    " AFP SHA:",
    RQ3_4_EXPECTED[
        "afp_input"
    ]
)

print(
    " questions per method:",
    NQ
)

print(
    " gold in generation inputs: NO"
)


# --------------------------------------------------------------------------------------------------
# 4. LOAD THE 8 CWQ RQ3-2 CHECKPOINT GENERATIONS
# --------------------------------------------------------------------------------------------------
assert Path(
    RQ3_2_MANIFEST_PATH
).exists()


assert (
    fsha(
        RQ3_2_MANIFEST_PATH
    )
    ==
    RQ3_4_EXPECTED[
        "rq3_2"
    ]
)


with Path(
    RQ3_2_MANIFEST_PATH
).open(
    "r",
    encoding="utf-8"
) as f:

    M2 = json.load(f)


cp_path = Path(
    M2[
        "outputs"
    ][
        "primary_generation_jsonl_gz"
    ]
)


assert (
    cp_path.exists()
)


assert (
    fsha(
        cp_path
    )
    ==
    M2[
        "outputs"
    ][
        "primary_generation_sha256"
    ]
)


CHECKPOINTS = {}


for r in read_gz(
    cp_path
):

    if (
        r[
            "dataset"
        ]
        ==
        "cwq"
    ):

        assert (
            r[
                "generation_status"
            ]
            ==
            "OK"
        )

        key = (
            r[
                "method"
            ],
            str(
                r[
                    "question_id"
                ]
            )
        )

        assert (
            key
            not in
            CHECKPOINTS
        )

        CHECKPOINTS[
            key
        ] = r


assert (
    len(
        CHECKPOINTS
    )
    ==
    8
)


print(
    "\nRQ3-2 CWQ checkpoint gate: PASSED | cases=8"
)


# --------------------------------------------------------------------------------------------------
# 5. PREDECLARE FULL ORDER AND OUTPUT PATHS
# --------------------------------------------------------------------------------------------------
CASES = []


for i in range(
    NQ
):

    qid = str(
        ROG[
            i
        ][
            "question_id"
        ]
    )


    assert (
        qid
        ==
        str(
            AFP[
                i
            ][
                "question_id"
            ]
        )
    )


    for method in [

        "RoG",
        "AFP",

    ]:

        CASES.append(
            {
                "question_index":
                    i,

                "question_id":
                    qid,

                "method":
                    method,
            }
        )


assert (
    len(
        CASES
    )
    ==
    NCALLS
)


ROOT = Path(
    "/kaggle/working/"
    "step4_rq3_llm_reasoning_v1/"
    "04_full_cwq_generation"
)


ROOT.mkdir(
    parents=True,
    exist_ok=True
)


WORK = (
    ROOT
    /
    "cwq_full_generation_WORKING.jsonl"
)


ERRORS = (
    ROOT
    /
    "cwq_full_generation_ERRORS.jsonl"
)


FINAL = (
    ROOT
    /
    "cwq_full_generation_FINAL.jsonl.gz"
)


DIAG = (
    ROOT
    /
    "cwq_full_generation_diagnostics.csv"
)


SUMMARY = (
    ROOT
    /
    "cwq_full_generation_nonsemantic_summary.csv"
)


PRE = (
    ROOT
    /
    "rq3_4_cwq_generation_protocol_PRE.json"
)


MANIFEST = (
    ROOT
    /
    "rq3_4_full_cwq_generation_manifest.json"
)


PRE_PROTOCOL = {

    "stage":
        "RQ3-4",

    "dataset":
        "cwq",

    "questions":
        NQ,

    "methods":
        [
            "RoG",
            "AFP",
        ],

    "expected_generation_calls":
        NCALLS,

    "upstream_manifest_sha256": {

        "RQ3-0":
            RQ3_4_EXPECTED[
                "rq3_0"
            ],

        "RQ3-1":
            RQ3_4_EXPECTED[
                "rq3_1"
            ],

        "RQ3-2":
            RQ3_4_EXPECTED[
                "rq3_2"
            ],
    },

    "rq3_2_rerun_provenance": {

        "current_authoritative_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_2"
            ],

        "historical_equivalent_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_2_historical_equivalent"
            ],

        "equivalence_audit_passed":
            True,

        "equivalence_basis":
            (
                "same frozen upstream provenance, decoding contract, "
                "sample protocol, deterministic token replay; runtime timing may differ"
            ),
    },

    "frozen_input_sha256": {

        "RoG":
            RQ3_4_EXPECTED[
                "rog_input"
            ],

        "AFP":
            RQ3_4_EXPECTED[
                "afp_input"
            ],
    },

    "execution_order":
        "question_index ascending; RoG then AFP",

    "generation_contract": {

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "inherited_generation_defaults":
            CURRENT_DEFAULTS,

        "rng_reset_immediately_before_every_generate":
            True,

        "seed_mapping":
            (
                "SHA256(base_seed|dataset|question_id) "
                "mod (2^31-1)"
            ),
    },

    "scientific_controls": {

        "gold_used":
            False,

        "qa_metrics_calculated":
            False,

        "qa_outputs_used_for_selection":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,

        "seed_tuning":
            False,

        "reasoner_tuning":
            False,

        "planner_rerun":
            False,

        "retrieval_rerun":
            False,
    },

    "latency_policy":
        (
            "operational diagnostic only; "
            "not final controlled comparative timing"
        ),
}


if PRE.exists():

    with PRE.open(
        "r",
        encoding="utf-8"
    ) as f:

        assert (
            json.load(
                f
            )
            ==
            PRE_PROTOCOL
        ), (
            "STOP: existing RQ3-4 PRE protocol differs."
        )


else:

    with PRE.open(
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            PRE_PROTOCOL,
            f,
            indent=2,
            ensure_ascii=False
        )


PRE_SHA = fsha(
    PRE
)


print(
    "\nRQ3-4 PRE protocol frozen before full generation:"
)

print(
    " ",
    PRE
)

print(
    " SHA256:",
    PRE_SHA
)


# --------------------------------------------------------------------------------------------------
# 6. VALIDATE / RESUME APPEND-ONLY OUTPUT
# --------------------------------------------------------------------------------------------------
existing = load_working(
    WORK
)


assert (
    len(
        existing
    )
    <=
    NCALLS
)


seen = set()


for pos, r in enumerate(
    existing
):

    e = CASES[
        pos
    ]


    assert (
        r[
            "generation_status"
        ]
        ==
        "OK"
    )


    assert (
        r[
            "dataset"
        ]
        ==
        "cwq"
    )


    assert (
        r[
            "method"
        ]
        ==
        e[
            "method"
        ]
    )


    assert (
        int(
            r[
                "question_index"
            ]
        )
        ==
        int(
            e[
                "question_index"
            ]
        )
    )


    assert (
        str(
            r[
                "question_id"
            ]
        )
        ==
        str(
            e[
                "question_id"
            ]
        )
    )


    key = (
        r[
            "method"
        ],
        str(
            r[
                "question_id"
            ]
        )
    )


    assert (
        key
        not in
        seen
    )


    seen.add(
        key
    )


    frozen = INPUT[
        key
    ]


    assert (
        r[
            "prompt_sha256"
        ]
        ==
        frozen[
            "prompt_sha256"
        ]
    )


    assert (
        int(
            r[
                "reasoner_seed"
            ]
        )
        ==
        int(
            frozen[
                "reasoner_seed"
            ]
        )
    )


    assert (
        int(
            r[
                "input_tokens"
            ]
        )
        ==
        int(
            frozen[
                "model_input_tokens"
            ]
        )
    )


    if (
        key
        in
        CHECKPOINTS
    ):

        assert (
            r[
                "generated_token_ids_sha256"
            ]
            ==
            CHECKPOINTS[
                key
            ][
                "generated_token_ids_sha256"
            ]
        )


        assert (
            r[
                "output_text_sha256"
            ]
            ==
            CHECKPOINTS[
                key
            ][
                "output_text_sha256"
            ]
        )


print(
    "\nResume audit: PASSED"
)

print(
    " already completed:",
    len(
        existing
    ),
    "/",
    NCALLS
)

print(
    " working file:",
    WORK
)


# --------------------------------------------------------------------------------------------------
# 7. EXACT GENERATION FUNCTION
# --------------------------------------------------------------------------------------------------
def generate_one(
    method,
    qid
):

    row = INPUT[
        (
            method,
            str(
                qid
            )
        )
    ]


    prompt = row[
        "prompt"
    ]


    assert (
        tsha(
            prompt
        )
        ==
        row[
            "prompt_sha256"
        ]
    )


    input_ids = (
        RQ3_TOKENIZER.encode(
            prompt,
            return_tensors="pt"
        )
    )


    input_tokens = int(
        input_ids.shape[
            1
        ]
    )


    assert (
        input_tokens
        ==
        int(
            row[
                "model_input_tokens"
            ]
        )
    )


    input_ids = (
        input_ids.to(
            RQ3_MODEL.device
        )
    )


    seed = int(
        rq3_set_question_seed(
            "cwq",
            qid
        )
    )


    assert (
        seed
        ==
        int(
            row[
                "reasoner_seed"
            ]
        )
    )


    sync_cuda()

    t0 = (
        time.perf_counter()
    )


    try:

        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )


            with torch.inference_mode():

                out = (
                    RQ3_MODEL.generate(
                        input_ids=
                            input_ids,

                        **RQ3_GENERATION_KWARGS,
                    )
                )


            warn = [
                str(
                    w.message
                )
                for w in caught
            ]


    except Exception as exc:

        sync_cuda()


        err = {

            "dataset":
                "cwq",

            "method":
                method,

            "question_index":
                int(
                    row[
                        "question_index"
                    ]
                ),

            "question_id":
                str(
                    qid
                ),

            "reasoner_seed":
                seed,

            "prompt_sha256":
                row[
                    "prompt_sha256"
                ],

            "input_tokens":
                input_tokens,

            "error_type":
                type(
                    exc
                ).__name__,

            "error_message":
                str(
                    exc
                ),

            "latency_sec_before_error":
                (
                    time.perf_counter()
                    -
                    t0
                ),

            "timestamp_unix":
                time.time(),
        }


        append_fsync(
            ERRORS,
            err
        )


        raise


    sync_cuda()


    latency = (
        time.perf_counter()
        -
        t0
    )


    new_t = (
        out[
            0
        ][
            input_tokens:
        ]
    )


    new_ids = [
        int(
            x
        )
        for x in (
            new_t
            .detach()
            .cpu()
            .tolist()
        )
    ]


    assert (
        len(
            new_ids
        )
        <=
        512
    )


    text = (
        RQ3_TOKENIZER.decode(
            new_t,
            skip_special_tokens=True
        )
        .strip()
    )


    eos = (
        RQ3_MODEL
        .generation_config
        .eos_token_id
    )


    eos_set = (
        set()
        if eos is None
        else
        (
            {
                int(x)
                for x in eos
            }
            if isinstance(
                eos,
                (
                    list,
                    tuple
                )
            )
            else
            {
                int(
                    eos
                )
            }
        )
    )


    actual_total = (
        input_tokens
        +
        len(
            new_ids
        )
    )


    return {

        "dataset":
            "cwq",

        "method":
            method,

        "question_index":
            int(
                row[
                    "question_index"
                ]
            ),

        "question_id":
            str(
                qid
            ),

        "generation_status":
            "OK",

        "reasoner_seed":
            seed,

        "rng_reset_immediately_before_generate":
            True,

        "prompt_sha256":
            row[
                "prompt_sha256"
            ],

        "input_tokens":
            input_tokens,

        "prompt_budget_triggered":
            bool(
                row[
                    "prompt_budget_triggered"
                ]
            ),

        "nominal_input_plus_512":
            (
                input_tokens
                +
                512
            ),

        "nominal_context_risk":
            bool(
                input_tokens
                +
                512
                >
                4096
            ),

        "generated_tokens":
            len(
                new_ids
            ),

        "actual_total_tokens":
            actual_total,

        "actual_total_gt_4096":
            bool(
                actual_total
                >
                4096
            ),

        "reached_max_new_tokens":
            bool(
                len(
                    new_ids
                )
                ==
                512
            ),

        "eos_seen":
            bool(
                any(
                    x in eos_set
                    for x in new_ids
                )
            ),

        "latency_sec_operational":
            latency,

        "python_warnings":
            warn,

        "generated_token_ids_sha256":
            isha(
                new_ids
            ),

        "output_text_sha256":
            tsha(
                text
            ),

        "generated_token_ids":
            new_ids,

        "output_text":
            text,
    }


# --------------------------------------------------------------------------------------------------
# 8. FULL RESUMABLE GENERATION
# --------------------------------------------------------------------------------------------------
start_pos = len(
    existing
)

session_t0 = (
    time.perf_counter()
)

session_n = (
    0
)


if (
    start_pos
    <
    NCALLS
):

    print(
        "\n"
        +
        "=" * 150
    )

    print(
        "RUNNING / RESUMING FULL CWQ GENERATION"
    )

    print(
        "=" * 150
    )


for pos in range(
    start_pos,
    NCALLS
):

    case = CASES[
        pos
    ]


    key = (
        case[
            "method"
        ],
        case[
            "question_id"
        ]
    )


    try:

        r = generate_one(
            *key
        )


    except Exception as exc:

        print(
            "\nRQ3-4 STOPPED SAFELY ON GENERATION ERROR"
        )

        print(
            " position:",
            pos,
            "/",
            NCALLS
        )

        print(
            " q_index :",
            case[
                "question_index"
            ]
        )

        print(
            " qid     :",
            case[
                "question_id"
            ]
        )

        print(
            " method  :",
            case[
                "method"
            ]
        )

        print(
            " error   :",
            type(
                exc
            ).__name__,
            str(
                exc
            )
        )

        print(
            "Rerun THIS SAME CELL after resolving only the runtime problem; "
            "successful rows are already fsynced."
        )

        raise


    if (
        key
        in
        CHECKPOINTS
    ):

        cp = CHECKPOINTS[
            key
        ]


        assert (
            r[
                "prompt_sha256"
            ]
            ==
            cp[
                "prompt_sha256"
            ]
        )


        assert (
            int(
                r[
                    "reasoner_seed"
                ]
            )
            ==
            int(
                cp[
                    "reasoner_seed"
                ]
            )
        )


        assert (
            r[
                "generated_token_ids_sha256"
            ]
            ==
            cp[
                "generated_token_ids_sha256"
            ]
        ), (
            f"Checkpoint replay failed: {key}"
        )


        assert (
            r[
                "output_text_sha256"
            ]
            ==
            cp[
                "output_text_sha256"
            ]
        )


    append_fsync(
        WORK,
        r
    )


    session_n += (
        1
    )


    done = (
        pos
        +
        1
    )


    if (
        done
        <=
        10

        or

        done
        %
        25
        ==
        0

        or

        done
        ==
        NCALLS
    ):

        elapsed = (
            time.perf_counter()
            -
            session_t0
        )


        avg = (
            elapsed
            /
            max(
                session_n,
                1
            )
        )


        eta_min = (
            (
                NCALLS
                -
                done
            )
            *
            avg
            /
            60.0
        )


        print(
            f" completed={done:4d}/{NCALLS} | "
            f"q_index={case['question_index']:4d} | "
            f"method={case['method']:3s} | "
            f"input={r['input_tokens']:4d} | "
            f"output={r['generated_tokens']:3d} | "
            f"actual>4096={r['actual_total_gt_4096']} | "
            f"avg={avg:.2f}s/call | "
            f"ETA~{eta_min:.1f} min"
        )


# --------------------------------------------------------------------------------------------------
# 9. FINAL VALIDATION + FREEZE
# --------------------------------------------------------------------------------------------------
rows = load_working(
    WORK
)


assert (
    len(
        rows
    )
    ==
    NCALLS
)


checkpoint_hits = (
    0
)


seen = set()


for pos, r in enumerate(
    rows
):

    e = CASES[
        pos
    ]


    assert (
        r[
            "generation_status"
        ]
        ==
        "OK"
    )


    assert (
        r[
            "dataset"
        ]
        ==
        "cwq"
    )


    assert (
        r[
            "method"
        ]
        ==
        e[
            "method"
        ]
    )


    assert (
        int(
            r[
                "question_index"
            ]
        )
        ==
        e[
            "question_index"
        ]
    )


    assert (
        str(
            r[
                "question_id"
            ]
        )
        ==
        e[
            "question_id"
        ]
    )


    key = (
        r[
            "method"
        ],
        str(
            r[
                "question_id"
            ]
        )
    )


    assert (
        key
        not in
        seen
    )


    seen.add(
        key
    )


    frozen = INPUT[
        key
    ]


    assert (
        r[
            "prompt_sha256"
        ]
        ==
        frozen[
            "prompt_sha256"
        ]
    )


    assert (
        int(
            r[
                "reasoner_seed"
            ]
        )
        ==
        int(
            frozen[
                "reasoner_seed"
            ]
        )
    )


    assert (
        int(
            r[
                "input_tokens"
            ]
        )
        ==
        int(
            frozen[
                "model_input_tokens"
            ]
        )
    )


    assert (
        int(
            r[
                "generated_tokens"
            ]
        )
        <=
        512
    )


    if (
        key
        in
        CHECKPOINTS
    ):

        cp = CHECKPOINTS[
            key
        ]


        assert (
            r[
                "generated_token_ids_sha256"
            ]
            ==
            cp[
                "generated_token_ids_sha256"
            ]
        )


        assert (
            r[
                "output_text_sha256"
            ]
            ==
            cp[
                "output_text_sha256"
            ]
        )


        checkpoint_hits += (
            1
        )


assert (
    len(
        seen
    )
    ==
    NCALLS
)


assert (
    checkpoint_hits
    ==
    8
)


assert (
    sum(
        r[
            "method"
        ]
        ==
        "RoG"
        for r in rows
    )
    ==
    NQ
)


assert (
    sum(
        r[
            "method"
        ]
        ==
        "AFP"
        for r in rows
    )
    ==
    NQ
)


write_gz(
    rows,
    FINAL
)


FINAL_SHA = fsha(
    FINAL
)


WORK_SHA = fsha(
    WORK
)


diag_rows = [
    {
        k:
            v

        for k, v in r.items()

        if k
        not in
        {
            "output_text",
            "generated_token_ids",
            "python_warnings",
        }
    }

    for r in rows
]


RQ3_4_DIAGNOSTIC_DF = pd.DataFrame(
    diag_rows
)


RQ3_4_DIAGNOSTIC_DF.to_csv(
    DIAG,
    index=False
)


DIAG_SHA = fsha(
    DIAG
)


sum_rows = []


for method in [

    "RoG",
    "AFP",

]:

    g = (
        RQ3_4_DIAGNOSTIC_DF[
            RQ3_4_DIAGNOSTIC_DF[
                "method"
            ]
            ==
            method
        ]
    )


    sum_rows.append(
        {

            "method":
                method,

            "questions":
                len(
                    g
                ),

            "input_tokens_mean":
                float(
                    g[
                        "input_tokens"
                    ].mean()
                ),

            "input_tokens_median":
                float(
                    g[
                        "input_tokens"
                    ].median()
                ),

            "generated_tokens_mean":
                float(
                    g[
                        "generated_tokens"
                    ].mean()
                ),

            "generated_tokens_median":
                float(
                    g[
                        "generated_tokens"
                    ].median()
                ),

            "nominal_context_risk_cases":
                int(
                    g[
                        "nominal_context_risk"
                    ].sum()
                ),

            "actual_total_gt_4096_cases":
                int(
                    g[
                        "actual_total_gt_4096"
                    ].sum()
                ),

            "max_actual_total_tokens":
                int(
                    g[
                        "actual_total_tokens"
                    ].max()
                ),

            "reached_512_token_cap":
                int(
                    g[
                        "reached_max_new_tokens"
                    ].sum()
                ),

            "eos_seen_cases":
                int(
                    g[
                        "eos_seen"
                    ].sum()
                ),

            "operational_latency_mean_sec":
                float(
                    g[
                        "latency_sec_operational"
                    ].mean()
                ),

            "operational_latency_median_sec":
                float(
                    g[
                        "latency_sec_operational"
                    ].median()
                ),
        }
    )


RQ3_4_SUMMARY_DF = pd.DataFrame(
    sum_rows
)


RQ3_4_SUMMARY_DF.to_csv(
    SUMMARY,
    index=False
)


SUMMARY_SHA = fsha(
    SUMMARY
)


actual_over = int(
    RQ3_4_DIAGNOSTIC_DF[
        "actual_total_gt_4096"
    ].sum()
)


cap_hits = int(
    RQ3_4_DIAGNOSTIC_DF[
        "reached_max_new_tokens"
    ].sum()
)


historical_errors = (
    len(
        load_working(
            ERRORS
        )
    )
    if ERRORS.exists()
    else
    0
)


# --------------------------------------------------------------------------------------------------
# 10. FINAL MANIFEST
# --------------------------------------------------------------------------------------------------
RQ3_4_MANIFEST = {

    "stage":
        "RQ3-4",

    "status":
        "full_cwq_frozen_generation_complete",

    "dataset":
        "cwq",

    "questions":
        NQ,

    "methods":
        [
            "RoG",
            "AFP",
        ],

    "successful_generation_calls":
        NCALLS,

    "generation_errors_in_final_complete_output":
        0,

    "historical_runtime_error_records":
        historical_errors,

    "pre_generation_protocol": {

        "path":
            str(
                PRE
            ),

        "sha256":
            PRE_SHA,
    },

    "upstream_contracts": {

        "rq3_0_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_0"
            ],

        "rq3_1_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_1"
            ],

        "rq3_2_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_2"
            ],

        "rq3_2_historical_equivalent_manifest_sha256":
            RQ3_4_EXPECTED[
                "rq3_2_historical_equivalent"
            ],

        "rq3_2_rerun_equivalence_audit_passed":
            True,
    },

    "frozen_input_artifacts": {

        "RoG": {

            "path":
                str(
                    INPUT_PATHS[
                        "RoG"
                    ]
                ),

            "sha256":
                RQ3_4_EXPECTED[
                    "rog_input"
                ],

            "questions":
                NQ,
        },

        "AFP": {

            "path":
                str(
                    INPUT_PATHS[
                        "AFP"
                    ]
                ),

            "sha256":
                RQ3_4_EXPECTED[
                    "afp_input"
                ],

            "questions":
                NQ,
        },
    },

    "generation_contract": {

        "model_id":
            "rmanluo/RoG",

        "tokenizer_class":
            type(
                RQ3_TOKENIZER
            ).__name__,

        "tokenizer_is_fast":
            bool(
                RQ3_TOKENIZER.is_fast
            ),

        "max_new_tokens":
            512,

        "do_sample":
            True,

        "explicit_generation_kwargs":
            sorted(
                RQ3_GENERATION_KWARGS.keys()
            ),

        "inherited_generation_defaults":
            CURRENT_DEFAULTS,

        "rng_reset_immediately_before_every_generate":
            True,

        "seed_mapping":
            (
                "SHA256(base_seed|dataset|question_id) "
                "mod (2^31-1)"
            ),

        "same_seed_for_matched_rog_afp":
            True,

        "prompt_truncation_added_in_rq3_4":
            False,
    },

    "execution_order": {

        "question_order":
            (
                "question_index ascending 0..3530"
            ),

        "method_order_per_question":
            [
                "RoG",
                "AFP",
            ],

        "latency_interpretation":
            (
                "operational diagnostic only; "
                "final comparison requires controlled timing"
            ),
    },

    "reproducibility": {

        "rq3_2_cwq_checkpoint_cases":
            8,

        "checkpoint_generated_token_hashes_exact":
            True,
    },

    "scientific_controls": {

        "gold_loaded_or_used_in_generation":
            False,

        "qa_metrics_calculated":
            False,

        "qa_semantic_outputs_used_for_selection":
            False,

        "test_tuning":
            False,

        "test_model_selection":
            False,

        "seed_tuning":
            False,

        "reasoner_tuning":
            False,

        "retrieval_rerun":
            False,

        "planner_rerun":
            False,
    },

    "context_audit": {

        "nominal_model_max_position_embeddings":
            int(
                getattr(
                    RQ3_MODEL.config,
                    "max_position_embeddings",
                    4096
                )
            ),

        "actual_input_plus_generated_gt_4096_cases":
            actual_over,

        "cases_reaching_512_new_token_cap":
            cap_hits,
    },

    "outputs": {

        "append_only_working_jsonl":
            str(
                WORK
            ),

        "append_only_working_jsonl_sha256":
            WORK_SHA,

        "final_deterministic_jsonl_gz":
            str(
                FINAL
            ),

        "final_deterministic_jsonl_gz_sha256":
            FINAL_SHA,

        "diagnostic_csv":
            str(
                DIAG
            ),

        "diagnostic_csv_sha256":
            DIAG_SHA,

        "nonsemantic_summary_csv":
            str(
                SUMMARY
            ),

        "nonsemantic_summary_csv_sha256":
            SUMMARY_SHA,

        "error_jsonl":
            str(
                ERRORS
            ),
    },
}


with MANIFEST.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        RQ3_4_MANIFEST,
        f,
        indent=2,
        ensure_ascii=False
    )


RQ3_4_MANIFEST_PATH = (
    MANIFEST
)


RQ3_4_MANIFEST_SHA256 = (
    fsha(
        MANIFEST
    )
)


RQ3_4_FINAL_GZIP_PATH = (
    FINAL
)


RQ3_4_FINAL_GZIP_SHA256 = (
    FINAL_SHA
)


RQ3_4_ACTUAL_CONTEXT_EXTENSION_CASES = (
    actual_over
)


RQ3_4_512_CAP_HITS = (
    cap_hits
)


RQ3_4_COMPLETE = (
    True
)


print(
    "\n"
    +
    "=" * 150
)

print(
    "RQ3-4 FINAL CWQ GENERATION REPORT"
)

print(
    "=" * 150
)

print(
    "Full frozen CWQ generation:         PASSED"
)

print(
    "RoG generations:                        3531"
)

print(
    "AFP generations:                        3531"
)

print(
    "Generation errors in final output:      0"
)

print(
    "Historical runtime-error records:       ",
    historical_errors
)

print(
    "RQ3-2 checkpoint token replays:         8/8 EXACT"
)

print(
    "Gold used in generation:                NO"
)

print(
    "Hits@1/F1 calculated:                   NO"
)

print(
    "TEST tuning/model selection:            NO"
)

print(
    "Latency interpretation:                 OPERATIONAL ONLY"
)

print(
    "\nNON-SEMANTIC GENERATION SUMMARY"
)

print(
    RQ3_4_SUMMARY_DF.to_string(
        index=False
    )
)

print(
    "\nActual input+generated >4096 cases:",
    actual_over
)

print(
    "Cases reaching 512-token cap:       ",
    cap_hits
)

print(
    "\nFrozen final output:",
    FINAL
)

print(
    "SHA256:",
    FINAL_SHA
)

print(
    "\nRQ3-4 manifest:",
    MANIFEST
)

print(
    "Manifest SHA256:",
    RQ3_4_MANIFEST_SHA256
)

print(
    "\nRQ3_4_COMPLETE:",
    RQ3_4_COMPLETE
)

print(
    "\nNEXT: RQ3-5 — CWQ Hits@1/F1 evaluation after full frozen generation."
)

print(
    "=" * 150
)


## 9. Full-Generation Completion Check

Checks that all expected WebQSP and CWQ RoG/AdaPruner generation records
are present.


In [ ]:

from pathlib import Path

files = {
    "WebQSP": Path(
        "/kaggle/working/step4_rq3_llm_reasoning_v1/"
        "03_full_webqsp_generation/webqsp_full_generation_WORKING.jsonl"
    ),
    "CWQ": Path(
        "/kaggle/working/step4_rq3_llm_reasoning_v1/"
        "04_full_cwq_generation/cwq_full_generation_WORKING.jsonl"
    ),
}

expected = {
    "WebQSP": 3256,
    "CWQ": 7062,
}

for name, path in files.items():

    assert path.exists(), f"Missing file: {path}"

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        n = sum(
            1
            for line in f
            if line.strip()
        )

    print(
        f"{name}: "
        f"{n} / {expected[name]}"
    )

    if n == expected[name]:

        print("  COMPLETE")

    else:

        print("  CHECK NEEDED")
